# Extended Pipeline v260430

In [ ]:
#!/usr/bin/env python3
"""
=============================================================================
 PSYCHIATRIC GENE-SET DIFFERENTIAL & PROXIMITY PIPELINE (TWAS / S-PrediXcan)
=============================================================================
 User-supplied axis space: any number of diseases / conditions.
 User supplies ONE custom gene set (txt file or comma-separated list),
 OR MULTIPLE gene sets.

 Analyses:
   1. Per-disease enrichment (Stouffer Z, mean |Z|, Wilcoxon vs 0,
      permutation p, bootstrap CIs)
   2. Differential tests across diseases (Kruskal-Wallis + pairwise
      Mann-Whitney + FDR)
   3. Profile proximity / distances between diseases (restricted to
      gene-set genes) + hierarchical clustering dendrogram
   4. Per-gene detail table with Z-scores across all diseases
   5. Pairwise disease statistical tests (gene-set-restricted):
        Unpaired : Mann-Whitney U, KS 2-sample, Welch's t, Levene,
                   Brunner-Munzel, Cohen's d
        Paired   : paired t, Wilcoxon signed-rank, Cohen's d, sign-flip
                   permutation
        Concordance: sign concordance rate + binomial test, Lin's CCC,
                     Kendall's tau, concordance/discordance counts
      with BH-FDR correction across all pairs.
   6. Gene-level leave-one-out (LOO) influence analysis to flag
      results driven by 1-2 outlier genes.
   7. Z-score QQ-plot diagnostics per disease.

 Output: CSVs, summary, boxplot, enrichment bar, proximity heatmap,
         scatter matrix, pairwise-stats heatmap, QQ-plot, dendrogram,
         gene-influence plot, input-provenance manifest.

 CLI usage:
   python geneset_pipeline.py \
       --dirs data/mdd/ data/bip/ data/ocd/ \
       --labels MDD BIP OCD \
       --geneset my_genes.txt --label "Dopamine_Receptors" --out results/

   python geneset_pipeline.py \
       --dirs data/mdd/ data/bip/ data/ocd/ \
       --labels MDD BIP OCD \
       --genesets_file genesets.py --out results/ \
       --random_seed 123 --robust --bootstrap

 Programmatic usage:
   from geneset_pipeline import run_pipeline, run_multi_geneset_pipeline

   results = run_pipeline(
       dirs=["data/mdd/", "data/bip/", "data/ocd/"],
       labels=["MDD", "BIP", "OCD"],
       geneset="DRD1,DRD2,DRD3,DRD4,DRD5,COMT,SLC6A3",
       label="Dopamine_System",
       out="results/",
       random_seed=42,
       robust=False,
       bootstrap=True,
       n_boot=2000,
   )
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from itertools import combinations
import argparse, warnings, os, sys, textwrap, traceback
import hashlib
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datetime import datetime
import ast

warnings.filterwarnings("ignore")


# ===================================================================
#  0.  SMALL UTILITIES (provenance, validation, robust stats)
# ===================================================================

def _sha256_file(path, blocksize=65536):
    """Compute SHA256 of a file (short-form, first 16 hex chars)."""
    try:
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(blocksize), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return "ERROR"


def compute_input_provenance(dirs, geneset_input=None):
    """
    Build a provenance manifest containing SHA256 hashes for every
    input file plus timestamps.  Used for auditability/reproducibility.
    """
    manifest = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "python_version": sys.version.split()[0],
        "platform": sys.platform,
        "input_dirs": {},
        "geneset_input": None,
    }

    for d in dirs:
        d_path = Path(d)
        entry = {"path": str(d_path.resolve()), "exists": d_path.exists(), "files": {}}
        if d_path.is_dir():
            for f in sorted(d_path.iterdir()):
                if f.is_file():
                    entry["files"][f.name] = {
                        "sha256": _sha256_file(f),
                        "size_bytes": f.stat().st_size,
                        "mtime": datetime.fromtimestamp(f.stat().st_mtime).isoformat(timespec="seconds"),
                    }
        manifest["input_dirs"][str(d)] = entry

    if geneset_input is not None:
        try:
            if isinstance(geneset_input, (str, Path)) and Path(geneset_input).is_file():
                p = Path(geneset_input)
                manifest["geneset_input"] = {
                    "type": "file",
                    "path": str(p.resolve()),
                    "sha256": _sha256_file(p),
                    "size_bytes": p.stat().st_size,
                    "mtime": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
                }
            elif isinstance(geneset_input, (list, tuple, set)):
                manifest["geneset_input"] = {
                    "type": "iterable",
                    "n_items": len(list(geneset_input)),
                }
            elif isinstance(geneset_input, dict):
                manifest["geneset_input"] = {
                    "type": "dict",
                    "n_sets": len(geneset_input),
                    "labels": list(map(str, geneset_input.keys())),
                }
            else:
                manifest["geneset_input"] = {"type": "string", "preview": str(geneset_input)[:120]}
        except Exception as e:
            manifest["geneset_input"] = {"type": "unknown", "error": str(e)}
    return manifest


def validate_zscores(meta_z_dict, threshold_extreme=8.0, pct_warn=30.0):
    """
    Sanity check Z-score distributions per disease.

    Flags:
      * very high proportion of |Z| > threshold_extreme
      * duplicated keys (already deduped on load – noted for completeness)

    Returns list of warning strings.
    """
    warnings_list = []
    for d, mz in meta_z_dict.items():
        if not mz:
            warnings_list.append(f"{d}: empty meta-Z dictionary")
            continue
        zs = np.asarray(list(mz.values()), dtype=float)
        zs = zs[np.isfinite(zs)]
        if len(zs) == 0:
            warnings_list.append(f"{d}: no finite Z-scores")
            continue
        n_extreme = int(np.sum(np.abs(zs) > threshold_extreme))
        pct_extreme = 100 * n_extreme / len(zs)
        if pct_extreme > pct_warn:
            warnings_list.append(
                f"{d}: {pct_extreme:.1f}% of |Z| > {threshold_extreme} "
                f"(n={n_extreme}/{len(zs)}) — distribution may be inflated"
            )
        n_huge = int(np.sum(np.abs(zs) > 50))
        if n_huge > 0:
            warnings_list.append(
                f"{d}: {n_huge} extremely large |Z| > 50 detected — verify input"
            )
    return warnings_list


def _winsorize(arr, lower_pct=1.0, upper_pct=99.0):
    """Clip values to the [lower_pct, upper_pct] percentile range."""
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 3:
        return arr
    lo = np.percentile(arr, lower_pct)
    hi = np.percentile(arr, upper_pct)
    return np.clip(arr, lo, hi)


def _trimmed_mean(arr, proportion=0.1):
    """Symmetric trimmed mean."""
    return float(stats.trim_mean(arr, proportion))


# ===================================================================
#  1.  GENERALISED LOADER  (returns meta_z dict, Ensembl↔symbol map,
#                            AND per-tissue Z dicts)
# ===================================================================

def load_spredixcan_folder(folder_path):
    """
    Load S-PrediXcan results from a folder.  Handles CSV / TSV / TXT.
    Reads BOTH gene-symbol and Ensembl-ID columns when available.
    Computes meta-Z across tissue files via Stouffer's method.

    Returns
    -------
    meta_z        : dict  {GENE_SYMBOL: meta_z_score}
    id_map        : dict  {ENSEMBL_ID: GENE_SYMBOL, ENSEMBL_NO_VER: GENE_SYMBOL, …}
    tissue_gene_z : dict  {tissue_filename_stem: {GENE_SYMBOL: z_score, …}, …}
    """
    folder = Path(folder_path)
    files = []
    for ext in ("*.csv", "*.txt", "*.tsv", "*.dat"):
        files.extend(folder.glob(ext))
    files = sorted(set(files))
    if not files:
        raise FileNotFoundError(f"No result files found in {folder_path}")

    print(f"    {len(files)} file(s) in {folder.name}/")

    gene_z = defaultdict(list)   # canonical_name → [z, z, …]
    tissue_gene_z = {}           # tissue_stem   → {canonical_name: z}
    id_map = {}                  # alternative_id  → canonical_name

    for f in files:
        tissue_name = f.stem
        tissue_dict = {}
        parsed = False
        for sep in [",", "\t", r"\s+"]:
            try:
                engine = "python" if sep == r"\s+" else "c"
                df = pd.read_csv(f, sep=sep, engine=engine)
                cols_lower = {c.lower(): c for c in df.columns}

                symbol_col = None
                for candidate in ["gene_name", "genename", "symbol", "gene_symbol"]:
                    if candidate in cols_lower:
                        symbol_col = cols_lower[candidate]
                        break

                id_col = None
                for candidate in ["gene", "gene_id", "geneid", "ensembl_id", "ensembl"]:
                    if candidate in cols_lower:
                        actual = cols_lower[candidate]
                        if actual != symbol_col:
                            id_col = actual
                            break

                primary_col = symbol_col or id_col

                z_col = None
                for candidate in ["zscore", "z_score", "z", "zstat", "z_stat"]:
                    if candidate in cols_lower:
                        z_col = cols_lower[candidate]
                        break

                if not (primary_col and z_col):
                    continue

                keep = list(dict.fromkeys(
                    [c for c in [primary_col, z_col, symbol_col, id_col] if c is not None]
                ))
                sub = df[keep].dropna(subset=[primary_col, z_col])

                # Detect duplicates BEFORE deduping
                n_dup = int(sub.duplicated(subset=primary_col).sum())
                if n_dup > 0:
                    print(f"    ⚠ {f.name}: {n_dup} duplicate '{primary_col}' rows "
                          f"(kept first occurrence)")
                sub = sub.drop_duplicates(subset=primary_col, keep="first")

                for _, row in sub.iterrows():
                    sym, ens = None, None
                    if symbol_col:
                        s = str(row[symbol_col]).strip().upper()
                        if s and s not in ("NAN", "NA", "NONE", ""):
                            sym = s
                    if id_col:
                        e = str(row[id_col]).strip().upper()
                        if e and e not in ("NAN", "NA", "NONE", ""):
                            ens = e

                    canonical = sym or ens
                    if not canonical:
                        continue
                    try:
                        z_val = float(row[z_col])
                        gene_z[canonical].append(z_val)
                        tissue_dict[canonical] = z_val
                    except (ValueError, TypeError):
                        continue

                    if ens and sym:
                        id_map[ens] = sym
                        if "." in ens:
                            id_map[ens.split(".")[0]] = sym

                parsed = True
                break
            except Exception:
                continue

        if parsed and tissue_dict:
            tissue_gene_z[tissue_name] = tissue_dict

        if not parsed:
            print(f"    ⚠ Could not parse {f.name}")

    meta = {}
    for g, zs in gene_z.items():
        if len(zs) > 0:
            meta[g] = np.sum(zs) / np.sqrt(len(zs))

    print(f"    → meta-Z computed for {len(meta):,} genes")
    print(f"    → Ensembl aliases mapped: {len(id_map):,}")
    print(f"    → {len(tissue_gene_z)} tissue file(s) indexed")
    return meta, id_map, tissue_gene_z


# ===================================================================
#  2.  GENE-SET I/O  (tabular-aware loader + Ensembl↔symbol harmoniser)
# ===================================================================

def _try_load_tabular_geneset(filepath):
    for sep in ["\t", ",", r"\s+"]:
        try:
            engine = "python" if sep == r"\s+" else "c"
            df = pd.read_csv(filepath, sep=sep, engine=engine)

            if len(df.columns) < 2:
                continue

            cols_lower = {c.lower(): c for c in df.columns}

            known_cols = {
                "gene_name", "genename", "symbol", "gene_symbol",
                "gene", "gene_id", "zscore", "z_score", "pvalue",
                "effect_size", "n_snps_used", "pred_perf_r2",
            }
            if not any(k in cols_lower for k in known_cols):
                continue

            symbol_col = None
            for c in ["gene_name", "genename", "symbol", "gene_symbol"]:
                if c in cols_lower:
                    symbol_col = cols_lower[c]
                    break

            id_col = None
            for c in ["gene", "gene_id", "geneid", "ensembl_id"]:
                if c in cols_lower:
                    if cols_lower[c] != symbol_col:
                        id_col = cols_lower[c]
                        break

            primary = symbol_col or id_col
            if primary is None:
                continue

            genes = []
            fallback_col = id_col if (symbol_col and id_col) else None

            for _, row in df.iterrows():
                val = str(row[primary]).strip().upper() if pd.notna(row[primary]) else ""
                if val and val not in ("NAN", "NA", "NONE", ""):
                    genes.append(val)
                elif fallback_col is not None:
                    val = str(row[fallback_col]).strip().upper() if pd.notna(row[fallback_col]) else ""
                    if val and val not in ("NAN", "NA", "NONE", ""):
                        genes.append(val)

            if genes:
                print(f"  Loaded {len(genes)} genes from tabular file "
                      f"(column: '{primary}'): {Path(filepath).name}")
                return genes

        except Exception:
            continue
    return None


def load_geneset(geneset_input):
    if isinstance(geneset_input, (list, tuple, set)):
        raw = [str(g).strip().upper() for g in geneset_input if str(g).strip()]
        print(f"  Loaded {len(raw)} genes from Python iterable")

    elif Path(geneset_input).is_file():
        raw = _try_load_tabular_geneset(geneset_input)
        if raw is None:
            with open(geneset_input) as fh:
                raw = [line.strip().upper() for line in fh
                       if line.strip() and not line.startswith("#")]
            print(f"  Loaded {len(raw)} genes from plain-text file: "
                  f"{Path(geneset_input).name}")
    else:
        raw = [g.strip().upper() for g in geneset_input.split(",") if g.strip()]
        print(f"  Loaded {len(raw)} genes from inline string")

    seen = set()
    unique = []
    for g in raw:
        if g not in seen:
            seen.add(g)
            unique.append(g)
    return unique


def load_multiple_genesets(genesets_input):
    if isinstance(genesets_input, dict):
        out = {}
        for label, gs in genesets_input.items():
            out[str(label)] = load_geneset(gs)
        return out

    path = Path(genesets_input)
    if not path.is_file():
        raise FileNotFoundError(f"Gene-set collection file not found: {genesets_input}")

    for sep in ["\t", ","]:
        try:
            df = pd.read_csv(path, sep=sep)
            cols = {c.lower(): c for c in df.columns}
            if "label" in cols and "genes" in cols:
                out = {}
                for _, row in df.iterrows():
                    label = str(row[cols["label"]]).strip()
                    genes = str(row[cols["genes"]]).strip()
                    if label and genes and label.upper() != "NAN" and genes.upper() != "NAN":
                        out[label] = load_geneset(genes)
                if out:
                    return out
        except Exception:
            pass

    text = path.read_text()

    try:
        tree = ast.parse(text, filename=str(path))
        for node in tree.body:
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == "gene_sets":
                        value = ast.literal_eval(node.value)
                        if isinstance(value, dict):
                            out = {}
                            for label, gs in value.items():
                                out[str(label)] = load_geneset(gs)
                            return out
    except Exception:
        pass

    out = {}
    namespace = {}
    try:
        exec(text, {}, namespace)
        for k, v in namespace.items():
            if k.startswith("_"):
                continue
            if isinstance(v, (list, tuple, set, str)):
                try:
                    out[k] = load_geneset(v)
                except Exception:
                    pass
        if out:
            return out
    except Exception:
        pass

    raise ValueError(
        "Could not parse multiple gene sets. "
        "Use a dict, a Python file with assignments, or a TSV/CSV with columns: label, genes."
    )


def harmonize_geneset(gene_list, meta_z_dict, id_maps, verbose=True):
    _print = print if verbose else lambda *a, **k: None

    all_canonical = set()
    for mz in meta_z_dict.values():
        all_canonical.update(mz.keys())

    merged = {}
    for im in id_maps.values():
        merged.update(im)

    harmonized = []
    direct = 0
    aliased = 0
    missed = []

    for g in gene_list:
        if g in all_canonical:
            harmonized.append(g)
            direct += 1
            continue
        if g in merged and merged[g] in all_canonical:
            harmonized.append(merged[g])
            aliased += 1
            continue
        if "." in g:
            base = g.split(".")[0]
            if base in all_canonical:
                harmonized.append(base)
                aliased += 1
                continue
            if base in merged and merged[base] in all_canonical:
                harmonized.append(merged[base])
                aliased += 1
                continue
        harmonized.append(g)
        missed.append(g)

    seen = set()
    unique = []
    for g in harmonized:
        if g not in seen:
            seen.add(g)
            unique.append(g)

    n_dup = len(harmonized) - len(unique)
    _print(f"  Gene-name harmonisation:")
    _print(f"    direct matches   : {direct}")
    _print(f"    alias-resolved   : {aliased}")
    _print(f"    still unmatched  : {len(missed)}")
    if n_dup:
        _print(f"    duplicates removed after aliasing: {n_dup}")
    if missed and len(missed) <= 20:
        _print(f"    unmatched genes  : {', '.join(missed)}")

    return unique


# ===================================================================
#  3.  ENRICHMENT ANALYSIS  (with bootstrap CIs + robust option)
# ===================================================================

def bootstrap_enrichment(zv, n_boot=2000, rng=None):
    """
    Bootstrap percentile 95% CIs for Stouffer Z, mean Z, mean |Z|.

    Returns dict with *_ci_low / *_ci_high keys (rounded to 4 d.p.).
    Returns NaNs for samples smaller than 3.
    """
    out = {
        "stouffer_ci_low":   np.nan, "stouffer_ci_high":   np.nan,
        "mean_z_ci_low":     np.nan, "mean_z_ci_high":     np.nan,
        "mean_abs_z_ci_low": np.nan, "mean_abs_z_ci_high": np.nan,
        "n_boot": n_boot,
    }
    zv = np.asarray(zv, dtype=float)
    zv = zv[np.isfinite(zv)]
    n = len(zv)
    if n < 3 or n_boot <= 0:
        return out

    if rng is None:
        rng = np.random.default_rng(42)

    boot_stouffer = np.empty(n_boot)
    boot_mean     = np.empty(n_boot)
    boot_mean_abs = np.empty(n_boot)
    sqrtn = np.sqrt(n)
    for i in range(n_boot):
        sample = rng.choice(zv, size=n, replace=True)
        boot_stouffer[i] = np.sum(sample) / sqrtn
        boot_mean[i]     = np.mean(sample)
        boot_mean_abs[i] = np.mean(np.abs(sample))

    out["stouffer_ci_low"]   = round(float(np.percentile(boot_stouffer, 2.5)), 4)
    out["stouffer_ci_high"]  = round(float(np.percentile(boot_stouffer, 97.5)), 4)
    out["mean_z_ci_low"]     = round(float(np.percentile(boot_mean, 2.5)), 4)
    out["mean_z_ci_high"]    = round(float(np.percentile(boot_mean, 97.5)), 4)
    out["mean_abs_z_ci_low"] = round(float(np.percentile(boot_mean_abs, 2.5)), 4)
    out["mean_abs_z_ci_high"]= round(float(np.percentile(boot_mean_abs, 97.5)), 4)
    return out


def gene_set_enrichment(meta_z, gene_list, n_perm=10000,
                        random_seed=42, robust=False,
                        bootstrap=False, n_boot=2000):
    """
    Enrichment statistics for one disease:
      - Stouffer Z
      - Mean |Z|, mean Z, median Z
      - Trimmed mean (always reported), winsorized Stouffer (if robust=True)
      - Wilcoxon signed-rank test vs 0
      - Permutation p-value (skipped if n_perm == 0)
      - Bootstrap 95% CIs (if bootstrap=True)
      - Reliability flags (low_n_warning, low_coverage_warning)
    """
    zv = np.array([meta_z.get(g, np.nan) for g in gene_list])
    zv = zv[~np.isnan(zv)]
    n = len(zv)
    n_total = len(gene_list)
    coverage = round(100 * n / n_total, 1) if n_total else 0

    result = {
        "n_genes_found": n,
        "n_genes_total": n_total,
        "coverage_pct": coverage,
        "stouffer_z": np.nan,
        "stouffer_z_winsorized": np.nan,
        "mean_abs_z": np.nan,
        "mean_z": np.nan,
        "trimmed_mean_z": np.nan,
        "median_z": np.nan,
        "wilcoxon_p": np.nan,
        "permutation_p": np.nan,
        "z_scores": [],
        # Bootstrap fields
        "stouffer_ci_low":   np.nan, "stouffer_ci_high":   np.nan,
        "mean_z_ci_low":     np.nan, "mean_z_ci_high":     np.nan,
        "mean_abs_z_ci_low": np.nan, "mean_abs_z_ci_high": np.nan,
        # Reliability flags
        "low_n_warning":         bool(n < 5),
        "low_coverage_warning":  bool(coverage < 60.0 and n_total > 0),
        "wilcoxon_unreliable":   bool(n < 6),
        "robust_used":           bool(robust),
    }

    if n < 3:
        return result

    # Optional winsorization for robust mode
    zv_for_stats = _winsorize(zv) if robust else zv

    stouffer = np.sum(zv_for_stats) / np.sqrt(n)
    result["stouffer_z"] = round(float(stouffer), 4)

    # Always compute winsorized Stouffer for reference
    zv_w = _winsorize(zv)
    result["stouffer_z_winsorized"] = round(float(np.sum(zv_w) / np.sqrt(n)), 4)

    result["mean_abs_z"]    = round(float(np.mean(np.abs(zv_for_stats))), 4)
    result["mean_z"]        = round(float(np.mean(zv_for_stats)), 4)
    result["median_z"]      = round(float(np.median(zv)), 4)
    try:
        result["trimmed_mean_z"] = round(_trimmed_mean(zv, proportion=0.1), 4)
    except Exception:
        pass
    result["z_scores"] = zv.tolist()

    if n >= 6:
        try:
            _, wil_p = stats.wilcoxon(zv, alternative="two-sided")
            result["wilcoxon_p"] = round(float(wil_p), 6)
        except Exception:
            pass

    all_z = np.array(list(meta_z.values()))
    if len(all_z) > n and n_perm > 0:
        obs_stat = abs(stouffer)
        rng_perm = np.random.default_rng(random_seed)
        count = 0
        for _ in range(n_perm):
            perm_z = rng_perm.choice(all_z, size=n, replace=False)
            if robust:
                perm_z = _winsorize(perm_z)
            if abs(np.sum(perm_z) / np.sqrt(n)) >= obs_stat:
                count += 1
        result["permutation_p"] = round((count + 1) / (n_perm + 1), 6)

    if bootstrap and n_boot > 0:
        rng_boot = np.random.default_rng(random_seed + 1)
        ci = bootstrap_enrichment(zv_for_stats, n_boot=n_boot, rng=rng_boot)
        for k in ("stouffer_ci_low", "stouffer_ci_high",
                  "mean_z_ci_low", "mean_z_ci_high",
                  "mean_abs_z_ci_low", "mean_abs_z_ci_high"):
            result[k] = ci[k]

    return result


# ===================================================================
#  3b.  GENE-LEVEL INFLUENCE ANALYSIS (LOO / jackknife)
# ===================================================================

def gene_influence_analysis(meta_z_dict, gene_list, threshold_pct=20.0):
    """
    Leave-one-out (LOO) influence analysis on the per-disease Stouffer Z
    and mean |Z|.

    For each disease and each gene present in the data, recompute the
    enrichment statistic without that gene and record the delta.

    A gene is flagged as "Influential" if removing it:
      - flips the sign of the Stouffer Z, OR
      - changes |Stouffer Z| by more than `threshold_pct` percent.

    Returns a tidy DataFrame (one row per gene × disease).
    """
    rows = []
    diseases = list(meta_z_dict.keys())
    for d in diseases:
        # Build the in-data Z vector for this disease
        zv = []
        gene_in_data = []
        for g in gene_list:
            z = meta_z_dict[d].get(g, np.nan)
            if not np.isnan(z):
                zv.append(z)
                gene_in_data.append(g)
        zv = np.asarray(zv, dtype=float)
        n = len(zv)
        if n < 4:
            continue

        full_stouffer = np.sum(zv) / np.sqrt(n)
        full_mean_abs = np.mean(np.abs(zv))

        for i, g in enumerate(gene_in_data):
            zv_loo = np.delete(zv, i)
            n_loo = len(zv_loo)
            loo_stouffer = np.sum(zv_loo) / np.sqrt(n_loo)
            loo_mean_abs = np.mean(np.abs(zv_loo))

            d_stouffer = full_stouffer - loo_stouffer
            d_mean_abs = full_mean_abs - loo_mean_abs

            sign_change = (np.sign(full_stouffer) != np.sign(loo_stouffer)
                           and full_stouffer != 0 and loo_stouffer != 0)
            pct_change = (abs(d_stouffer) / abs(full_stouffer) * 100
                          if full_stouffer != 0 else np.nan)
            influential = bool(sign_change or
                               (not np.isnan(pct_change) and pct_change > threshold_pct))

            rows.append({
                "Disease": d,
                "Gene": g,
                "Z_score": round(float(zv[i]), 4),
                "Full_Stouffer_Z": round(float(full_stouffer), 4),
                "LOO_Stouffer_Z":  round(float(loo_stouffer), 4),
                "Delta_Stouffer":  round(float(d_stouffer), 4),
                "Pct_Change_Stouffer": (round(float(pct_change), 2)
                                        if not np.isnan(pct_change) else np.nan),
                "Sign_Change": bool(sign_change),
                "Full_Mean_AbsZ":  round(float(full_mean_abs), 4),
                "LOO_Mean_AbsZ":   round(float(loo_mean_abs), 4),
                "Delta_Mean_AbsZ": round(float(d_mean_abs), 4),
                "Influential": influential,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["Disease", "Pct_Change_Stouffer"],
                            ascending=[True, False]).reset_index(drop=True)
    return df


# ===================================================================
#  4.  DIFFERENTIAL TESTS
# ===================================================================

def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return np.array([])
    order = np.argsort(pvals)
    sorted_p = pvals[order]
    adjusted = np.empty(n)
    cum_min = 1.0
    for i in range(n - 1, -1, -1):
        adj = sorted_p[i] * n / (i + 1)
        cum_min = min(cum_min, adj)
        adjusted[order[i]] = min(cum_min, 1.0)
    return adjusted


def differential_tests(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    common = [g for g in gene_list if all(g in meta_z_dict[d] for d in diseases)]

    result = {
        "n_common_genes": len(common),
        "common_genes": common,
        "kruskal_h": np.nan,
        "kruskal_p": np.nan,
        "pairwise_df": pd.DataFrame()
    }

    if len(common) < 3:
        print("  ⚠ Fewer than 3 genes overlap across all diseases – skipping differential tests")
        return result

    z_by_disease = {d: np.array([meta_z_dict[d][g] for g in common]) for d in diseases}

    h_stat, kw_p = stats.kruskal(*z_by_disease.values())
    result["kruskal_h"] = round(h_stat, 4)
    result["kruskal_p"] = round(kw_p, 6)

    rows = []
    for d1, d2 in combinations(diseases, 2):
        v1, v2 = z_by_disease[d1], z_by_disease[d2]
        u_stat, mwu_p = stats.mannwhitneyu(v1, v2, alternative="two-sided")
        n1, n2 = len(v1), len(v2)
        rbc = 1 - (2 * u_stat) / (n1 * n2)

        try:
            _, paired_p = stats.wilcoxon(v1 - v2, alternative="two-sided")
            paired_p = round(paired_p, 6)
        except Exception:
            paired_p = np.nan

        rows.append({
            "Disease_1": d1,
            "Disease_2": d2,
            "MWU_U": int(u_stat),
            "MWU_p": round(mwu_p, 6),
            "Rank_Biserial_r": round(rbc, 4),
            "Paired_Wilcoxon_p": paired_p,
            "Mean_Z_diff": round(np.mean(v1) - np.mean(v2), 4)
        })

    pw_df = pd.DataFrame(rows)
    if len(pw_df) > 0:
        pw_df["MWU_FDR"] = benjamini_hochberg(pw_df["MWU_p"].values)
        pw_p = pw_df["Paired_Wilcoxon_p"].values
        if not np.any(np.isnan(pw_p)):
            pw_df["Paired_FDR"] = benjamini_hochberg(pw_p)
    result["pairwise_df"] = pw_df
    return result


# ===================================================================
#  4b.  PAIRWISE DISEASE STATISTICAL TESTS  (gene-set-restricted)
# ===================================================================

def _lins_ccc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3:
        return np.nan
    mx, my = np.mean(x), np.mean(y)
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    if sx == 0 or sy == 0:
        return np.nan
    r, _ = stats.pearsonr(x, y)
    numerator = 2 * r * sx * sy
    denominator = sx ** 2 + sy ** 2 + (mx - my) ** 2
    if denominator == 0:
        return np.nan
    return numerator / denominator


def pairwise_disease_geneset_tests(meta_z_dict, gene_list,
                                   n_perm=5000, random_seed=42, robust=False):
    """
    Comprehensive pairwise statistical tests between diseases,
    restricted to gene-set genes.

    With `robust=True`, Z-vectors are winsorized (1st/99th percentile)
    before parametric (t-test, Cohen's d, Levene) computations.
    Non-parametric tests are unaffected by winsorization.
    """
    diseases = list(meta_z_dict.keys())
    if len(diseases) < 2:
        return pd.DataFrame()

    rows = []
    for d1, d2 in combinations(diseases, 2):
        z1_all = np.array([meta_z_dict[d1][g] for g in gene_list
                           if g in meta_z_dict[d1]
                           and np.isfinite(meta_z_dict[d1][g])])
        z2_all = np.array([meta_z_dict[d2][g] for g in gene_list
                           if g in meta_z_dict[d2]
                           and np.isfinite(meta_z_dict[d2][g])])

        common = [g for g in gene_list
                  if g in meta_z_dict[d1] and g in meta_z_dict[d2]
                  and np.isfinite(meta_z_dict[d1][g])
                  and np.isfinite(meta_z_dict[d2][g])]
        z1_p = np.array([meta_z_dict[d1][g] for g in common])
        z2_p = np.array([meta_z_dict[d2][g] for g in common])

        # Optional robust transformation for parametric tests
        z1_all_r = _winsorize(z1_all) if robust else z1_all
        z2_all_r = _winsorize(z2_all) if robust else z2_all
        z1_p_r   = _winsorize(z1_p)   if robust else z1_p
        z2_p_r   = _winsorize(z2_p)   if robust else z2_p

        row = {
            "Disease_1": d1,
            "Disease_2": d2,
            "n_genes_D1": len(z1_all),
            "n_genes_D2": len(z2_all),
            "n_genes_paired": len(common),
            "mean_Z_D1":   round(float(np.mean(z1_all)),   4) if len(z1_all) else np.nan,
            "mean_Z_D2":   round(float(np.mean(z2_all)),   4) if len(z2_all) else np.nan,
            "sd_Z_D1":     round(float(np.std(z1_all, ddof=1)), 4) if len(z1_all) > 1 else np.nan,
            "sd_Z_D2":     round(float(np.std(z2_all, ddof=1)), 4) if len(z2_all) > 1 else np.nan,
            "median_Z_D1": round(float(np.median(z1_all)), 4) if len(z1_all) else np.nan,
            "median_Z_D2": round(float(np.median(z2_all)), 4) if len(z2_all) else np.nan,
        }

        if len(common) >= 1:
            diffs = z1_p - z2_p
            diffs_r = z1_p_r - z2_p_r
            row["mean_diff"]   = round(float(np.mean(diffs)),   4)
            row["median_diff"] = round(float(np.median(diffs)), 4)
            row["sd_diff"]     = round(float(np.std(diffs, ddof=1)), 4) if len(common) > 1 else np.nan
        else:
            diffs = np.array([])
            diffs_r = np.array([])
            row["mean_diff"]   = np.nan
            row["median_diff"] = np.nan
            row["sd_diff"]     = np.nan

        _enough_unpaired = len(z1_all) >= 3 and len(z2_all) >= 3

        row["MWU_U"] = np.nan; row["MWU_p"] = np.nan; row["Rank_Biserial_r"] = np.nan
        if _enough_unpaired:
            try:
                u, p = stats.mannwhitneyu(z1_all, z2_all, alternative="two-sided")
                n1, n2 = len(z1_all), len(z2_all)
                row["MWU_U"] = int(u)
                row["MWU_p"] = round(p, 6)
                row["Rank_Biserial_r"] = round(1 - 2 * u / (n1 * n2), 4)
            except Exception:
                pass

        row["KS_stat"] = np.nan; row["KS_p"] = np.nan
        if _enough_unpaired:
            try:
                ks, p = stats.ks_2samp(z1_all, z2_all)
                row["KS_stat"] = round(ks, 4)
                row["KS_p"]    = round(p, 6)
            except Exception:
                pass

        row["Welch_t"] = np.nan; row["Welch_p"] = np.nan
        if _enough_unpaired:
            try:
                t, p = stats.ttest_ind(z1_all_r, z2_all_r, equal_var=False)
                row["Welch_t"] = round(t, 4)
                row["Welch_p"] = round(p, 6)
            except Exception:
                pass

        row["Levene_W"] = np.nan; row["Levene_p"] = np.nan
        if _enough_unpaired:
            try:
                w, p = stats.levene(z1_all_r, z2_all_r)
                row["Levene_W"] = round(w, 4)
                row["Levene_p"] = round(p, 6)
            except Exception:
                pass

        row["BM_W"] = np.nan; row["BM_p"] = np.nan
        if _enough_unpaired:
            try:
                bm, p = stats.brunnermunzel(z1_all, z2_all)
                row["BM_W"] = round(bm, 4)
                row["BM_p"] = round(p, 6)
            except Exception:
                pass

        row["Cohens_d_unpaired"] = np.nan
        if _enough_unpaired:
            try:
                s1 = np.var(z1_all_r, ddof=1)
                s2 = np.var(z2_all_r, ddof=1)
                pooled = np.sqrt((s1 + s2) / 2)
                if pooled > 0:
                    row["Cohens_d_unpaired"] = round(
                        float((np.mean(z1_all_r) - np.mean(z2_all_r)) / pooled), 4
                    )
            except Exception:
                pass

        row["Paired_t_stat"] = np.nan; row["Paired_t_p"] = np.nan
        row["Paired_Wilcoxon_W"] = np.nan; row["Paired_Wilcoxon_p"] = np.nan
        row["Cohens_d_paired"] = np.nan
        row["Permutation_p"] = np.nan

        if len(common) >= 3:
            try:
                t, p = stats.ttest_rel(z1_p_r, z2_p_r)
                row["Paired_t_stat"] = round(t, 4)
                row["Paired_t_p"]    = round(p, 6)
            except Exception:
                pass

            if len(common) >= 6:
                try:
                    w, p = stats.wilcoxon(diffs, alternative="two-sided")
                    row["Paired_Wilcoxon_W"] = int(w)
                    row["Paired_Wilcoxon_p"] = round(p, 6)
                except Exception:
                    pass

            try:
                sd_d = np.std(diffs_r, ddof=1)
                if sd_d > 0:
                    row["Cohens_d_paired"] = round(float(np.mean(diffs_r) / sd_d), 4)
            except Exception:
                pass

            if n_perm > 0:
                try:
                    obs = abs(np.mean(diffs))
                    rng = np.random.default_rng(random_seed)
                    count = 0
                    for _ in range(n_perm):
                        signs = rng.choice([-1, 1], size=len(diffs))
                        if abs(np.mean(diffs * signs)) >= obs:
                            count += 1
                    row["Permutation_p"] = round((count + 1) / (n_perm + 1), 6)
                except Exception:
                    pass

        row["n_concordant"]          = np.nan
        row["n_discordant"]          = np.nan
        row["n_zero_either"]         = np.nan
        row["sign_concordance_rate"] = np.nan
        row["sign_concordance_p"]    = np.nan
        row["Lin_CCC"]               = np.nan
        row["Kendall_tau"]           = np.nan
        row["Kendall_p"]             = np.nan

        if len(common) >= 3:
            try:
                signs1 = np.sign(z1_p)
                signs2 = np.sign(z2_p)
                zero_mask = (signs1 == 0) | (signs2 == 0)
                n_zero = int(np.sum(zero_mask))
                row["n_zero_either"] = n_zero
                nz_mask = ~zero_mask
                s1_nz = signs1[nz_mask]
                s2_nz = signs2[nz_mask]
                n_nz = len(s1_nz)
                if n_nz >= 1:
                    concordant_mask = (s1_nz == s2_nz)
                    n_conc = int(np.sum(concordant_mask))
                    n_disc = n_nz - n_conc
                    row["n_concordant"] = n_conc
                    row["n_discordant"] = n_disc
                    row["sign_concordance_rate"] = round(n_conc / n_nz, 4)
                    if n_nz >= 3:
                        try:
                            binom_p = stats.binomtest(n_conc, n_nz, 0.5).pvalue
                        except AttributeError:
                            binom_p = stats.binom_test(n_conc, n_nz, 0.5)
                        row["sign_concordance_p"] = round(float(binom_p), 6)
            except Exception:
                pass

            try:
                ccc = _lins_ccc(z1_p, z2_p)
                if not np.isnan(ccc):
                    row["Lin_CCC"] = round(float(ccc), 4)
            except Exception:
                pass

            try:
                tau, tau_p = stats.kendalltau(z1_p, z2_p)
                row["Kendall_tau"] = round(float(tau), 4)
                row["Kendall_p"]   = round(float(tau_p), 6)
            except Exception:
                pass

        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    p_cols = [c for c in df.columns if c.endswith("_p")]
    for pc in p_cols:
        vals = df[pc].values.astype(float)
        mask = ~np.isnan(vals)
        if mask.sum() > 1:
            fdr_vals = np.full(len(vals), np.nan)
            fdr_vals[mask] = benjamini_hochberg(vals[mask])
            df[pc.replace("_p", "_FDR")] = fdr_vals
        elif mask.sum() == 1:
            df[pc.replace("_p", "_FDR")] = vals

    return df


# ===================================================================
#  5.  PROXIMITY / DISTANCE
# ===================================================================

def profile_proximity(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    common = [g for g in gene_list if all(g in meta_z_dict[d] for d in diseases)]

    if len(common) < 5:
        print("  ⚠ Fewer than 5 overlapping genes – proximity metrics unreliable")
    if len(common) < 3:
        return pd.DataFrame(), pd.DataFrame()

    profile_matrix = pd.DataFrame(
        {d: [meta_z_dict[d][g] for g in common] for d in diseases},
        index=common
    )

    rows = []
    for d1, d2 in combinations(diseases, 2):
        v1 = profile_matrix[d1].values
        v2 = profile_matrix[d2].values

        pr, pr_p = stats.pearsonr(v1, v2)
        sr, sr_p = stats.spearmanr(v1, v2)
        cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-15)
        eucl = np.linalg.norm(v1 - v2)
        manhattan = np.sum(np.abs(v1 - v2))

        rows.append({
            "Disease_1": d1,
            "Disease_2": d2,
            "Pearson_r": round(pr, 4),
            "Pearson_p": round(pr_p, 6),
            "Spearman_rho": round(sr, 4),
            "Spearman_p": round(sr_p, 6),
            "Cosine_sim": round(cos_sim, 4),
            "Euclidean_dist": round(eucl, 4),
            "Manhattan_dist": round(manhattan, 4),
            "n_genes": len(common)
        })

    return pd.DataFrame(rows), profile_matrix


# ===================================================================
#  6.  PER-GENE DETAIL TABLE
# ===================================================================

def build_gene_detail_table(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    rows = []
    for g in gene_list:
        row = {"Gene": g}
        zvals = []
        for d in diseases:
            z = meta_z_dict[d].get(g, np.nan)
            row[f"Z_{d}"] = round(z, 4) if not np.isnan(z) else np.nan
            if not np.isnan(z):
                zvals.append(z)
        row["N_diseases"] = len(zvals)
        if len(zvals) >= 2:
            row["Z_range"] = round(max(zvals) - min(zvals), 4)
            row["Z_mean"] = round(np.mean(zvals), 4)
        else:
            row["Z_range"] = np.nan
            row["Z_mean"] = np.nan
        rows.append(row)

    df = pd.DataFrame(rows)
    sort_cols = [f"Z_{d}" for d in diseases]
    df["_sort"] = df[sort_cols].abs().mean(axis=1)
    df = df.sort_values("_sort", ascending=False).drop(columns="_sort").reset_index(drop=True)
    return df


# ===================================================================
#  7.  VISUALISATION
# ===================================================================

def _make_disease_colors(diseases):
    _LEGACY = {"MDD": "#e74c3c", "BIP": "#3498db", "OCD": "#2ecc71"}
    pal = sns.color_palette("husl", max(len(diseases), 3))
    colors = {}
    pal_idx = 0
    for d in diseases:
        if d in _LEGACY:
            colors[d] = _LEGACY[d]
        else:
            colors[d] = pal[pal_idx]
            pal_idx += 1
    return colors


def plot_zscore_boxplot(enrichment_dict, label, out_dir):
    data = []
    for dis, info in enrichment_dict.items():
        for z in info["z_scores"]:
            data.append({"Disease": dis, "meta-Z": z})
    if not data:
        return
    df = pd.DataFrame(data)

    order = list(enrichment_dict.keys())
    dcol = _make_disease_colors(order)
    colors = [dcol.get(d, "#999") for d in order]

    fig, ax = plt.subplots(figsize=(max(7, len(order) * 2), 5))

    sns.boxplot(data=df, x="Disease", y="meta-Z", order=order,
                palette=colors, width=0.5, ax=ax)
    sns.stripplot(data=df, x="Disease", y="meta-Z", order=order,
                  color=".25", size=3, alpha=0.6, ax=ax)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(f"Gene-set meta-Z Distribution – {label}", fontsize=13)
    ax.set_ylabel("meta-Z (TWAS association)")

    for i, d in enumerate(order):
        n = enrichment_dict.get(d, {}).get("n_genes_found", 0)
        ax.text(i, ax.get_ylim()[0], f"n={n}", ha="center", va="top",
                fontsize=9, color="gray")

    plt.tight_layout()
    path = os.path.join(out_dir, f"zscore_boxplot_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_enrichment_bar(enrichment_dict, label, out_dir):
    order = list(enrichment_dict.keys())
    dcol = _make_disease_colors(order)

    rows = []
    for d in order:
        rows.append({
            "Disease": d,
            "Stouffer Z": enrichment_dict[d].get("stouffer_z", 0) or 0,
            "Mean |Z|":   enrichment_dict[d].get("mean_abs_z", 0) or 0,
            "Stouffer_lo": enrichment_dict[d].get("stouffer_ci_low", np.nan),
            "Stouffer_hi": enrichment_dict[d].get("stouffer_ci_high", np.nan),
            "MeanAbs_lo":  enrichment_dict[d].get("mean_abs_z_ci_low", np.nan),
            "MeanAbs_hi":  enrichment_dict[d].get("mean_abs_z_ci_high", np.nan),
        })
    if not rows:
        return
    df = pd.DataFrame(rows)
    colors = [dcol.get(d, "#999") for d in df["Disease"]]

    fig, axes = plt.subplots(1, 2, figsize=(max(10, len(order) * 2.5), 5))

    bars0 = axes[0].bar(df["Disease"], df["Stouffer Z"], color=colors, edgecolor="white")
    # Draw bootstrap CI error bars when available
    if df["Stouffer_lo"].notna().any():
        for b, lo, hi, val in zip(bars0, df["Stouffer_lo"], df["Stouffer_hi"], df["Stouffer Z"]):
            if not (np.isnan(lo) or np.isnan(hi)):
                axes[0].plot([b.get_x() + b.get_width() / 2] * 2, [lo, hi],
                             color="black", lw=1.2)
                axes[0].plot([b.get_x() + b.get_width() / 2 - 0.08,
                              b.get_x() + b.get_width() / 2 + 0.08], [lo, lo],
                             color="black", lw=1.2)
                axes[0].plot([b.get_x() + b.get_width() / 2 - 0.08,
                              b.get_x() + b.get_width() / 2 + 0.08], [hi, hi],
                             color="black", lw=1.2)

    axes[0].axhline(0, color="gray", ls="--", lw=0.8)
    axes[0].axhline(1.96, color="red", ls=":", lw=0.7, alpha=0.5)
    axes[0].axhline(-1.96, color="red", ls=":", lw=0.7, alpha=0.5)
    axes[0].set_title("Stouffer Z (with 95% bootstrap CI if available)")
    axes[0].set_ylabel("Combined Z-score")

    bars1 = axes[1].bar(df["Disease"], df["Mean |Z|"], color=colors, edgecolor="white")
    if df["MeanAbs_lo"].notna().any():
        for b, lo, hi in zip(bars1, df["MeanAbs_lo"], df["MeanAbs_hi"]):
            if not (np.isnan(lo) or np.isnan(hi)):
                axes[1].plot([b.get_x() + b.get_width() / 2] * 2, [lo, hi],
                             color="black", lw=1.2)
    axes[1].set_title("Mean |Z|")
    axes[1].set_ylabel("Mean absolute Z-score")

    fig.suptitle(f"Gene-set Enrichment – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"enrichment_bar_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_proximity_heatmap(prox_df, diseases, label, out_dir):
    if prox_df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(max(12, len(diseases) * 3), max(5, len(diseases) * 2)))
    for ax, metric, title in zip(axes,
                                  ["Pearson_r", "Cosine_sim"],
                                  ["Pearson r", "Cosine Similarity"]):
        mat = pd.DataFrame(np.eye(len(diseases)), index=diseases, columns=diseases)
        for _, row in prox_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[metric]
            if d1 in diseases and d2 in diseases:
                mat.loc[d1, d2] = val
                mat.loc[d2, d1] = val

        sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap="RdBu_r",
                    center=0, vmin=-1, vmax=1, square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(title)

    fig.suptitle(f"Profile Proximity – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"proximity_heatmap_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_proximity_dendrogram(prox_df, diseases, label, out_dir):
    """
    Hierarchical clustering of diseases based on (1 - Pearson r) distance.
    Requires at least 3 diseases.
    """
    if prox_df.empty or len(diseases) < 3:
        return
    try:
        from scipy.cluster.hierarchy import linkage, dendrogram
        from scipy.spatial.distance import squareform
    except Exception as e:
        print(f"  ⚠ scipy clustering unavailable: {e}")
        return

    n = len(diseases)
    dmat = np.zeros((n, n))
    idx = {d: i for i, d in enumerate(diseases)}
    for _, row in prox_df.iterrows():
        d1, d2 = row["Disease_1"], row["Disease_2"]
        if d1 in idx and d2 in idx:
            dist = max(0.0, 1 - float(row["Pearson_r"]))
            dmat[idx[d1], idx[d2]] = dist
            dmat[idx[d2], idx[d1]] = dist

    try:
        condensed = squareform(dmat, checks=False)
        Z = linkage(condensed, method="average")
        fig, ax = plt.subplots(figsize=(max(6, n * 1.3), 5))
        dendrogram(Z, labels=diseases, ax=ax, color_threshold=0)
        ax.set_ylabel("Distance (1 − Pearson r)")
        ax.set_title(f"Disease Hierarchical Clustering – {label}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        path = os.path.join(out_dir, f"proximity_dendrogram_{label}.png")
        plt.savefig(path, dpi=300)
        plt.close()
        print(f"  → {path}")
    except Exception as e:
        print(f"  ⚠ Dendrogram failed: {e}")


def plot_zscore_qq(meta_z_dict, gene_list, label, out_dir):
    """
    Per-disease QQ-plot: gene-set Z-scores vs N(0,1) theoretical quantiles.
    Useful sanity check for calibration.
    """
    diseases = list(meta_z_dict.keys())
    n = len(diseases)
    if n == 0:
        return
    fig, axes = plt.subplots(1, n, figsize=(max(5 * n, 6), 5))
    if n == 1:
        axes = [axes]
    for ax, d in zip(axes, diseases):
        gs_z = np.array([meta_z_dict[d].get(g, np.nan) for g in gene_list])
        gs_z = gs_z[~np.isnan(gs_z)]
        if len(gs_z) < 3:
            ax.text(0.5, 0.5, f"{d}: n<3", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11)
            ax.set_xticks([]); ax.set_yticks([])
            continue
        observed = np.sort(gs_z)
        theoretical = stats.norm.ppf((np.arange(1, len(observed) + 1) - 0.5) / len(observed))
        ax.scatter(theoretical, observed, alpha=0.7, s=30,
                   c="steelblue", edgecolor="black", lw=0.3)
        lims = [min(theoretical.min(), observed.min()),
                max(theoretical.max(), observed.max())]
        ax.plot(lims, lims, "r--", lw=1, label="y = x (N(0,1))")
        ax.axhline(0, color="gray", lw=0.5, ls=":")
        ax.axvline(0, color="gray", lw=0.5, ls=":")
        ax.set_xlabel("Theoretical N(0,1) quantile")
        ax.set_ylabel("Observed gene-set meta-Z")
        ax.set_title(f"{d} (n={len(observed)})", fontsize=11)
        ax.legend(fontsize=8, loc="upper left")
    fig.suptitle(f"Z-score QQ-plots (gene set vs N(0,1)) – {label}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"qq_plot_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_gene_influence(influence_df, label, out_dir, top_n=15):
    """
    Per-disease lollipop / bar plot of the top-|ΔStouffer| genes.
    Influential genes (sign-flip or > threshold% change) are highlighted in red.
    """
    if influence_df.empty:
        return
    diseases = list(influence_df["Disease"].unique())
    n = len(diseases)
    fig, axes = plt.subplots(1, n,
        figsize=(max(6 * n, 8), max(5, top_n * 0.35)))
    if n == 1:
        axes = [axes]
    for ax, d in zip(axes, diseases):
        sub = influence_df[influence_df["Disease"] == d].copy()
        sub = sub.reindex(sub["Delta_Stouffer"].abs().sort_values(ascending=False).index)
        sub = sub.head(top_n).iloc[::-1]
        if sub.empty:
            ax.text(0.5, 0.5, f"{d}: insufficient data",
                    ha="center", va="center", transform=ax.transAxes)
            continue
        colors = ["#e74c3c" if x else "#3498db" for x in sub["Influential"]]
        ax.barh(sub["Gene"], sub["Delta_Stouffer"], color=colors,
                edgecolor="black", lw=0.3)
        ax.axvline(0, color="gray", ls="--", lw=0.8)
        ax.set_xlabel("Δ Stouffer Z (full − LOO)")
        ax.set_title(f"{d}: top influential genes", fontsize=11)
    fig.suptitle(f"Gene-level Leave-One-Out Influence – {label}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"gene_influence_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_scatter_matrix(profile_matrix, label, out_dir):
    if profile_matrix.empty:
        return

    diseases = list(profile_matrix.columns)
    pairs = list(combinations(diseases, 2))
    n_pairs = len(pairs)
    if n_pairs == 0:
        return

    fig, axes = plt.subplots(1, n_pairs, figsize=(5 * n_pairs, 5))
    if n_pairs == 1:
        axes = [axes]

    for ax, (d1, d2) in zip(axes, pairs):
        v1 = profile_matrix[d1].values
        v2 = profile_matrix[d2].values
        ax.scatter(v1, v2, alpha=0.5, s=20, color="#555")

        slope, intercept = np.polyfit(v1, v2, 1)
        xr = np.linspace(v1.min(), v1.max(), 100)
        ax.plot(xr, slope * xr + intercept, color="red", lw=1.5, ls="--")

        r, _ = stats.pearsonr(v1, v2)
        ax.set_xlabel(f"{d1} meta-Z")
        ax.set_ylabel(f"{d2} meta-Z")
        ax.set_title(f"{d1} vs {d2}  (r = {r:.3f})")
        ax.axhline(0, color="gray", ls=":", lw=0.5)
        ax.axvline(0, color="gray", ls=":", lw=0.5)

    fig.suptitle(f"Cross-Disease Z-score Scatter – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"scatter_matrix_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_pairwise_stats_heatmap(pw_df, diseases, label, out_dir):
    if pw_df.empty or len(diseases) < 2:
        return

    p_candidates = [
        ("KS_p",               "KS 2-sample"),
        ("Welch_p",            "Welch t"),
        ("Paired_Wilcoxon_p",  "Paired Wilcoxon"),
        ("Paired_t_p",         "Paired t"),
        ("Permutation_p",      "Permutation"),
        ("sign_concordance_p", "Sign Concordance"),
        ("Kendall_p",          "Kendall τ"),
    ]
    p_metrics = [(col, lbl) for col, lbl in p_candidates
                 if col in pw_df.columns and pw_df[col].notna().any()]

    d_candidates = [
        ("Cohens_d_unpaired", "Cohen's d (unpaired)"),
        ("Cohens_d_paired",   "Cohen's d (paired)"),
    ]
    d_metrics = [(col, lbl) for col, lbl in d_candidates
                 if col in pw_df.columns and pw_df[col].notna().any()]

    conc_candidates = [
        ("sign_concordance_rate", "Sign Concordance Rate"),
        ("Lin_CCC",               "Lin's CCC"),
        ("Kendall_tau",            "Kendall τ"),
    ]
    conc_metrics = [(col, lbl) for col, lbl in conc_candidates
                    if col in pw_df.columns and pw_df[col].notna().any()]

    n_panels = len(p_metrics) + len(d_metrics) + len(conc_metrics)
    if n_panels == 0:
        return

    fig, axes = plt.subplots(
        1, n_panels,
        figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
    )
    if n_panels == 1:
        axes = [axes]

    panel_idx = 0

    for col, lbl in p_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                neg_log = -np.log10(max(val, 1e-300))
                mat.loc[d1, d2] = neg_log
                mat.loc[d2, d1] = neg_log
        for d in diseases:
            mat.loc[d, d] = 0.0

        sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                    square=True, ax=ax, linewidths=0.5, linecolor="white",
                    vmin=0)
        ax.set_title(f"−log₁₀(p)\n{lbl}", fontsize=10)

    for col, lbl in d_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        mat = pd.DataFrame(0.0, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                mat.loc[d1, d2] =  val
                mat.loc[d2, d1] = -val

        vabs = max(abs(mat.values.min()), abs(mat.values.max()), 0.01)
        sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
                    center=0, vmin=-vabs, vmax=vabs, square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(lbl, fontsize=10)

    for col, lbl in conc_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        if col == "sign_concordance_rate":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = 0.0, 1.0, 0.5
            cmap = "YlGn"
        elif col == "Lin_CCC":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"
        elif col == "Kendall_tau":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"
        else:
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"

        mat = pd.DataFrame(diag_val, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                mat.loc[d1, d2] = val
                mat.loc[d2, d1] = val

        sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap=cmap,
                    center=center_val, vmin=vmin_val, vmax=vmax_val,
                    square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(lbl, fontsize=10)

    fig.suptitle(f"Pairwise Disease Tests – {label}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"pairwise_stats_heatmap_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


# ===================================================================
#  8.  CORE PIPELINE FUNCTION (callable directly)
# ===================================================================

# Static block appended to every summary file
_LIMITATIONS_BLOCK = textwrap.dedent("""\
    LIMITATIONS & ASSUMPTIONS
    -------------------------
    * Stouffer's method assumes (approximately) independent gene-level
      associations.  Linkage disequilibrium and shared eQTLs in TWAS can
      induce gene–gene correlations, slightly inflating Type I error.
    * Permutation p-values are computed against a random gene-set null
      and capture competitive (vs. genome) enrichment; they do not test
      self-containment vs. biological pathway annotation.
    * Wilcoxon/paired tests with n < 6 are flagged as unreliable.
      Coverage < 60% should be interpreted with caution.
    * Bootstrap CIs reflect resampling uncertainty in the gene-set
      members, not in the upstream TWAS Z-scores themselves.
    * Concordance metrics treat each gene independently and do not
      account for cross-tissue Z-score correlation.

    HOW TO INTERPRET
    ----------------
    * |Stouffer Z| > 1.96 ↔ two-sided p < 0.05 (uncorrected); compare
      against permutation p for a competitive null.
    * Cohen's d benchmarks (between-disease):
        small ≈ 0.2 | medium ≈ 0.5 | large ≈ 0.8
    * Rank-biserial r benchmarks:
        small ≈ 0.1 | medium ≈ 0.3 | large ≈ 0.5
    * Lin's CCC: 0 = no concordance, 1 = perfect agreement on identity line.
    * Sign concordance rate = 0.5 expected under independence; values
      ≫ 0.5 indicate shared directional effects.
""")


def run_pipeline(
    dirs,
    labels,
    geneset,
    label="CustomGeneSet",
    out="results",
    n_perm=10000,
    save_outputs=True,
    plot=True,
    verbose=True,
    # ── New robustness parameters (all backward-compatible) ─────────
    random_seed=42,
    robust=False,
    bootstrap=False,
    n_boot=2000,
    family_wise_fdr=False,
    influence_threshold_pct=20.0,
):
    """
    Run the full gene-set differential & proximity pipeline.

    New parameters
    --------------
    random_seed : int
        Master RNG seed used for permutation, sign-flip, and bootstrap
        steps.  Logged into every summary for reproducibility.
    robust : bool
        If True, winsorize Z-scores at 1st/99th percentile before
        parametric statistics (Stouffer Z, Welch t, Cohen's d, ...).
        Non-parametric tests are unaffected.
    bootstrap : bool
        If True, compute 95% bootstrap percentile CIs for Stouffer Z,
        mean Z, and mean |Z| (added to the enrichment table and plots).
    n_boot : int
        Number of bootstrap resamples.
    family_wise_fdr : bool
        If True, additionally compute a single global BH-FDR across
        ALL p-values produced by enrichment + differential + pairwise
        + proximity tests within this run.  Result saved as
        family_wise_fdr_<label>.csv.
    influence_threshold_pct : float
        Percent change in |Stouffer Z| at which a single gene removal
        is flagged as "influential" in the LOO analysis.

    Returns dict (existing keys preserved, new keys added):
        ... (existing) ...
        "influence_df"  : per-disease LOO influence table
        "provenance"    : input-file SHA256 manifest dict
        "warnings"      : list of pipeline-level warnings
    """
    _print = print if verbose else lambda *a, **k: None
    step_errors = {}
    pipeline_warnings = []

    # ── Validate dirs / labels ────────────────────────────────────
    if isinstance(dirs, (str, Path)):
        dirs = [dirs]
    if isinstance(labels, str):
        labels = [labels]
    dirs = list(dirs)
    labels = list(labels)
    if len(dirs) != len(labels):
        raise ValueError(
            f"dirs (length {len(dirs)}) and labels (length {len(labels)}) "
            f"must have the same length."
        )

    if save_outputs or plot:
        os.makedirs(out, exist_ok=True)

    # Master RNG (children spawned for sub-steps for reproducibility)
    master_rng = np.random.default_rng(random_seed)

    _print(f"\n{'='*72}")
    _print(f"  PSYCHIATRIC GENE-SET DIFFERENTIAL & PROXIMITY PIPELINE")
    _print(f"  Gene set         : {label}")
    _print(f"  Conditions       : {', '.join(labels)}")
    _print(f"  Output           : {out}/")
    _print(f"  Random seed      : {random_seed}")
    _print(f"  Robust mode      : {robust}")
    _print(f"  Bootstrap CIs    : {bootstrap}  (n_boot={n_boot})")
    _print(f"  n_perm           : {n_perm}")
    _print(f"  Family-wise FDR  : {family_wise_fdr}")
    _print(f"{'='*72}\n")

    # ── Compute provenance early ───────────────────────────────────
    try:
        provenance = compute_input_provenance(dirs, geneset)
        if save_outputs:
            with open(os.path.join(out, f"input_manifest_{label}.json"), "w") as fh:
                json.dump(provenance, fh, indent=2)
    except Exception as e:
        provenance = {"error": str(e)}
        pipeline_warnings.append(f"Provenance manifest failed: {e}")

    # ─── Step 1: Load disease data ─────────────────────────────────
    _print("[1/7] Loading S-PrediXcan meta-Z scores …")
    meta_z = {}
    id_maps = {}
    tissue_z = {}
    for tag, folder in zip(labels, dirs):
        _print(f"  {tag}:")
        mz, im, tz = load_spredixcan_folder(folder)
        meta_z[tag] = mz
        id_maps[tag] = im
        tissue_z[tag] = tz

    # Z-score sanity validation
    val_warnings = validate_zscores(meta_z)
    pipeline_warnings.extend(val_warnings)
    for w in val_warnings:
        _print(f"  ⚠ {w}")

    all_genes_union = set()
    for mz in meta_z.values():
        all_genes_union.update(mz.keys())
    _print(f"\n  Total unique genes across all diseases: {len(all_genes_union):,}")

    # ─── Step 2: Load gene set ─────────────────────────────────────
    _print(f"\n[2/7] Loading gene set …")
    gene_list = load_geneset(geneset)
    gene_list = harmonize_geneset(gene_list, meta_z, id_maps, verbose=verbose)

    found_any = sum(1 for g in gene_list if g in all_genes_union)
    _print(f"  {found_any}/{len(gene_list)} genes found in at least one disease dataset")

    if found_any == 0:
        raise ValueError(
            "No genes from the set were found in the data. "
            "Check gene names (should match gene_name column, case-insensitive)."
        )
    if found_any < 5:
        msg = (f"Only {found_any} gene(s) found across diseases — results "
               f"are EXPLORATORY. Differential / proximity tests may be unreliable.")
        pipeline_warnings.append(msg)
        _print(f"  ⚠ {msg}")

    # ─── Initialise default results ────────────────────────────────
    enrichment = {}
    enrich_df = pd.DataFrame()
    diff = {
        "n_common_genes": 0, "common_genes": [],
        "kruskal_h": np.nan, "kruskal_p": np.nan,
        "pairwise_df": pd.DataFrame()
    }
    prox_df = pd.DataFrame()
    profile_matrix = pd.DataFrame()
    pairwise_stats_df = pd.DataFrame()
    gene_detail = pd.DataFrame()
    influence_df = pd.DataFrame()
    family_wise_fdr_df = pd.DataFrame()

    # ─── Step 3: Enrichment ────────────────────────────────────────
    _print(f"\n[3/7] Computing per-disease enrichment "
           f"(perm={n_perm}, boot={'on' if bootstrap else 'off'}, robust={robust}) …")
    try:
        for d in labels:
            # Spawn distinct seed per disease for full reproducibility
            disease_seed = int(master_rng.integers(0, 2**31 - 1))
            enrichment[d] = gene_set_enrichment(
                meta_z[d], gene_list,
                n_perm=n_perm,
                random_seed=disease_seed,
                robust=robust,
                bootstrap=bootstrap,
                n_boot=n_boot,
            )
            info = enrichment[d]

            ci_str = ""
            if bootstrap and not np.isnan(info.get("stouffer_ci_low", np.nan)):
                ci_str = (f" CI95=[{info['stouffer_ci_low']}, "
                          f"{info['stouffer_ci_high']}]")
            _print(f"  {d}: {info['n_genes_found']}/{info['n_genes_total']} "
                   f"(coverage={info['coverage_pct']}%) | "
                   f"Stouffer Z = {info['stouffer_z']}{ci_str} | "
                   f"|Z| = {info['mean_abs_z']} | Perm p = {info['permutation_p']}")
            if info["low_coverage_warning"]:
                pipeline_warnings.append(
                    f"{d}: gene-set coverage {info['coverage_pct']}% < 60% — interpret cautiously")
            if info["low_n_warning"]:
                pipeline_warnings.append(
                    f"{d}: only {info['n_genes_found']} genes found — exploratory")

        enrich_rows = []
        for d in labels:
            row = {"Disease": d}
            row.update({k: v for k, v in enrichment[d].items() if k != "z_scores"})
            enrich_rows.append(row)
        enrich_df = pd.DataFrame(enrich_rows)

        if save_outputs:
            enrich_df.to_csv(os.path.join(out, f"enrichment_{label}.csv"), index=False)
            _print(f"  → {out}/enrichment_{label}.csv")
    except Exception as e:
        step_errors["enrichment"] = str(e)
        _print(f"  ⚠ Step 3 (Enrichment) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 4: Differential tests ────────────────────────────────
    _print(f"\n[4/7] Differential tests across diseases …")
    try:
        diff = differential_tests(meta_z, gene_list)
        _print(f"  Common genes in all {len(labels)} diseases: {diff['n_common_genes']}")
        _print(f"  Kruskal-Wallis H = {diff['kruskal_h']}, p = {diff['kruskal_p']}")

        if save_outputs and not diff["pairwise_df"].empty:
            pw_path = os.path.join(out, f"differential_pairwise_{label}.csv")
            diff["pairwise_df"].to_csv(pw_path, index=False)
            _print(f"  → {pw_path}")
        if verbose and not diff["pairwise_df"].empty:
            _print(diff["pairwise_df"].to_string(index=False))
    except Exception as e:
        step_errors["differential"] = str(e)
        _print(f"  ⚠ Step 4 (Differential tests) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 5: Proximity / distance ──────────────────────────────
    _print(f"\n[5/7] Profile proximity & distance …")
    try:
        prox_df, profile_matrix = profile_proximity(meta_z, gene_list)

        if save_outputs and not prox_df.empty:
            prox_path = os.path.join(out, f"proximity_{label}.csv")
            prox_df.to_csv(prox_path, index=False)
            _print(f"  → {prox_path}")
        if verbose and not prox_df.empty:
            _print(prox_df.to_string(index=False))
    except Exception as e:
        step_errors["proximity"] = str(e)
        _print(f"  ⚠ Step 5 (Proximity) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 6: Pairwise disease statistical tests ────────────────
    _print(f"\n[6/7] Pairwise disease statistical tests "
           f"(gene-set-restricted, robust={robust}) …")
    try:
        pw_seed = int(master_rng.integers(0, 2**31 - 1))
        pairwise_stats_df = pairwise_disease_geneset_tests(
            meta_z, gene_list, n_perm=n_perm,
            random_seed=pw_seed, robust=robust,
        )
        if not pairwise_stats_df.empty:
            _print(f"  {len(pairwise_stats_df)} disease pair(s) tested")
            for _, row in pairwise_stats_df.iterrows():
                conc_rate_str = (f"{row['sign_concordance_rate']:.3f}"
                                 if not pd.isna(row.get('sign_concordance_rate', np.nan))
                                 else "NA")
                conc_p_str = (f"{row['sign_concordance_p']:.4f}"
                              if not pd.isna(row.get('sign_concordance_p', np.nan))
                              else "NA")
                ccc_str = (f"{row['Lin_CCC']:.3f}"
                           if not pd.isna(row.get('Lin_CCC', np.nan)) else "NA")
                tau_str = (f"{row['Kendall_tau']:.3f}"
                           if not pd.isna(row.get('Kendall_tau', np.nan)) else "NA")
                _print(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"n_paired={row['n_genes_paired']}, "
                    f"mean_diff={row.get('mean_diff', 'NA')}, "
                    f"KS p={row.get('KS_p', 'NA')}, "
                    f"Welch p={row.get('Welch_p', 'NA')}, "
                    f"Paired-t p={row.get('Paired_t_p', 'NA')}, "
                    f"Cohen d(paired)={row.get('Cohens_d_paired', 'NA')}, "
                    f"SignConc={conc_rate_str} (p={conc_p_str}), "
                    f"CCC={ccc_str}, τ={tau_str}"
                )
            if save_outputs:
                pw_stats_path = os.path.join(out, f"pairwise_disease_stats_{label}.csv")
                pairwise_stats_df.to_csv(pw_stats_path, index=False)
                _print(f"  → {pw_stats_path}")
        else:
            _print("  (fewer than 2 diseases – skipped)")
    except Exception as e:
        step_errors["pairwise_stats"] = str(e)
        _print(f"  ⚠ Step 6 (Pairwise disease stats) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 7: Gene-level LOO influence analysis ─────────────────
    _print(f"\n[7/7] Gene-level leave-one-out influence analysis …")
    try:
        influence_df = gene_influence_analysis(
            meta_z, gene_list, threshold_pct=influence_threshold_pct
        )
        if not influence_df.empty:
            n_inf = int(influence_df["Influential"].sum())
            _print(f"  {n_inf} gene(s) flagged as influential "
                   f"(>|{influence_threshold_pct}%| change or sign-flip)")
            if save_outputs:
                inf_path = os.path.join(out, f"gene_influence_{label}.csv")
                influence_df.to_csv(inf_path, index=False)
                _print(f"  → {inf_path}")
        else:
            _print("  (insufficient data for LOO analysis)")
    except Exception as e:
        step_errors["influence"] = str(e)
        _print(f"  ⚠ Step 7 (Influence) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Per-gene detail ───────────────────────────────────────────
    try:
        gene_detail = build_gene_detail_table(meta_z, gene_list)
        if save_outputs:
            detail_path = os.path.join(out, f"gene_detail_{label}.csv")
            gene_detail.to_csv(detail_path, index=False)
            _print(f"  → {detail_path}")
    except Exception as e:
        step_errors["gene_detail"] = str(e)
        _print(f"  ⚠ Per-gene detail table FAILED: {e}")

    # ─── Optional: family-wise FDR across all p-values ─────────────
    if family_wise_fdr:
        try:
            fw_rows = []
            # Enrichment p's
            if not enrich_df.empty:
                for _, r in enrich_df.iterrows():
                    for col in ("wilcoxon_p", "permutation_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "enrichment", "Test": col,
                                "Comparison": r["Disease"], "p_value": float(v)})
            # Differential pairwise
            if isinstance(diff.get("pairwise_df"), pd.DataFrame) and not diff["pairwise_df"].empty:
                for _, r in diff["pairwise_df"].iterrows():
                    for col in ("MWU_p", "Paired_Wilcoxon_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "differential", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
                # Kruskal-Wallis
                if pd.notna(diff.get("kruskal_p", np.nan)):
                    fw_rows.append({
                        "Source": "differential", "Test": "kruskal_p",
                        "Comparison": "all_diseases",
                        "p_value": float(diff["kruskal_p"])})
            # Proximity p's
            if isinstance(prox_df, pd.DataFrame) and not prox_df.empty:
                for _, r in prox_df.iterrows():
                    for col in ("Pearson_p", "Spearman_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "proximity", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
            # Pairwise stats p's
            if isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
                p_cols = [c for c in pairwise_stats_df.columns if c.endswith("_p")]
                for _, r in pairwise_stats_df.iterrows():
                    for col in p_cols:
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "pairwise_stats", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
            family_wise_fdr_df = pd.DataFrame(fw_rows)
            if not family_wise_fdr_df.empty:
                family_wise_fdr_df["family_wise_FDR"] = benjamini_hochberg(
                    family_wise_fdr_df["p_value"].values
                )
                family_wise_fdr_df = family_wise_fdr_df.sort_values(
                    "p_value").reset_index(drop=True)
                if save_outputs:
                    fw_path = os.path.join(out, f"family_wise_fdr_{label}.csv")
                    family_wise_fdr_df.to_csv(fw_path, index=False)
                    _print(f"  → {fw_path}  ({len(family_wise_fdr_df)} p-values)")
        except Exception as e:
            step_errors["family_wise_fdr"] = str(e)
            _print(f"  ⚠ Family-wise FDR FAILED: {e}")

    # ─── Plots ─────────────────────────────────────────────────────
    if plot:
        _print(f"\nGenerating plots …")
        try:
            if enrichment:
                plot_zscore_boxplot(enrichment, label, out)
        except Exception as e:
            _print(f"  ⚠ Boxplot failed: {e}")

        try:
            if enrichment:
                plot_enrichment_bar(enrichment, label, out)
        except Exception as e:
            _print(f"  ⚠ Enrichment bar plot failed: {e}")

        try:
            if not prox_df.empty:
                plot_proximity_heatmap(prox_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Proximity heatmap failed: {e}")

        try:
            if not prox_df.empty:
                plot_proximity_dendrogram(prox_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Proximity dendrogram failed: {e}")

        try:
            if isinstance(profile_matrix, pd.DataFrame) and not profile_matrix.empty:
                plot_scatter_matrix(profile_matrix, label, out)
        except Exception as e:
            _print(f"  ⚠ Scatter matrix plot failed: {e}")

        try:
            if isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
                plot_pairwise_stats_heatmap(pairwise_stats_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Pairwise stats heatmap failed: {e}")

        try:
            if gene_list:
                plot_zscore_qq(meta_z, gene_list, label, out)
        except Exception as e:
            _print(f"  ⚠ QQ-plot failed: {e}")

        try:
            if isinstance(influence_df, pd.DataFrame) and not influence_df.empty:
                plot_gene_influence(influence_df, label, out)
        except Exception as e:
            _print(f"  ⚠ Gene-influence plot failed: {e}")

    # ─── Summary ───────────────────────────────────────────────────
    summary_text = ""
    try:
        lines = [
            f"{'='*72}",
            f"  GENE-SET ANALYSIS SUMMARY – {label}",
            f"  {len(gene_list)} genes in set, {found_any} found in data",
            f"  Conditions: {', '.join(labels)}",
            f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"{'='*72}",
            "",
            "  RUN PARAMETERS:",
            f"    Random seed         : {random_seed}",
            f"    Robust (winsorized) : {robust}",
            f"    Bootstrap CIs       : {bootstrap}  (n_boot={n_boot})",
            f"    Permutations (n)    : {n_perm}",
            f"    Family-wise FDR     : {family_wise_fdr}",
            f"    Influence threshold : ±{influence_threshold_pct}%",
            "",
            "  INPUT PROVENANCE (SHA256, first 16 hex):"
        ]
        try:
            for dir_key, dir_info in provenance.get("input_dirs", {}).items():
                lines.append(f"    [{dir_key}]")
                for fname, finfo in dir_info.get("files", {}).items():
                    lines.append(f"      {fname}: {finfo.get('sha256', '')[:16]}")
            gs_info = provenance.get("geneset_input")
            if isinstance(gs_info, dict) and "sha256" in gs_info:
                lines.append(f"    [geneset]: {gs_info['sha256'][:16]} "
                             f"({Path(gs_info.get('path','')).name})")
            elif isinstance(gs_info, dict):
                lines.append(f"    [geneset]: type={gs_info.get('type','?')}")
        except Exception:
            lines.append("    (provenance unavailable)")

        lines += ["", "  ENRICHMENT (per disease):"]
        if enrichment:
            for d in labels:
                e = enrichment.get(d, {})
                ci_blurb = ""
                if bootstrap and not pd.isna(e.get("stouffer_ci_low", np.nan)):
                    ci_blurb = (f" [CI95: {e['stouffer_ci_low']}, "
                                f"{e['stouffer_ci_high']}]")
                lines.append(
                    f"    {d}: n={e.get('n_genes_found','?')}/{e.get('n_genes_total','?')} "
                    f"(cov={e.get('coverage_pct','?')}%) | "
                    f"Stouffer Z={e.get('stouffer_z','?')}{ci_blurb} | "
                    f"|Z|={e.get('mean_abs_z','?')} | "
                    f"trimZ={e.get('trimmed_mean_z','?')} | "
                    f"Wilcoxon p={e.get('wilcoxon_p','?')} | "
                    f"Perm p={e.get('permutation_p','?')}"
                )
                flags = []
                if e.get("low_n_warning"): flags.append("LOW_N")
                if e.get("low_coverage_warning"): flags.append("LOW_COVERAGE")
                if e.get("wilcoxon_unreliable"): flags.append("WILCOXON<6")
                if flags:
                    lines.append(f"        ⚠ Flags: {', '.join(flags)}")
        else:
            lines.append("    (enrichment step failed)")

        if "enrichment" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['enrichment']}")

        lines += [
            "",
            "  DIFFERENTIAL TESTS:",
            f"    Common genes (all {len(labels)}): {diff['n_common_genes']}",
            f"    Kruskal-Wallis H={diff['kruskal_h']}, p={diff['kruskal_p']}"
        ]
        if "differential" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['differential']}")
        elif not diff["pairwise_df"].empty:
            for _, row in diff["pairwise_df"].iterrows():
                fdr_val = row.get("MWU_FDR", np.nan)
                fdr_str = f"{fdr_val:.4f}" if not pd.isna(fdr_val) else "NA"
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"MWU p={row['MWU_p']}, FDR={fdr_str}, "
                    f"mean-Z diff={row['Mean_Z_diff']}"
                )

        lines += ["", "  PROFILE PROXIMITY:"]
        if "proximity" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['proximity']}")
        elif not prox_df.empty:
            for _, row in prox_df.iterrows():
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"r={row['Pearson_r']}, rho={row['Spearman_rho']}, "
                    f"cos={row['Cosine_sim']}, Eucl={row['Euclidean_dist']}"
                )
        else:
            lines.append("    (insufficient gene overlap)")

        lines += ["", "  PAIRWISE DISEASE STAT TESTS (gene-set-restricted):"]
        if "pairwise_stats" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['pairwise_stats']}")
        elif isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
            for _, row in pairwise_stats_df.iterrows():
                cd_str = (f"{row['Cohens_d_paired']:.3f}"
                          if not pd.isna(row.get('Cohens_d_paired', np.nan)) else "NA")
                pt_str = (f"{row['Paired_t_p']:.4f}"
                          if not pd.isna(row.get('Paired_t_p', np.nan)) else "NA")
                ks_str = (f"{row['KS_p']:.4f}"
                          if not pd.isna(row.get('KS_p', np.nan)) else "NA")
                we_str = (f"{row['Welch_p']:.4f}"
                          if not pd.isna(row.get('Welch_p', np.nan)) else "NA")
                pm_str = (f"{row['Permutation_p']:.4f}"
                          if not pd.isna(row.get('Permutation_p', np.nan)) else "NA")
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']} "
                    f"(n_paired={row['n_genes_paired']}): "
                    f"mean_diff={row.get('mean_diff','NA')}, "
                    f"Cohen_d={cd_str}, "
                    f"KS_p={ks_str}, Welch_p={we_str}, "
                    f"Paired_t_p={pt_str}, Perm_p={pm_str}"
                )
            lines.append("")
            lines.append("  CONCORDANCE (gene-set-restricted, paired genes):")
            for _, row in pairwise_stats_df.iterrows():
                conc_rate = row.get("sign_concordance_rate", np.nan)
                conc_p    = row.get("sign_concordance_p", np.nan)
                n_conc    = row.get("n_concordant", np.nan)
                n_disc    = row.get("n_discordant", np.nan)
                ccc_val   = row.get("Lin_CCC", np.nan)
                tau_val   = row.get("Kendall_tau", np.nan)
                tau_p     = row.get("Kendall_p", np.nan)

                conc_rate_str = f"{conc_rate:.3f}" if not pd.isna(conc_rate) else "NA"
                conc_p_str    = f"{conc_p:.4f}"    if not pd.isna(conc_p)    else "NA"
                n_conc_str    = f"{int(n_conc)}"   if not pd.isna(n_conc)    else "NA"
                n_disc_str    = f"{int(n_disc)}"   if not pd.isna(n_disc)    else "NA"
                ccc_str       = f"{ccc_val:.3f}"   if not pd.isna(ccc_val)   else "NA"
                tau_str       = f"{tau_val:.3f}"   if not pd.isna(tau_val)   else "NA"
                tau_p_str     = f"{tau_p:.4f}"     if not pd.isna(tau_p)     else "NA"
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']} "
                    f"(n_paired={row['n_genes_paired']}): "
                    f"concordant={n_conc_str}, discordant={n_disc_str}, "
                    f"sign_conc_rate={conc_rate_str} (p={conc_p_str}), "
                    f"Lin_CCC={ccc_str}, "
                    f"Kendall_τ={tau_str} (p={tau_p_str})"
                )
        else:
            lines.append("    (fewer than 2 diseases or insufficient genes)")

        # ── Influence summary ──────────────────────────────────────
        lines += ["", "  GENE INFLUENCE (LOO):"]
        if "influence" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['influence']}")
        elif isinstance(influence_df, pd.DataFrame) and not influence_df.empty:
            n_total_inf = int(influence_df["Influential"].sum())
            lines.append(f"    Genes flagged as influential "
                         f"(|ΔStouffer|>{influence_threshold_pct}% or sign-flip): "
                         f"{n_total_inf}")
            for d in labels:
                sub = influence_df[influence_df["Disease"] == d]
                if sub.empty:
                    continue
                flagged = sub[sub["Influential"]]
                if not flagged.empty:
                    top = flagged.head(5)[["Gene", "Delta_Stouffer",
                                           "Pct_Change_Stouffer", "Sign_Change"]]
                    desc = ", ".join(
                        f"{r.Gene} (Δ={r.Delta_Stouffer:+.2f}, "
                        f"{r.Pct_Change_Stouffer:.1f}%"
                        f"{', SIGN' if r.Sign_Change else ''})"
                        for r in top.itertuples(index=False))
                    lines.append(f"    {d}: {desc}")
                else:
                    lines.append(f"    {d}: no genes exceed influence threshold")
        else:
            lines.append("    (insufficient data)")

        # ── Family-wise FDR section ────────────────────────────────
        if family_wise_fdr:
            lines += ["", "  FAMILY-WISE FDR (across all in-run p-values):"]
            if "family_wise_fdr" in step_errors:
                lines.append(f"    ⚠ STEP FAILED: {step_errors['family_wise_fdr']}")
            elif not family_wise_fdr_df.empty:
                n_total = len(family_wise_fdr_df)
                n_sig = int((family_wise_fdr_df["family_wise_FDR"] < 0.05).sum())
                lines.append(f"    Total p-values pooled: {n_total}")
                lines.append(f"    Significant after global FDR (<0.05): {n_sig}")
                top5 = family_wise_fdr_df.head(5)
                for _, r in top5.iterrows():
                    lines.append(
                        f"    {r['Source']}/{r['Test']} [{r['Comparison']}]: "
                        f"p={r['p_value']:.2e}, FDR={r['family_wise_FDR']:.4f}"
                    )
            else:
                lines.append("    (no p-values aggregated)")

        # ── Pipeline warnings ──────────────────────────────────────
        if pipeline_warnings:
            lines += ["", "  PIPELINE WARNINGS:"]
            for w in pipeline_warnings:
                lines.append(f"    ⚠ {w}")

        if step_errors:
            lines += ["", "  STEP ERRORS:"]
            for step_name, err_msg in step_errors.items():
                lines.append(f"    {step_name}: {err_msg}")

        # ── Limitations & interpretation block ─────────────────────
        lines += ["", _LIMITATIONS_BLOCK.rstrip(), "", f"{'='*72}"]

        summary_text = "\n".join(lines)
        _print(f"\n{summary_text}")

        if save_outputs:
            summary_path = os.path.join(out, f"summary_{label}.txt")
            with open(summary_path, "w") as fh:
                fh.write(summary_text)
    except Exception as e:
        summary_text = f"Summary generation failed: {e}"
        _print(f"  ⚠ Summary generation FAILED: {e}")

    _print(f"\n{'='*72}")
    if step_errors:
        _print(f"  DONE (with {len(step_errors)} step error(s)) – outputs in {out}/")
        for step_name, err_msg in step_errors.items():
            _print(f"    ⚠ {step_name}: {err_msg}")
    else:
        _print(f"  DONE – all outputs in {out}/")
    _print(f"{'='*72}\n")

    return {
        "meta_z": meta_z,
        "tissue_z": tissue_z,
        "gene_list": gene_list,
        "enrichment": enrichment,
        "enrichment_df": enrich_df,
        "differential": diff,
        "proximity_df": prox_df,
        "profile_matrix": profile_matrix,
        "pairwise_stats_df": pairwise_stats_df,
        "gene_detail_df": gene_detail,
        "influence_df": influence_df,
        "family_wise_fdr_df": family_wise_fdr_df,
        "summary_text": summary_text,
        "step_errors": step_errors,
        "warnings": pipeline_warnings,
        "provenance": provenance,
        "run_params": {
            "random_seed": random_seed,
            "robust": robust,
            "bootstrap": bootstrap,
            "n_boot": n_boot,
            "n_perm": n_perm,
            "family_wise_fdr": family_wise_fdr,
            "influence_threshold_pct": influence_threshold_pct,
        },
    }


def run_multi_geneset_pipeline(
    dirs,
    labels,
    gene_sets,
    out="results",
    n_perm=10000,
    tissue_n_perm=0,
    save_outputs=True,
    plot=True,
    verbose=True,
    # ── New robustness parameters (forwarded to run_pipeline) ─────
    random_seed=42,
    robust=False,
    bootstrap=False,
    n_boot=2000,
    family_wise_fdr=False,
    influence_threshold_pct=20.0,
):
    """
    Run the same pipeline over multiple gene sets.

    Same new parameters as run_pipeline; they are forwarded per-set.
    """
    _print = print if verbose else lambda *a, **k: None

    gene_sets = load_multiple_genesets(gene_sets)

    if save_outputs or plot:
        os.makedirs(out, exist_ok=True)

    all_results = {}
    summary_rows = []
    tissue_summary_rows = []
    gene_detail_frames = []
    pairwise_stats_frames = []
    influence_frames = []

    for gs_label, genes in gene_sets.items():
        safe_label = str(gs_label).replace(" ", "_")
        sub_out = os.path.join(out, safe_label)

        _print(f"\n{'#'*72}")
        _print(f"Running gene set: {gs_label}  (n={len(genes)})")
        _print(f"{'#'*72}")

        try:
            res = run_pipeline(
                dirs=dirs,
                labels=labels,
                geneset=genes,
                label=gs_label,
                out=sub_out,
                n_perm=n_perm,
                save_outputs=save_outputs,
                plot=plot,
                verbose=verbose,
                random_seed=random_seed,
                robust=robust,
                bootstrap=bootstrap,
                n_boot=n_boot,
                family_wise_fdr=family_wise_fdr,
                influence_threshold_pct=influence_threshold_pct,
            )
            all_results[gs_label] = res

            if isinstance(res.get("enrichment_df"), pd.DataFrame) and not res["enrichment_df"].empty:
                for _, row in res["enrichment_df"].iterrows():
                    summary_rows.append({
                        "GeneSet": gs_label,
                        "Disease": row["Disease"],
                        "Tissue": "meta_across_tissues",
                        "n_genes_found": row["n_genes_found"],
                        "n_genes_total": row["n_genes_total"],
                        "coverage_pct": row["coverage_pct"],
                        "stouffer_z": row["stouffer_z"],
                        "stouffer_ci_low":  row.get("stouffer_ci_low", np.nan),
                        "stouffer_ci_high": row.get("stouffer_ci_high", np.nan),
                        "mean_abs_z": row["mean_abs_z"],
                        "mean_abs_z_ci_low":  row.get("mean_abs_z_ci_low", np.nan),
                        "mean_abs_z_ci_high": row.get("mean_abs_z_ci_high", np.nan),
                        "mean_z": row["mean_z"],
                        "mean_z_ci_low":  row.get("mean_z_ci_low", np.nan),
                        "mean_z_ci_high": row.get("mean_z_ci_high", np.nan),
                        "trimmed_mean_z": row.get("trimmed_mean_z", np.nan),
                        "median_z": row["median_z"],
                        "wilcoxon_p": row["wilcoxon_p"],
                        "permutation_p": row["permutation_p"],
                        "low_n_warning":        row.get("low_n_warning", False),
                        "low_coverage_warning": row.get("low_coverage_warning", False),
                    })

            res_tissue_z = res.get("tissue_z", {})
            harmonized_genes = res.get("gene_list", [])
            for disease_label in labels:
                disease_tissues = res_tissue_z.get(disease_label, {})
                for tissue_name in sorted(disease_tissues.keys()):
                    tissue_dict = disease_tissues[tissue_name]
                    try:
                        te = gene_set_enrichment(
                            tissue_dict, harmonized_genes,
                            n_perm=tissue_n_perm,
                            random_seed=random_seed,
                            robust=robust,
                            bootstrap=False,  # keep speed for tissues
                        )
                        tissue_summary_rows.append({
                            "GeneSet": gs_label,
                            "Disease": disease_label,
                            "Tissue": tissue_name,
                            "n_genes_found": te["n_genes_found"],
                            "n_genes_total": te["n_genes_total"],
                            "coverage_pct": te["coverage_pct"],
                            "stouffer_z": te["stouffer_z"],
                            "mean_abs_z": te["mean_abs_z"],
                            "mean_z": te["mean_z"],
                            "trimmed_mean_z": te.get("trimmed_mean_z", np.nan),
                            "median_z": te["median_z"],
                            "wilcoxon_p": te["wilcoxon_p"],
                            "permutation_p": te["permutation_p"],
                        })
                    except Exception as te_err:
                        _print(f"  ⚠ Tissue enrichment failed for "
                               f"{disease_label}/{tissue_name}: {te_err}")

            pw_res = res.get("pairwise_stats_df", pd.DataFrame())
            if isinstance(pw_res, pd.DataFrame) and not pw_res.empty:
                pw_copy = pw_res.copy()
                pw_copy.insert(0, "GeneSet", gs_label)
                pairwise_stats_frames.append(pw_copy)

            inf_res = res.get("influence_df", pd.DataFrame())
            if isinstance(inf_res, pd.DataFrame) and not inf_res.empty:
                inf_copy = inf_res.copy()
                inf_copy.insert(0, "GeneSet", gs_label)
                influence_frames.append(inf_copy)

            if ("gene_detail_df" in res
                    and isinstance(res["gene_detail_df"], pd.DataFrame)
                    and not res["gene_detail_df"].empty):
                gdf = res["gene_detail_df"].copy()
                gdf.insert(0, "GeneSet", gs_label)

                for disease_label in labels:
                    disease_tissues = res_tissue_z.get(disease_label, {})
                    for tissue_name in sorted(disease_tissues.keys()):
                        col_name = f"Z_{disease_label}__{tissue_name}"
                        td = disease_tissues[tissue_name]
                        gdf[col_name] = gdf["Gene"].map(
                            lambda g, _td=td: round(_td[g], 4) if g in _td else np.nan
                        )

                gene_detail_frames.append(gdf)

        except Exception as e:
            _print(f"  ⚠ Failed for gene set '{gs_label}': {e}")
            all_results[gs_label] = {"error": str(e)}

    # ── Save enrichment summary (meta + tissue rows) ─────────────────
    if save_outputs and (summary_rows or tissue_summary_rows):
        all_enrich = summary_rows + tissue_summary_rows
        enrich_summary_df = pd.DataFrame(all_enrich)
        front = ["GeneSet", "Disease", "Tissue"]
        rest = [c for c in enrich_summary_df.columns if c not in front]
        enrich_summary_df = enrich_summary_df[front + rest]
        enrich_summary_df.sort_values(
            ["GeneSet", "Disease", "Tissue"], ignore_index=True, inplace=True
        )
        enrich_path = os.path.join(out, "multi_geneset_enrichment_summary.csv")
        enrich_summary_df.to_csv(enrich_path, index=False)
        _print(f"\n  → Enrichment summary (with by-tissue + bootstrap CI cols): {enrich_path}")
        n_meta = len(summary_rows)
        n_tiss = len(tissue_summary_rows)
        _print(f"    ({n_meta} meta-level rows + {n_tiss} tissue-level rows)")

    # ── Save combined pairwise disease stats (with global FDR) ───────
    combined_pairwise_stats_df = pd.DataFrame()
    if pairwise_stats_frames:
        combined_pairwise_stats_df = pd.concat(
            pairwise_stats_frames, ignore_index=True
        )
        p_cols = [c for c in combined_pairwise_stats_df.columns if c.endswith("_p")]
        for pc in p_cols:
            global_fdr_col = pc.replace("_p", "_globalFDR")
            vals = combined_pairwise_stats_df[pc].values.astype(float)
            mask = ~np.isnan(vals)
            if mask.sum() > 1:
                fdr_vals = np.full(len(vals), np.nan)
                fdr_vals[mask] = benjamini_hochberg(vals[mask])
                combined_pairwise_stats_df[global_fdr_col] = fdr_vals
            elif mask.sum() == 1:
                combined_pairwise_stats_df[global_fdr_col] = vals

        if save_outputs:
            pw_path = os.path.join(out, "multi_geneset_pairwise_disease_stats.csv")
            combined_pairwise_stats_df.to_csv(pw_path, index=False)
            _print(f"\n  → Pairwise disease stats (all gene sets): {pw_path}")
            _print(f"    ({len(combined_pairwise_stats_df)} rows across "
                   f"{combined_pairwise_stats_df['GeneSet'].nunique()} gene set(s))")

    all_results["_combined_pairwise_stats_df"] = combined_pairwise_stats_df

    # ── Combined influence table ─────────────────────────────────────
    combined_influence_df = pd.DataFrame()
    if influence_frames:
        combined_influence_df = pd.concat(influence_frames, ignore_index=True)
        if save_outputs:
            inf_path = os.path.join(out, "multi_geneset_gene_influence.csv")
            combined_influence_df.to_csv(inf_path, index=False)
            _print(f"\n  → Combined gene-influence (LOO) table: {inf_path}")
    all_results["_combined_influence_df"] = combined_influence_df

    # ── Combined gene-detail CSV ─────────────────────────────────────
    combined_gene_df = pd.DataFrame()
    if gene_detail_frames:
        combined_gene_df = pd.concat(gene_detail_frames, ignore_index=True)
        front_cols = ["GeneSet", "Gene"]
        meta_z_cols = [c for c in combined_gene_df.columns
                       if c.startswith("Z_") and "__" not in c
                       and c not in front_cols]
        tissue_z_cols = [c for c in combined_gene_df.columns
                         if c.startswith("Z_") and "__" in c]
        other_cols = [c for c in combined_gene_df.columns
                      if c not in front_cols + meta_z_cols + tissue_z_cols]
        combined_gene_df = combined_gene_df[
            front_cols + meta_z_cols + other_cols + tissue_z_cols
        ]
        if save_outputs:
            combined_path = os.path.join(out, "all_genes_all_genesets.csv")
            combined_gene_df.to_csv(combined_path, index=False)
            _print(f"\n  → Combined gene-level CSV (with by-tissue Z): {combined_path}")
            _print(f"    ({len(combined_gene_df)} rows across "
                   f"{combined_gene_df['GeneSet'].nunique()} gene set(s), "
                   f"{len(tissue_z_cols)} tissue Z columns)")

    all_results["_combined_gene_df"] = combined_gene_df

    # ── Top-level run-parameter manifest ─────────────────────────────
    if save_outputs:
        try:
            run_manifest = {
                "generated_at": datetime.now().isoformat(timespec="seconds"),
                "n_gene_sets": len(gene_sets),
                "labels": labels,
                "dirs": [str(d) for d in dirs],
                "params": {
                    "random_seed": random_seed,
                    "robust": robust,
                    "bootstrap": bootstrap,
                    "n_boot": n_boot,
                    "n_perm": n_perm,
                    "tissue_n_perm": tissue_n_perm,
                    "family_wise_fdr": family_wise_fdr,
                    "influence_threshold_pct": influence_threshold_pct,
                },
            }
            with open(os.path.join(out, "multi_geneset_run_manifest.json"), "w") as fh:
                json.dump(run_manifest, fh, indent=2)
        except Exception as e:
            _print(f"  ⚠ Failed to write run manifest: {e}")

    _print(f"\n{'='*72}")
    _print(f"  MULTI-GENESET PIPELINE COMPLETE – all outputs in {out}/")
    _print(f"{'='*72}\n")

    return all_results


# ===================================================================
#  9.  CLI ENTRY POINT
# ===================================================================

def main():
    """Thin CLI wrapper around run_pipeline()."""
    parser = argparse.ArgumentParser(
        description="Psychiatric Gene-Set Differential & Proximity Pipeline",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=textwrap.dedent("""\
        Examples:
          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --geneset my_genes.txt --label "Dopamine_Receptors"

          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --geneset DRD1,DRD2,DRD3,DRD4,DRD5,COMT --label "DA_Genes" \
              --bootstrap --robust --random_seed 1234

          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --genesets_file genesets.py --out results/ \
              --family_wise_fdr
        """)
    )
    parser.add_argument("--dirs", required=True, nargs="+",
                        help="Folders with S-PrediXcan results (one per condition)")
    parser.add_argument("--labels", required=True, nargs="+",
                        help="Labels for each folder (same order as --dirs)")
    parser.add_argument("--geneset", default=None,
                        help="Path to txt/tsv file OR comma-separated gene list")
    parser.add_argument("--genesets_file", default=None,
                        help="Path to file containing multiple gene sets")
    parser.add_argument("--label", default="CustomGeneSet",
                        help="Label for gene set (used in titles and filenames)")
    parser.add_argument("--n_perm", type=int, default=10000,
                        help="Permutations for enrichment p-value (default: 10000; "
                             "set to 0 to skip permutations entirely)")
    parser.add_argument("--tissue_n_perm", type=int, default=0,
                        help="Permutations for per-tissue enrichment (default: 0)")
    parser.add_argument("--out", default="results", help="Output directory")
    parser.add_argument("--no_plot", action="store_true", help="Skip plot generation")
    # ── New flags ──────────────────────────────────────────────────
    parser.add_argument("--random_seed", type=int, default=42,
                        help="Master RNG seed for reproducibility (default: 42)")
    parser.add_argument("--robust", action="store_true",
                        help="Winsorize Z-scores (1st/99th pct) before parametric stats")
    parser.add_argument("--bootstrap", action="store_true",
                        help="Compute 95% bootstrap CIs for Stouffer Z, mean Z, mean |Z|")
    parser.add_argument("--n_boot", type=int, default=2000,
                        help="Number of bootstrap resamples (default: 2000)")
    parser.add_argument("--family_wise_fdr", action="store_true",
                        help="Compute global FDR across ALL p-values in the run")
    parser.add_argument("--influence_threshold_pct", type=float, default=20.0,
                        help="LOO percent-change threshold for flagging influential genes")
    args = parser.parse_args()

    if len(args.dirs) != len(args.labels):
        parser.error(
            f"--dirs ({len(args.dirs)} items) and --labels ({len(args.labels)} items) "
            f"must have the same number of arguments."
        )

    if not args.geneset and not args.genesets_file:
        parser.error("Provide either --geneset or --genesets_file")
    if args.geneset and args.genesets_file:
        parser.error("Use only one of --geneset or --genesets_file")

    common_kwargs = dict(
        random_seed=args.random_seed,
        robust=args.robust,
        bootstrap=args.bootstrap,
        n_boot=args.n_boot,
        family_wise_fdr=args.family_wise_fdr,
        influence_threshold_pct=args.influence_threshold_pct,
    )

    try:
        if args.genesets_file:
            run_multi_geneset_pipeline(
                dirs=args.dirs,
                labels=args.labels,
                gene_sets=args.genesets_file,
                out=args.out,
                n_perm=args.n_perm,
                tissue_n_perm=args.tissue_n_perm,
                save_outputs=True,
                plot=not args.no_plot,
                verbose=True,
                **common_kwargs,
            )
        else:
            run_pipeline(
                dirs=args.dirs,
                labels=args.labels,
                geneset=args.geneset,
                label=args.label,
                out=args.out,
                n_perm=args.n_perm,
                save_outputs=True,
                plot=not args.no_plot,
                verbose=True,
                **common_kwargs,
            )
    except Exception as e:
        # Clean error report + traceback to file
        os.makedirs(args.out, exist_ok=True)
        err_path = os.path.join(args.out, "PIPELINE_ERROR.txt")
        with open(err_path, "w") as fh:
            fh.write(f"Pipeline failed at {datetime.now().isoformat(timespec='seconds')}\n")
            fh.write(f"Error: {e}\n\nTraceback:\n{traceback.format_exc()}")
        print(f"\n✗ Pipeline FAILED: {e}")
        print(f"  Full traceback written to: {err_path}")
        sys.exit(1)


# ===================================================================
#  10.  REPORT GENERATOR
# ===================================================================

def save_geneset_report(results, gene_set_name=None, output_dir=None):
    """
    Generate and save a full report from gene-set enrichment results.

    Multi-result support: if `results` is a dict of {label: single_run_result},
    reports are generated into subfolders.

    Outputs (additions in this update marked NEW):
    <output_dir>/
    ├── summary.txt
    ├── tables/
    │   ├── enrichment_summary.csv         (now includes bootstrap CI cols)
    │   ├── pairwise_differential.csv
    │   ├── pairwise_disease_stats.csv
    │   ├── profile_proximity.csv
    │   ├── profile_matrix.csv
    │   ├── gene_detail.csv
    │   ├── meta_z_geneset.csv
    │   ├── gene_influence.csv             (NEW)
    │   ├── family_wise_fdr.csv            (NEW; if computed)
    │   └── input_manifest.json            (NEW)
    └── plots/
        ├── 01_enrichment_overview.png/.pdf
        ├── 02_profile_heatmap.png/.pdf
        ├── 03_gene_barplot.png/.pdf
        ├── 04_pairwise_scatter.png/.pdf
        ├── 05_gene_z_range.png/.pdf
        ├── 06_z_distribution_boxplot.png/.pdf
        ├── 07_enrichment_metrics.png/.pdf
        ├── 08_pairwise_stats_heatmap.png/.pdf
        ├── 09_concordance_heatmap.png/.pdf
        ├── 10_qq_plot.png/.pdf            (NEW)
        ├── 11_dendrogram.png/.pdf         (NEW)
        └── 12_gene_influence.png/.pdf     (NEW)
    """
    # Multi-result support
    if "enrichment" not in results and all(isinstance(v, dict) for v in results.values()):
        out_dirs = {}
        base_dir = output_dir
        if base_dir is None:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            base_dir = f"./geneset_reports_{ts}"
        os.makedirs(base_dir, exist_ok=True)

        for label, res in results.items():
            if not isinstance(res, dict) or "enrichment" not in res:
                continue
            subdir = os.path.join(base_dir, str(label).replace(" ", "_"))
            out_dirs[label] = save_geneset_report(
                res, gene_set_name=label, output_dir=subdir
            )
        return out_dirs

    if gene_set_name is None:
        try:
            line = results["summary_text"].split("\n")[1]
            gene_set_name = line.split("–")[-1].strip() if "–" in line else "gene_set"
        except Exception:
            gene_set_name = "gene_set"

    safe_name = gene_set_name.replace(" ", "_")

    if output_dir is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = f"./geneset_report_{safe_name}_{ts}"

    tables_dir = os.path.join(output_dir, "tables")
    plots_dir  = os.path.join(output_dir, "plots")
    os.makedirs(tables_dir, exist_ok=True)
    os.makedirs(plots_dir,  exist_ok=True)

    diseases = list(results["enrichment"].keys()) if results.get("enrichment") else []
    edf      = results.get("enrichment_df", pd.DataFrame()).copy()

    _pal = sns.color_palette("Set2", max(len(diseases), 3))
    dcol = {d: _pal[i] for i, d in enumerate(diseases)}

    # ── 1. Summary text ───────────────────────────────────────────
    try:
        summary_path = os.path.join(output_dir, "summary.txt")
        with open(summary_path, "w") as fh:
            fh.write(results.get("summary_text", "(no summary available)"))
            fh.write(f"\n\nReport generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            fh.write(f"Output directory : {os.path.abspath(output_dir)}\n")
        print(f"[✓] summary.txt")
    except Exception as e:
        print(f"[⚠] summary.txt failed: {e}")

    # ── 2. Tables ─────────────────────────────────────────────────
    try:
        if not edf.empty:
            edf.to_csv(os.path.join(tables_dir, "enrichment_summary.csv"), index=False)
    except Exception as e:
        print(f"[⚠] enrichment_summary.csv failed: {e}")

    try:
        if "differential" in results and "pairwise_df" in results["differential"]:
            pw = results["differential"]["pairwise_df"]
            if isinstance(pw, pd.DataFrame) and not pw.empty:
                pw.to_csv(os.path.join(tables_dir, "pairwise_differential.csv"), index=False)
    except Exception as e:
        print(f"[⚠] pairwise_differential.csv failed: {e}")

    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty:
            pw_stats.to_csv(os.path.join(tables_dir, "pairwise_disease_stats.csv"), index=False)
    except Exception as e:
        print(f"[⚠] pairwise_disease_stats.csv failed: {e}")

    try:
        prox = results.get("proximity_df", pd.DataFrame())
        if isinstance(prox, pd.DataFrame) and not prox.empty:
            prox.to_csv(os.path.join(tables_dir, "profile_proximity.csv"), index=False)
    except Exception as e:
        print(f"[⚠] profile_proximity.csv failed: {e}")

    try:
        pm = results.get("profile_matrix", pd.DataFrame())
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            pm.to_csv(os.path.join(tables_dir, "profile_matrix.csv"))
    except Exception as e:
        print(f"[⚠] profile_matrix.csv failed: {e}")

    try:
        gd = results.get("gene_detail_df", pd.DataFrame())
        if isinstance(gd, pd.DataFrame) and not gd.empty:
            gd.to_csv(os.path.join(tables_dir, "gene_detail.csv"), index=False)
    except Exception as e:
        print(f"[⚠] gene_detail.csv failed: {e}")

    # NEW: gene influence table
    try:
        inf = results.get("influence_df", pd.DataFrame())
        if isinstance(inf, pd.DataFrame) and not inf.empty:
            inf.to_csv(os.path.join(tables_dir, "gene_influence.csv"), index=False)
    except Exception as e:
        print(f"[⚠] gene_influence.csv failed: {e}")

    # NEW: family-wise FDR
    try:
        fw = results.get("family_wise_fdr_df", pd.DataFrame())
        if isinstance(fw, pd.DataFrame) and not fw.empty:
            fw.to_csv(os.path.join(tables_dir, "family_wise_fdr.csv"), index=False)
    except Exception as e:
        print(f"[⚠] family_wise_fdr.csv failed: {e}")

    # NEW: input manifest
    try:
        prov = results.get("provenance", {})
        if prov:
            with open(os.path.join(tables_dir, "input_manifest.json"), "w") as fh:
                json.dump(prov, fh, indent=2)
    except Exception as e:
        print(f"[⚠] input_manifest.json failed: {e}")

    try:
        meta_rows = []
        for gene in results.get("gene_list", []):
            row = {"Gene": gene}
            for d in diseases:
                row[f"meta_z_{d}"] = results.get("meta_z", {}).get(d, {}).get(gene, np.nan)
            meta_rows.append(row)
        if meta_rows:
            pd.DataFrame(meta_rows).to_csv(
                os.path.join(tables_dir, "meta_z_geneset.csv"), index=False
            )
    except Exception as e:
        print(f"[⚠] meta_z_geneset.csv failed: {e}")

    n_table_files = len([f for f in os.listdir(tables_dir)]) if os.path.isdir(tables_dir) else 0
    print(f"[✓] tables/  ({n_table_files} files)")

    def _save(fig, stem):
        for ext in ("png", "pdf"):
            fig.savefig(
                os.path.join(plots_dir, f"{stem}.{ext}"),
                dpi=200, bbox_inches="tight",
            )
        plt.close(fig)

    pm = results.get("profile_matrix", pd.DataFrame())
    if isinstance(pm, pd.DataFrame) and not pm.empty:
        pm = pm.dropna()

    # ---- 01  Enrichment overview (Stouffer Z + −log10 perm-p) ----
    try:
        if not edf.empty and diseases:
            fig, axes = plt.subplots(1, 2, figsize=(max(12, len(diseases) * 3), 5))

            bars = axes[0].bar(
                edf["Disease"], edf["stouffer_z"],
                color=[dcol[d] for d in edf["Disease"]],
                edgecolor="black", linewidth=0.5,
            )
            axes[0].axhline(0, color="grey", lw=0.8, ls="--")
            for thr, c in [(1.96, "red"), (-1.96, "red")]:
                axes[0].axhline(thr, color=c, lw=0.6, ls=":", alpha=0.6)
            axes[0].set_ylabel("Stouffer Z")
            # Add CI error bars when available
            if "stouffer_ci_low" in edf.columns and edf["stouffer_ci_low"].notna().any():
                for b, lo, hi in zip(bars, edf["stouffer_ci_low"], edf["stouffer_ci_high"]):
                    if not (pd.isna(lo) or pd.isna(hi)):
                        axes[0].plot([b.get_x() + b.get_width()/2]*2, [lo, hi],
                                     color="black", lw=1.2)
                axes[0].set_title("Stouffer Z per Disease (95% bootstrap CI)")
            else:
                axes[0].set_title("Stouffer Z per Disease")
            for b, pv in zip(bars, edf["permutation_p"]):
                if pd.isna(pv):
                    continue
                sig = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else "ns"
                va  = "bottom" if b.get_height() >= 0 else "top"
                axes[0].text(
                    b.get_x() + b.get_width() / 2, b.get_height(),
                    f"p={pv:.4f}\n{sig}", ha="center", va=va, fontsize=9,
                )

            neg_log = -np.log10(edf["permutation_p"].replace(0, 1e-10).fillna(1.0))
            axes[1].bar(
                edf["Disease"], neg_log,
                color=[dcol[d] for d in edf["Disease"]],
                edgecolor="black", linewidth=0.5,
            )
            axes[1].axhline(-np.log10(0.05),  color="red",     lw=0.8, ls="--", label="p = 0.05")
            axes[1].axhline(-np.log10(0.001), color="darkred",  lw=0.8, ls=":",  label="p = 0.001")
            axes[1].set_ylabel("−log₁₀(permutation p)")
            axes[1].set_title("Enrichment Significance")
            axes[1].legend(fontsize=8)

            fig.suptitle(f"{gene_set_name} — Enrichment Overview", fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "01_enrichment_overview")
    except Exception as e:
        print(f"[⚠] Plot 01 (enrichment overview) failed: {e}")
        plt.close("all")

    # ---- 02  Heatmap of profile matrix --------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            vabs = max(abs(pm.values.min()), abs(pm.values.max()))
            fig, ax = plt.subplots(figsize=(max(6, len(diseases) * 2.5), max(6, len(pm) * 0.38)))
            sns.heatmap(
                pm, cmap="RdBu_r", center=0, vmin=-vabs, vmax=vabs,
                annot=True, fmt=".2f", linewidths=0.5, linecolor="white",
                cbar_kws={"label": "meta-Z"}, ax=ax,
            )
            ax.set_title(f"{gene_set_name} — Gene × Disease Z-score Heatmap",
                         fontsize=12, fontweight="bold")
            ax.set_ylabel("Gene"); ax.set_xlabel("Disease")
            plt.tight_layout()
            _save(fig, "02_profile_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 02 (profile heatmap) failed: {e}")
        plt.close("all")

    # ---- 03  Grouped bar chart ----------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            order = pm.loc[pm.abs().mean(axis=1).sort_values(ascending=True).index]
            fig, ax = plt.subplots(figsize=(max(9, len(order) * 0.55), 6))
            x = np.arange(len(order))
            w = 0.8 / len(diseases)
            for i, d in enumerate(diseases):
                off = (i - len(diseases) / 2 + 0.5) * w
                ax.bar(x + off, order[d], w, label=d,
                       color=dcol[d], edgecolor="black", linewidth=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels(order.index, rotation=45, ha="right", fontsize=8)
            ax.axhline(0, color="grey", lw=0.8)
            ax.set_ylabel("meta-Z score")
            ax.set_title(f"{gene_set_name} — Gene-level Z-scores by Disease",
                         fontsize=12, fontweight="bold")
            ax.legend(title="Disease")
            plt.tight_layout()
            _save(fig, "03_gene_barplot")
    except Exception as e:
        print(f"[⚠] Plot 03 (gene barplot) failed: {e}")
        plt.close("all")

    # ---- 04  Pairwise scatter ------------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty and len(diseases) >= 2:
            pairs  = list(combinations(diseases, 2))
            fig, axes_sc = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 5.5))
            if len(pairs) == 1:
                axes_sc = [axes_sc]
            prox = results.get("proximity_df", None)

            for idx, (d1, d2) in enumerate(pairs):
                ax = axes_sc[idx]
                xv, yv = pm[d1].values, pm[d2].values
                ax.scatter(xv, yv, c="steelblue", edgecolor="black", lw=0.3, s=50, alpha=0.8)
                for g, xi, yi in zip(pm.index, xv, yv):
                    ax.annotate(g, (xi, yi), fontsize=6, alpha=0.7,
                                xytext=(3, 3), textcoords="offset points")
                m = ~(np.isnan(xv) | np.isnan(yv))
                if m.sum() > 2:
                    coef = np.polyfit(xv[m], yv[m], 1)
                    xl   = np.linspace(xv[m].min(), xv[m].max(), 100)
                    ax.plot(xl, np.polyval(coef, xl), "r--", alpha=0.6, lw=1)
                if prox is not None and isinstance(prox, pd.DataFrame) and not prox.empty:
                    row = prox[((prox["Disease_1"] == d1) & (prox["Disease_2"] == d2)) |
                               ((prox["Disease_1"] == d2) & (prox["Disease_2"] == d1))]
                    if not row.empty:
                        r = row["Pearson_r"].values[0]; p = row["Pearson_p"].values[0]
                        ax.set_title(f"{d1} vs {d2}\nr = {r:.3f}, p = {p:.4f}", fontsize=11)
                    else:
                        ax.set_title(f"{d1} vs {d2}")
                else:
                    ax.set_title(f"{d1} vs {d2}")
                ax.set_xlabel(f"{d1} meta-Z"); ax.set_ylabel(f"{d2} meta-Z")
                ax.axhline(0, color="grey", lw=0.5, ls="--")
                ax.axvline(0, color="grey", lw=0.5, ls="--")

            fig.suptitle(f"{gene_set_name} — Pairwise Z-score Correlations",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "04_pairwise_scatter")
    except Exception as e:
        print(f"[⚠] Plot 04 (pairwise scatter) failed: {e}")
        plt.close("all")

    # ---- 05  Z-range lollipop -----------------------------------
    try:
        gdf = results.get("gene_detail_df", pd.DataFrame())
        if isinstance(gdf, pd.DataFrame) and not gdf.empty and "Z_range" in gdf.columns:
            gdf = gdf.dropna(subset=["Z_range"]).sort_values("Z_range", ascending=False)
            if not gdf.empty:
                fig, ax = plt.subplots(figsize=(8, max(5, len(gdf) * 0.38)))
                y = np.arange(len(gdf))
                ax.barh(y, gdf["Z_range"], color="salmon", edgecolor="black", lw=0.3, height=0.6)
                ax.set_yticks(y)
                ax.set_yticklabels(gdf["Gene"], fontsize=9)
                ax.set_xlabel("Z-score range (max − min across diseases)")
                ax.set_title(f"{gene_set_name} — Cross-Disease Variability per Gene",
                             fontsize=12, fontweight="bold")
                ax.invert_yaxis()
                plt.tight_layout()
                _save(fig, "05_gene_z_range")
    except Exception as e:
        print(f"[⚠] Plot 05 (gene z-range) failed: {e}")
        plt.close("all")

    # ---- 06  Box + strip plot -----------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            fig, ax = plt.subplots(figsize=(max(6, len(diseases) * 2.2), 5.5))
            bp = ax.boxplot(
                [pm[d].values for d in diseases],
                labels=diseases, patch_artist=True, showmeans=True,
                meanprops=dict(marker="D", markerfacecolor="red", markersize=6),
            )
            for patch, d in zip(bp["boxes"], diseases):
                patch.set_facecolor(dcol[d]); patch.set_alpha(0.55)
            for i, d in enumerate(diseases):
                jitter = np.random.default_rng(42).normal(0, 0.04, len(pm))
                ax.scatter(
                    np.full(len(pm), i + 1) + jitter, pm[d].values,
                    c="black", s=18, alpha=0.5, zorder=3,
                )
            ax.axhline(0, color="grey", lw=0.8, ls="--")
            ax.set_ylabel("meta-Z score")
            ax.set_title(f"{gene_set_name} — Z-score Distributions",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            _save(fig, "06_z_distribution_boxplot")
    except Exception as e:
        print(f"[⚠] Plot 06 (z-distribution boxplot) failed: {e}")
        plt.close("all")

    # ---- 07  Enrichment metrics comparison -----------------------
    try:
        if len(diseases) >= 2 and not edf.empty:
            metrics = ["stouffer_z", "mean_z", "median_z", "mean_abs_z"]
            metric_labels = ["Stouffer Z", "Mean Z", "Median Z", "Mean |Z|"]
            fig, ax = plt.subplots(figsize=(max(10, len(diseases) * 2.5), 5))
            x = np.arange(len(metrics))
            w = 0.8 / len(diseases)
            for i, d in enumerate(diseases):
                row  = edf[edf["Disease"] == d].iloc[0]
                vals = [float(row[m]) for m in metrics]
                off  = (i - len(diseases) / 2 + 0.5) * w
                ax.bar(x + off, vals, w, label=d,
                       color=dcol[d], edgecolor="black", linewidth=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels(metric_labels, fontsize=10)
            ax.axhline(0, color="grey", lw=0.8, ls="--")
            ax.set_ylabel("Value")
            ax.set_title(f"{gene_set_name} — Enrichment Metrics Comparison",
                         fontsize=12, fontweight="bold")
            ax.legend(title="Disease")
            plt.tight_layout()
            _save(fig, "07_enrichment_metrics")
    except Exception as e:
        print(f"[⚠] Plot 07 (enrichment metrics) failed: {e}")
        plt.close("all")

    # ---- 08  Pairwise stats heatmap -----------------------------
    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty and len(diseases) >= 2:
            p_candidates = [
                ("KS_p",               "KS 2-sample"),
                ("Welch_p",            "Welch t"),
                ("Paired_Wilcoxon_p",  "Paired Wilcoxon"),
                ("Paired_t_p",         "Paired t"),
                ("Permutation_p",      "Permutation"),
            ]
            p_metrics = [(col, lbl) for col, lbl in p_candidates
                         if col in pw_stats.columns and pw_stats[col].notna().any()]
            d_candidates = [
                ("Cohens_d_unpaired", "Cohen's d (unpaired)"),
                ("Cohens_d_paired",   "Cohen's d (paired)"),
            ]
            d_metrics = [(col, lbl) for col, lbl in d_candidates
                         if col in pw_stats.columns and pw_stats[col].notna().any()]
            n_panels = len(p_metrics) + len(d_metrics)
            if n_panels > 0:
                fig, axes_pw = plt.subplots(
                    1, n_panels,
                    figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
                )
                if n_panels == 1:
                    axes_pw = [axes_pw]
                panel_idx = 0
                for col, lbl in p_metrics:
                    ax = axes_pw[panel_idx]; panel_idx += 1
                    mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            neg_log = -np.log10(max(val, 1e-300))
                            mat.loc[d1, d2] = neg_log
                            mat.loc[d2, d1] = neg_log
                    for d in diseases:
                        mat.loc[d, d] = 0.0
                    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                                square=True, ax=ax, linewidths=0.5, linecolor="white", vmin=0)
                    ax.set_title(f"−log₁₀(p)\n{lbl}", fontsize=10)
                for col, lbl in d_metrics:
                    ax = axes_pw[panel_idx]; panel_idx += 1
                    mat = pd.DataFrame(0.0, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            mat.loc[d1, d2] =  val
                            mat.loc[d2, d1] = -val
                    vabs = max(abs(mat.values.min()), abs(mat.values.max()), 0.01)
                    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
                                center=0, vmin=-vabs, vmax=vabs, square=True, ax=ax,
                                linewidths=0.5, linecolor="white")
                    ax.set_title(lbl, fontsize=10)
                fig.suptitle(f"{gene_set_name} — Pairwise Disease Stat Tests",
                             fontsize=13, fontweight="bold")
                plt.tight_layout()
                _save(fig, "08_pairwise_stats_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 08 (pairwise stats heatmap) failed: {e}")
        plt.close("all")

    # ---- 09  Concordance heatmap --------------------------------
    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty and len(diseases) >= 2:
            conc_candidates = [
                ("sign_concordance_rate", "Sign Concordance Rate", "YlGn",    0.0, 1.0, 0.5),
                ("Lin_CCC",               "Lin's CCC",            "RdBu_r", -1.0, 1.0, 0.0),
                ("Kendall_tau",            "Kendall τ",            "RdBu_r", -1.0, 1.0, 0.0),
            ]
            conc_metrics_avail = [
                (col, lbl, cmap, vmin, vmax, center)
                for col, lbl, cmap, vmin, vmax, center in conc_candidates
                if col in pw_stats.columns and pw_stats[col].notna().any()
            ]
            conc_p_avail = []
            if "sign_concordance_p" in pw_stats.columns and pw_stats["sign_concordance_p"].notna().any():
                conc_p_avail.append(("sign_concordance_p", "Sign Conc. −log₁₀(p)"))
            if "Kendall_p" in pw_stats.columns and pw_stats["Kendall_p"].notna().any():
                conc_p_avail.append(("Kendall_p", "Kendall τ −log₁₀(p)"))
            n_panels = len(conc_metrics_avail) + len(conc_p_avail)
            if n_panels > 0:
                fig, axes_c = plt.subplots(
                    1, n_panels,
                    figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
                )
                if n_panels == 1:
                    axes_c = [axes_c]
                pi = 0
                for col, lbl, cmap, vmin_val, vmax_val, center_val in conc_metrics_avail:
                    ax = axes_c[pi]; pi += 1
                    mat = pd.DataFrame(1.0, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            mat.loc[d1, d2] = val
                            mat.loc[d2, d1] = val
                    sns.heatmap(
                        mat.astype(float), annot=True, fmt=".3f", cmap=cmap,
                        center=center_val, vmin=vmin_val, vmax=vmax_val,
                        square=True, ax=ax, linewidths=0.5, linecolor="white",
                    )
                    ax.set_title(lbl, fontsize=10)
                for col, lbl in conc_p_avail:
                    ax = axes_c[pi]; pi += 1
                    mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            neg_log = -np.log10(max(val, 1e-300))
                            mat.loc[d1, d2] = neg_log
                            mat.loc[d2, d1] = neg_log
                    for d in diseases:
                        mat.loc[d, d] = 0.0
                    sns.heatmap(
                        mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                        square=True, ax=ax, linewidths=0.5, linecolor="white", vmin=0,
                    )
                    ax.set_title(lbl, fontsize=10)
                fig.suptitle(f"{gene_set_name} — Concordance Metrics",
                             fontsize=13, fontweight="bold")
                plt.tight_layout()
                _save(fig, "09_concordance_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 09 (concordance heatmap) failed: {e}")
        plt.close("all")

    # ---- 10  QQ-plot (NEW) --------------------------------------
    try:
        meta_z = results.get("meta_z", {})
        gene_list = results.get("gene_list", [])
        if meta_z and gene_list:
            diseases_local = list(meta_z.keys())
            n = len(diseases_local)
            fig, axes = plt.subplots(1, n, figsize=(max(5 * n, 6), 5))
            if n == 1:
                axes = [axes]
            for ax, d in zip(axes, diseases_local):
                gs_z = np.array([meta_z[d].get(g, np.nan) for g in gene_list])
                gs_z = gs_z[~np.isnan(gs_z)]
                if len(gs_z) < 3:
                    ax.text(0.5, 0.5, f"{d}: n<3", ha="center", va="center",
                            transform=ax.transAxes); continue
                observed = np.sort(gs_z)
                theoretical = stats.norm.ppf((np.arange(1, len(observed) + 1) - 0.5) / len(observed))
                ax.scatter(theoretical, observed, alpha=0.7, s=30,
                           c=dcol.get(d, "steelblue"), edgecolor="black", lw=0.3)
                lims = [min(theoretical.min(), observed.min()),
                        max(theoretical.max(), observed.max())]
                ax.plot(lims, lims, "r--", lw=1, label="N(0,1) reference")
                ax.axhline(0, color="gray", lw=0.5, ls=":")
                ax.set_xlabel("Theoretical N(0,1) quantile")
                ax.set_ylabel("Observed gene-set Z")
                ax.set_title(f"{d} (n={len(observed)})", fontsize=11)
                ax.legend(fontsize=8)
            fig.suptitle(f"{gene_set_name} — QQ-plot vs N(0,1)",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "10_qq_plot")
    except Exception as e:
        print(f"[⚠] Plot 10 (QQ-plot) failed: {e}")
        plt.close("all")

    # ---- 11  Dendrogram (NEW) ------------------------------------
    try:
        prox = results.get("proximity_df", pd.DataFrame())
        if isinstance(prox, pd.DataFrame) and not prox.empty and len(diseases) >= 3:
            from scipy.cluster.hierarchy import linkage, dendrogram
            from scipy.spatial.distance import squareform
            n = len(diseases)
            dmat = np.zeros((n, n))
            idx = {d: i for i, d in enumerate(diseases)}
            for _, row in prox.iterrows():
                d1, d2 = row["Disease_1"], row["Disease_2"]
                if d1 in idx and d2 in idx:
                    dist = max(0.0, 1 - float(row["Pearson_r"]))
                    dmat[idx[d1], idx[d2]] = dist
                    dmat[idx[d2], idx[d1]] = dist
            condensed = squareform(dmat, checks=False)
            Z = linkage(condensed, method="average")
            fig, ax = plt.subplots(figsize=(max(6, n * 1.3), 5))
            dendrogram(Z, labels=diseases, ax=ax, color_threshold=0)
            ax.set_ylabel("Distance (1 − Pearson r)")
            ax.set_title(f"{gene_set_name} — Disease Hierarchical Clustering",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            _save(fig, "11_dendrogram")
    except Exception as e:
        print(f"[⚠] Plot 11 (dendrogram) failed: {e}")
        plt.close("all")

    # ---- 12  Gene influence (NEW) --------------------------------
    try:
        inf = results.get("influence_df", pd.DataFrame())
        if isinstance(inf, pd.DataFrame) and not inf.empty:
            diseases_local = list(inf["Disease"].unique())
            n = len(diseases_local)
            fig, axes = plt.subplots(1, n, figsize=(max(6 * n, 8), max(5, 15 * 0.35)))
            if n == 1:
                axes = [axes]
            for ax, d in zip(axes, diseases_local):
                sub = inf[inf["Disease"] == d].copy()
                sub = sub.reindex(sub["Delta_Stouffer"].abs().sort_values(ascending=False).index)
                sub = sub.head(15).iloc[::-1]
                if sub.empty:
                    continue
                colors = ["#e74c3c" if x else "#3498db" for x in sub["Influential"]]
                ax.barh(sub["Gene"], sub["Delta_Stouffer"], color=colors,
                        edgecolor="black", lw=0.3)
                ax.axvline(0, color="gray", ls="--", lw=0.8)
                ax.set_xlabel("Δ Stouffer Z (full − LOO)")
                ax.set_title(f"{d}: top influential genes", fontsize=11)
            fig.suptitle(f"{gene_set_name} — Gene-level LOO Influence",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "12_gene_influence")
    except Exception as e:
        print(f"[⚠] Plot 12 (gene influence) failed: {e}")
        plt.close("all")

    n_plots = len([f for f in os.listdir(plots_dir) if f.endswith(".png")]) if os.path.isdir(plots_dir) else 0
    print(f"[✓] plots/   ({n_plots} PNGs + {n_plots} PDFs)")
    print(f"\n{'='*60}")
    print(f"  Report saved → {os.path.abspath(output_dir)}/")
    print(f"{'='*60}")

    return output_dir



### Loading

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/bip" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/migraine" /content/ -r

### Gene sets


#### Newly added gene sets

In [ ]:
HALLMARK_INFLAMMATORY_RESPONSE = [
    "IL1B", "IL6", "TNF", "IL1A", "IL8", "CXCL1", "CXCL2", "CXCL3",
    "CCL2", "CCL3", "CCL4", "CCL5", "NFKB1", "RELA", "STAT1", "STAT3",
    "JUN", "FOS", "PTGS2", "NOS2", "ICAM1", "VCAM1", "SELE"
]

REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM = [
    "IL6", "IL6R", "IL1B", "IL1R1", "TNF", "TNFRSF1A", "TNFRSF1B",
    "IFNG", "IFNGR1", "IFNGR2", "IL2", "IL2RA", "IL4", "IL4R", "IL10",
    "IL10RA", "IL12A", "IL12B", "IL17A", "IL17RA", "CXCL8", "CXCL10",
    "CCL2", "CCL5"
]

HALLMARK_TNFA_SIGNALING_VIA_NFKB = [
    "TNF", "IL1B", "IL6", "NFKB1", "RELA", "IKBKB", "CHUK", "NFKBIA",
    "NFKBIB", "RELB", "NFKB2", "ICAM1", "VCAM1", "SELE", "PTGS2",
    "NOS2", "JUN", "FOS", "ATF3", "EGR1", "MYC", "CCL2", "CCL5", "CXCL8"
]

REACTOME_CIRCADIAN_CLOCK = [
    "CLOCK", "ARNTL", "PER1", "PER2", "PER3", "CRY1", "CRY2", "TIMELESS",
    "CSNK1E", "CSNK1D", "NR1D1", "NR1D2", "RORA", "RORB", "RORC",
    "NPAS2", "TEF", "DBP", "HLF"
]

KEGG_DOPAMINERGIC_SYNAPSE = [
    "DRD1", "DRD2", "DRD3", "DRD4", "DRD5", "TH", "DDC", "MAOA", "MAOB",
    "COMT", "SLC6A3", "SLC18A2", "SNAP25", "STX1A", "VAMP2", "SYN1",
    "SYN2", "PPP1R1B", "PPP2CA", "PPP2CB", "PPP3CA", "PPP3CB", "PPP3CC"
]

REACTOME_NEUROTRANSMITTER_RELEASE_CYCLE = [
    "SNAP25", "STX1A", "VAMP2", "SYN1", "SYN2", "SYT1", "SYT2", "CPLX1",
    "CPLX2", "RAB3A", "RAB3B", "UNC13A", "UNC13B", "STXBP1", "SLC17A6",
    "SLC17A7", "SLC32A1", "SLC6A1", "SLC6A11"
]

REACTOME_MITOPHAGY = [
    "PINK1", "PARK2", "PARK7", "OPTN", "SQSTM1", "CALCOCO2", "TAX1BP1",
    "NBR1", "ATG5", "ATG7", "ATG12", "ATG16L1", "MAP1LC3A", "MAP1LC3B",
    "GABARAP", "GABARAPL1", "GABARAPL2", "ULK1", "ULK2"
]

GOBP_POSITIVE_REGULATION_OF_MICROGLIAL_ACTIVATION = [
    "CSF1", "CSF1R", "IL1B", "IL6", "TNF", "CX3CL1", "CX3CR1", "P2RY12",
    "TREM2", "APOE", "SPP1", "ITGAM", "ITGAX", "CD68", "AIF1", "TLR2",
    "TLR4", "NLRP3"
]

REACTOME_ACTIVATION_OF_INNATE_IMMUNE_RESPONSE = [
    "TLR2", "TLR4", "TLR9", "MYD88", "IRAK1", "IRAK4", "TRAF6", "IKBKB",
    "CHUK", "NFKB1", "RELA", "IFIH1", "DDX58", "MAVS", "STING1", "CGAS",
    "NLRP3", "PYCARD", "CASP1", "IL1B", "IL18"
]

#### Gene Sets GLP-related

In [ ]:
GOBP_INSULIN_SECRETION_INVOLVED_IN_CELLULAR_RESPONSE_TO_GLUCOSE_STIMULUS_ALL = [
    "ABCA12","ABCC8","ADCY5","ADCY8","ADRA2A","ALDOA","BAD","BAIAP3","BRSK2",
    "C1QTNF12","C2CD2L","CAMP","CARTPT","CCDC186","CDK16","CLTRN","CRH",
    "DYNLL1","EFNA5","EPHA5","FKBP1B","FOXA2","GCG","GHRL","GNAS","GPLD1",
    "GPR27","GPR68","GPRC6A","HIF1A","HLA-DRB1","JAGN1","KCNK16","KLF7",
    "LRP5","MLXIPL","MPC2","MRGPRE","NDUFAF2","NR1D1","NR1H4","OSBP","OXCT1",
    "PDE8B","PDX1","PHPT1","PIM3","PLA2G6","PPARD","PPP3CB","PRKCE","PRKN",
    "PTPRN","PTPRN2","RAB11B","RAB11FIP2","RAB11FIP5","RAC1","RAF1","RBM4",
    "RFX6","SELENOT","SIDT2","SLC2A2","SLC9B2","SRI","STX4","STXBP4","TRPA1",
    "TRPM4","TRPM5","TUNAR","UCP2","VSNL1","ZBED6"
]

REACTOME_REGULATION_OF_INSULIN_SECRETION_ALL = [
    "ABCC8","ACSL3","ACSL4","ADCY5","ADCY6","ADCY8","ADRA2A","ADRA2C",
    "AHCYL1","AKAP5","CACNA1A","CACNA1C","CACNA1D","CACNA1E","CACNA2D2",
    "CACNB2","CACNB3","CD36","CHRM3","FFAR1","GCG","GLP1R","GNA11","GNA14",
    "GNA15","GNAI1","GNAI2","GNAQ","GNAS","GNB1","GNB2","GNB3","GNB4","GNB5",
    "GNG10","GNG11","GNG12","GNG13","GNG2","GNG3","GNG4","GNG5","GNG7","GNG8",
    "GNGT1","GNGT2","INS","IQGAP1","ITPR1","ITPR2","ITPR3","KCNB1","KCNC2",
    "KCNG2","KCNJ11","KCNS3","MARCKS","PLCB1","PLCB2","PLCB3","PRKACA",
    "PRKACB","PRKACG","PRKAR1A","PRKAR1B","PRKAR2A","PRKAR2B","PRKCA",
    "RAP1A","RAPGEF3","RAPGEF4","SLC2A1","SLC2A2","SNAP25","STX1A","STXBP1",
    "SYT5","VAMP2"
]

GOBP_REGULATION_OF_CELLULAR_SENESCENCE_ALL = [
    "ABI3","ABL1","AKT3","ARG2","B2M","BCL2L12","BCL6","BMAL1","BMPR1A",
    "CDK6","CGAS","EEF1E1","FBXO5","FZR1","HLA-G","HMGA2","ING2","KIR2DL4",
    "KRAS","MIF","MIR10A","MIR146A","MIR17","MIR20B","MIR217","MIR22",
    "MIR34A","MIR543","MIR590","MORC3","NEK4","NEK6","NUAK1","PAWR","PLK2",
    "PNPT1","PTEN","RBL1","RSL1D1","SIRT1","SIRT6","SUV39H1","TBX2","TERC",
    "TERF2","TERT","TP53","TP63","TWIST1","VASH1","WNT1","YBX1","YPEL3",
    "ZKSCAN3","ZMPSTE24"
]

GOBP_CELLULAR_SENESCENCE_ALL = [
    "ABI3","ABL1","AKT3","ARG2","ATM","B2M","BCL2L12","BCL6","BMAL1",
    "BMPR1A","BRCA2","CALR","CDK2","CDK6","CDKN1A","CDKN1B","CDKN2A",
    "CDKN2B","CGAS","COMP","DNAJA3","ECRG4","EEF1E1","FBXO4","FBXO5","FZR1",
    "GINS3","HLA-G","HMGA1","HMGA2","HRAS","ID2","ING2","KAT5","KAT6A",
    "KIR2DL4","KRAS","LMNA","MAGEA2","MAGEA2B","MAP2K1","MAP2K3","MAP2K4",
    "MAP2K6","MAP2K7","MAP3K5","MAPK10","MAPK11","MAPK14","MAPK8","MAPK9",
    "MAPKAPK5","MIF","MIR10A","MIR146A","MIR17","MIR20B","MIR217","MIR22",
    "MIR34A","MIR543","MIR590","MNT","MORC3","NDUFS6","NEK4","NEK6","NPM1",
    "NSMCE2","NUAK1","NUP62","OPA1","PAWR","PLA2R1","PLK2","PML","PNPT1",
    "PRELP","PRKCD","PRMT6","PTEN","RBL1","RSL1D1","SIRT1","SIRT6","SMC5",
    "SMC6","SPI1","SRF","SUV39H1","TBX2","TBX3","TERC","TERF2","TERT",
    "TOP2B","TP53","TP63","TWIST1","ULK3","VASH1","WNT1","WNT16","WRN",
    "YBX1","YPEL3","ZKSCAN3","ZMIZ1","ZMPSTE24"
]

REACTOME_CELLULAR_SENESCENCE_ALL = [
    "ACD","AGO1","AGO3","AGO4","ANAPC1","ANAPC10","ANAPC11","ANAPC15",
    "ANAPC16","ANAPC2","ANAPC4","ANAPC5","ANAPC7","ASF1A","ATM","BMI1",
    "CABIN1","CBX2","CBX4","CBX6","CBX8","CCNA1","CCNA2","CCNE1","CCNE2",
    "CDC16","CDC23","CDC26","CDC27","CDK2","CDK4","CDK6","CDKN1A","CDKN1B",
    "CDKN2A","CDKN2B","CDKN2C","CDKN2D","CEBPB","CXCL8","E2F1","E2F2",
    "E2F3","EED","EHMT1","EHMT2","EP400","ERF","ETS1","ETS2","EZH2","FOS",
    "FZR1","H1-0","H1-1","H1-2","H1-3","H1-4","H1-5","H2AB1","H2AC14",
    "H2AC18","H2AC19","H2AC20","H2AC4","H2AC6","H2AC7","H2AC8","H2AJ",
    "H2AX","H2AZ2","H2BC1","H2BC10","H2BC11","H2BC12","H2BC12L","H2BC13",
    "H2BC14","H2BC15","H2BC17","H2BC21","H2BC26","H2BC3","H2BC4","H2BC5",
    "H2BC6","H2BC7","H2BC8","H2BC9","H3-3A","H3-3B","H3-4","H3C1","H3C10",
    "H3C11","H3C12","H3C13","H3C14","H3C15","H3C2","H3C3","H3C4","H3C6",
    "H3C7","H3C8","H4C1","H4C11","H4C12","H4C13","H4C14","H4C15","H4C16",
    "H4C2","H4C3","H4C4","H4C5","H4C6","H4C8","H4C9","HIRA","HMGA1","HMGA2",
    "ID1","IFNB1","IGFBP7","IL1A","IL6","JUN","KAT5","KDM6B","LMNB1",
    "MAP2K3","MAP2K4","MAP2K6","MAP2K7","MAP3K5","MAP4K4","MAPK1","MAPK10",
    "MAPK11","MAPK14","MAPK3","MAPK7","MAPK8","MAPK9","MAPKAPK2","MAPKAPK3",
    "MAPKAPK5","MDM2","MDM4","MINK1","MIR24-1","MIR24-2","MOV10","MRE11",
    "NBN","NFKB1","PHC1","PHC2","PHC3","POT1","RAD50","RB1","RBBP4","RBBP7",
    "RELA","RING1","RNF2","RPS27A","RPS6KA1","RPS6KA2","RPS6KA3","SCMH1",
    "SP1","STAT3","SUZ12","TERF1","TERF2","TERF2IP","TFDP1","TFDP2","TINF2",
    "TNIK","TNRC6A","TNRC6B","TNRC6C","TP53","TXN","UBA52","UBB","UBC",
    "UBE2C","UBE2D1","UBE2E1","UBE2S","UBN1","VENTX"
]

MOOTHA_PGC_ALL = [
    "ABCA2","ABCF1","ABHD5","ACADL","ACADM","ACADS","ACKR3","ACO2","ACOT13",
    "ACOT7","ACTR2","ACYP1","ACYP2","ADSL","ADSS2","AGPAT3","AIFM1","AK1",
    "AKAP1","AKR1B10","ALAS1","AMACR","ANKRD40","APBA3","ARIH1","ARMC1",
    "ARVCF","ATN1","ATOSB","ATP1A1","ATP1B1","ATP2A3","ATP5F1A","ATP5F1C",
    "ATP5MG","ATP5MJ","ATP5PO","ATP6V0E2","ATP6V1D","AURKAIP1","AVPI1",
    "AZIN1","BAG6","BBLN","BEX3","BID","BNIP3L","BSG","CA2","CA4","CAB39",
    "CARHSP1","CDC34","CDK18","CDKN2D","CDR2L","CHMP7","CHP1","CHRNE",
    "CIAO1","CIAO2B","CISD1","CKMT2","CKS2","CLPP","CMC2","CNOT7","COL7A1",
    "COMT","COPS3","COX5B","COX6A1","COX7A1","COX7A2","COX7B","COX7C",
    "COX8A","CPT1A","CRAT","CRLF3","CRMP1","CS","CTSF","CX3CL1","CYC1",
    "CYCS","DBI","DDIT4","DDT","DFFA","DGAT1","DHX16","DLD","DNAJA3",
    "DNAJB9","DNAJC15","DPYSL3","DUSP1","DUSP6","DXO","EDNRA","EEF1A2",
    "EEF1AKMT3","EFHD1","EGLN1","EIF2AK1","EIF3L","EIF4A2","EMC2","ENO1",
    "ERO1A","ESRRA","ETFB","F2RL2","FABP3","FAM162A","FARS2","FASTK","FBRS",
    "FDX1","FH","FOS","FOXM1","FXR2","FZD4","GABARAPL1","GABPA","GAPDH",
    "GATA3","GCDH","GCH1","GDF15","GHITM","GK","GLRX","GLRX5","GNA13",
    "GOLPH3","GPI","GPX4","GRPEL1","GRSF1","GSS","GSTO1","GTF2H4","GTPBP2",
    "GTPBP4","GUK1","GYS1","HADHB","HBA1","HBD","HCCS","HLA-DRA","HMGB3",
    "HSD17B10","HSPA4L","HSPA5","HSPA9","HSPB8","HSPBP1","HSPD1","HSPE1",
    "ID2","IDH2","IDH3A","IDH3G","IFI16","IFRD1","INHBA","INPP5J","ISCA1",
    "ITPK1","ITPR2","JMJD6","KCNN1","KIDINS220","LATS1","LCN2","LDHA",
    "LDHB","LMBRD1","LONP1","LPGAT1","LPIN1","LRBA","LRP5","LRP5L","LRPPRC",
    "LRRFIP2","LSM1","LSM8","MAIP1","MAL","MANF","MAP2K1","MAP2K3","MAPK8",
    "MAPKAPK2","MARS1","MAT2B","MDH2","ME1","MFN1","MKRN3","MNDA","MPC2",
    "MRPL11","MRPL13","MRPL15","MRPL18","MRPL20","MRPL34","MRPL39","MRPL4",
    "MRPL40","MRPL46","MRPL58","MRPS12","MRPS17","MRPS18A","MRPS18B","MRPS2",
    "MRPS34","MSRB1","MT2A","MTARC1","MTCH2","MTHFD2","MTOR","MTX1","MTX2",
    "MUC5AC","MXRA8","MYH11","MYL2","NCAM1","NDRG2","NDRG4","NDUFA5",
    "NDUFA6","NDUFA8","NDUFAB1","NDUFAF1","NDUFB3","NDUFB5","NDUFB6",
    "NDUFC1","NDUFS3","NDUFS5","NDUFV2","NECAP2","NFIX","NFS1","NFYC",
    "NINJ1","NIPSNAP2","NME7","NOP10","NPPB","NR0B2","NRP1","NSF","NTAN1",
    "OSBPL1A","OXCT1","P4HA2","PALS1","PCK1","PCOLCE","PCYT2","PDCD6",
    "PDGFRB","PDHA1","PDHB","PDK4","PDXK","PFKM","PFKP","PGAM1","PGK1",
    "PGLS","PGM1","PHB2","PHLDA1","PHYH","PIGQ","PKM","PLA2G12A","PLEKHO1",
    "PMM1","POLDIP2","POLR2E","POLR2F","POPDC2","PPM1G","PRDX2","PRDX3",
    "PRELID3B","PRSS23","PSCA","PSMA5","PSMA6","PSMB10","PSMB5","PSMB7",
    "PSME3","PTCD2","PTCD3","PTDSS2","PTGES2","PTPRD","PUS1","PYGM",
    "RAB11FIP5","RAB31","RAB40C","RAMP3","RANGAP1","RANGRF","RHOD","RNF5",
    "RPA3","RRM1","RRM2","RTN4","RUVBL2","RXRB","S100A1","S100A13","SAC3D1",
    "SACM1L","SAR1B","SAT1","SCD","SCN1B","SDHA","SDHB","SF1","SGTA",
    "SH3BGR","SHMT1","SKIC2","SKIC8","SLC12A7","SLC20A1","SLC25A11",
    "SLC25A20","SLC25A4","SLC25A5","SLC2A4","SLC31A1","SMPD1","SNW1","SOD2",
    "SOD3","SPINT1","SPP1","SPSB2","SPTLC2","SRC","ST6GALNAC4","STARD7",
    "STK16","SUCLA2","SUCLG1","SUMO1","SUPV3L1","TARBP1","TBL1X","TCIRG1",
    "TCN2","TFB2M","TFRC","TIMM10B","TIMM13","TIMM17A","TIMM23","TIMM44",
    "TIMM8A","TIMM8B","TINAGL1","TLE5","TMEM41B","TMEM45A","TMX2","TNNI3",
    "TOMM40","TPI1","TRAK2","TRIAP1","TSC2","TSPAN12","TTC1","TUFM","TVP23B",
    "TWF2","TXN2","TYMP","UBC","UBE2B","UCK2","ULK1","UNC119B","UNG","UPP1",
    "UQCR10","UQCRB","UQCRC1","UROD","USP2","USP8","VAMP8","VDAC3","VEGFA",
    "VIPR1","VPS13B","WHR1","ZFAND5","ZNF33B","ZNF589"
]

REACTOME_ENERGY_DEPENDENT_REGULATION_OF_MTOR_BY_LKB1_AMPK_ALL = [
    "CAB39","CAB39L","LAMTOR1","LAMTOR2","LAMTOR3","LAMTOR4","LAMTOR5",
    "MLST8","MTOR","PPM1A","PRKAA1","PRKAA2","PRKAB1","PRKAB2","PRKAG1",
    "PRKAG2","PRKAG3","RHEB","RPTOR","RRAGA","RRAGB","RRAGC","RRAGD",
    "SLC38A9","STK11","STRADA","STRADB","TSC1","TSC2"
]

GOBP_CAMKK_AMPK_SIGNALING_CASCADE_ALL = [
    "CAMKK2","HTT","IRGM","KCNJ11","KCNJ8","LRRK2","MYH7B","PCP4","PRKAA1",
    "TREM2","ZMPSTE24"
]

WANG_ADIPOGENIC_GENES_REPRESSED_BY_SIRT1_ALL = [
    "ALDOA","CBR1","CHCHD10","COX7A1","EGLN3","EHHADH","ELOVL3","ENO1",
    "ERO1A","FGF21","GAPDH","GK","GPI","HIGD1A","IGFBP4","KLB","KLF15",
    "MRAP","NDRG1","OLR1","PDXP","PFKP","PGK1","RBP4","SCD","SLC2A1","TPI1",
    "TRIB3"
]

WP_NAD_METABOLISM_SIRTUINS_AND_AGING_ALL = [
    "FOXO1","FOXO3","HIF1A","NAMPT","NFKB1","PARP1","PPARG","ROS1","SIRT1",
    "SIRT3","TFAM"
]

REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION_ALL = [
    "H2AB1","H2AC14","H2AC18","H2AC19","H2AC20","H2AC4","H2AC6","H2AC7",
    "H2AC8","H2AJ","H2AX","H2AZ2","H2BC1","H2BC10","H2BC11","H2BC12",
    "H2BC12L","H2BC13","H2BC14","H2BC15","H2BC17","H2BC21","H2BC26","H2BC3",
    "H2BC4","H2BC5","H2BC6","H2BC7","H2BC8","H2BC9","H3-3A","H3-3B","H3C1",
    "H3C10","H3C11","H3C12","H3C13","H3C14","H3C15","H3C2","H3C3","H3C4",
    "H3C6","H3C7","H3C8","H4C1","H4C11","H4C12","H4C13","H4C14","H4C15",
    "H4C16","H4C2","H4C3","H4C4","H4C5","H4C6","H4C8","H4C9","RRP8","SIRT1",
    "SUV39H1","TAF1A","TAF1B","TAF1C","TAF1D","TBP"
]

KEGG_MEDICUS_REFERENCE_EP_NE_ADRB_CAMP_SIGNALING_PATHWAY_ALL = [
    "ADCY1","ADCY2","ADCY3","ADCY4","ADCY5","ADCY6","ADCY7","ADCY8","ADCY9",
    "ADRB2","GNAS","PLN","PRKACA","PRKACB","PRKACG"
]

WP_GLP1_FROM_INTESTINE_AND_PANCREAS_AND_ROLE_IN_GLUCOSE_HOMEOSTASIS_ALL = [
    "GCG","GLP1R","INS","PCSK1","PCSK2","SLC2A1"
]

REACTOME_SYNTHESIS_SECRETION_AND_INACTIVATION_OF_GLUCAGON_LIKE_PEPTIDE_1_GLP_1_ALL = [
    "CDX2","CTNNB1","DPP4","FFAR1","FFAR4","GCG","GNAT3","GNB1","GNB3",
    "GNG13","GPR119","GRP","LEP","PAX6","PCSK1","SEC11A","SEC11C","SPCS1",
    "SPCS2","SPCS3","TCF7L2"
]

REACTOME_GLUCAGON_LIKE_PEPTIDE_1_GLP1_REGULATES_INSULIN_SECRETION_ALL = [
    "ADCY5","ADCY6","ADCY8","AKAP5","GCG","GLP1R","GNAS","GNB1","GNB2",
    "GNB3","GNB4","GNB5","GNG10","GNG11","GNG12","GNG13","GNG2","GNG3",
    "GNG4","GNG5","GNG7","GNG8","GNGT1","GNGT2","IQGAP1","ITPR1","ITPR2",
    "ITPR3","KCNB1","KCNC2","KCNG2","KCNS3","PRKACA","PRKACB","PRKACG",
    "PRKAR1A","PRKAR1B","PRKAR2A","PRKAR2B","RAP1A","RAPGEF3","RAPGEF4"
]

#### Gene Sets Senolytics-Related

In [ ]:
GOCC_BCL_2_FAMILY_PROTEIN_COMPLEX_ALL = [
    "BAD","BAK1","BAX","BCL2","BCL2L1","BCL2L11","BCL2L2","BIK","MCL1","PMAIP1"
]

GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL = [
    "ABL1","ACKR3","ADCY10","AEN","AIFM1","AKT1","APAF1","APP","ARHGEF2","ARL6IP5",
    "ARMC10","ATAD5","ATF2","ATF4","ATM","ATP2A1","ATP2A3","BAD","BAG5","BAG6",
    "BAK1","BAX","BBC3","BCAP31","BCL2","BCL2A1","BCL2L1","BCL2L10","BCL2L11",
    "BCL2L12","BCL2L2","BCL3","BCLAF1","BDKRB2","BECN1","BID","BIRC2","BIRC3",
    "BNIP3","BOK","BRCA1","BRCA2","BRSK2","CA3","CARD8","CASP2","CASP3","CASP4",
    "CASP6","CASP7","CASP9","CAV1","CCAR2","CD24","CD44","CD74","CDIP1","CDKN1A",
    "CDKN2D","CEBPB","CHAC1","CHEK2","CIDEB","CLU","COA8","CREB3","CREB3L1","CRIP1",
    "CTNNB1","CUL1","CUL2","CUL3","CUL4A","CUL5","CXCL12","CYCS","CYLD","CYP1B1",
    "DAB2IP","DAPK2","DAXX","DDIAS","DDIT3","DDIT4","DDX3X","DDX5","DIABLO","DNAJA1",
    "DNAJC10","DYRK2","E2F1","E2F2","EDA2R","EI24","EIF2AK3","EIF5A","ELL3","ENO1",
    "EP300","EPHA2","EPO","ERCC2","ERCC6","ERN1","ERN2","ERO1A","ERP29","FBH1",
    "FBXO7","FBXW7","FCGR2B","FGF2","FHIT","FIGNL1","FIS1","FLCN","FNIP2","FYN",
    "FZD1","GATA4","GPX1","GRINA","GSDME","GSKIP","HDAC1","HELLS","HERPUD1","HIC1",
    "HIF1A","HINT1","HIP1R","HIPK1","HIPK2","HNRNPK","HRAS","HSPA1A","HSPB1","HSPE1",
    "HTRA2","HYOU1","IER3","IFI16","IFI27","IFI27L1","IFI27L2","IFI6","IKBKE","IKBKG",
    "IL10","IL19","IL20RA","ING2","INS","ITPR1","IVNS1ABP","JAK2","JMY","KDM1A",
    "LAPTM5","LCK","LGALS12","LRRK2","MAEL","MAGEA3","MAP2K1","MAP2K4","MAP3K5",
    "MAPK7","MAPK8IP1","MARCHF7","MCL1","MDM2","MELK","MIF","MIR132","MIR133A1",
    "MIR15A","MIR16-1","MIR17","MIR186","MIR195","MIR19A","MIR21","MIR27B","MIR29B1",
    "MIR92A1","MLH1","MLLT11","MMP2","MMP9","MOAP1","MSH2","MSH6","MSX1","MTCH2",
    "MUC1","MYBBP1A","MYC","NACC2","NBN","NCK1","NCK2","NFATC4","NFE2L2","NHERF1",
    "NKX3-1","NLRP1","NME5","NOC2L","NOL3","NONO","NOX1","NUPR1","OPA1","P4HB",
    "PARK7","PARL","PARP1","PDCD10","PDK1","PDK2","PDX1","PERP","PHLDA3","PIAS4",
    "PIK3CB","PIK3R1","PINK1","PLAGL2","PLAUR","PMAIP1","PML","POLB","POU4F1","POU4F2",
    "PPIA","PPIF","PPP1R13B","PPP1R15A","PPP2R5C","PRKCD","PRKDC","PRKN","PRKRA",
    "PRODH","PTGS2","PTPMT1","PTPN1","PTPN2","PTTG1IP","PYCARD","PYCR1","QRICH1",
    "RACK1","RAD9A","RIPK1","RIPK3","RNF183","RNF186","RPL26","RPS27L","RPS3","RPS7",
    "RRM2B","RRN3","RRP8","RTKN2","S100A8","S100A9","SCN2A","SELENOK","SELENOS",
    "SEPTIN4","SERINC3","SFN","SFPQ","SFRP2","SGMS1","SGPP1","SHISA5","SIAH1","SIRT1",
    "SIVA1","SKIL","SNAI1","SNAI2","SNW1","SOD1","SOD2","SRC","STAT3","STK11",
    "STK24","STK25","STYXL1","SYVN1","TAF6","TAF9","TAF9B","TIFAB","TMBIM6","TMEM109",
    "TMEM117","TMEM161A","TMEM238L","TNF","TNFRSF10B","TNFRSF1A","TNFRSF1B","TOPORS",
    "TP53","TP53BP1","TP53BP2","TP63","TP73","TPT1","TRADD","TRAF2","TRAF5","TRAP1",
    "TREM2","TRIAP1","TRIB3","TRIM32","TXNDC12","UACA","UBB","UBQLN1","URI1","USP15",
    "USP28","USP47","VNN1","WFS1","WNT1","WWOX","XBP1","XPA","YBX3","ZNF385A",
    "ZNF385B","ZNF622"
]

REACTOME_INTRINSIC_PATHWAY_FOR_APOPTOSIS_ALL = [
    "AKT1","AKT2","AKT3","APAF1","APIP","AVEN","BAD","BAK1","BAX","BBC3","BCL2",
    "BCL2L1","BCL2L11","BID","BMF","C1QBP","CARD8","CASP3","CASP7","CASP8","CASP9",
    "CDKN2A","CYCS","DIABLO","DYNLL1","DYNLL2","E2F1","GSDMD","GSDME","GZMB","MAPK1",
    "MAPK3","MAPK8","NMT1","PMAIP1","PPP1R13B","PPP3CC","PPP3R1","SEPTIN4","SFN",
    "STAT3","TFDP1","TFDP2","TP53","TP53BP2","TP63","TP73","UACA","XIAP","YWHAB",
    "YWHAE","YWHAG","YWHAH","YWHAQ","YWHAZ"
]

KEGG_APOPTOSIS_ALL = [
    "AIFM1","AKT1","AKT2","AKT3","APAF1","ATM","BAD","BAX","BCL2","BCL2L1","BID",
    "BIRC2","BIRC3","CAPN1","CAPN2","CASP10","CASP3","CASP6","CASP7","CASP8","CASP9",
    "CFLAR","CHP1","CHP2","CHUK","CSF2RB","CYCS","DFFA","DFFB","ENDOD1","ENDOG",
    "EXOG","FADD","FAS","FASLG","IKBKB","IKBKG","IL1A","IL1B","IL1R1","IL1RAP","IL3",
    "IL3RA","IRAK1","IRAK2","IRAK3","IRAK4","MAP3K14","MYD88","NFKB1","NFKBIA","NGF",
    "NTRK1","PIK3CA","PIK3CB","PIK3CD","PIK3CG","PIK3R1","PIK3R2","PIK3R3","PIK3R5",
    "PPP3CA","PPP3CB","PPP3CC","PPP3R1","PPP3R2","PRKACA","PRKACB","PRKACG","PRKAR1A",
    "PRKAR1B","PRKAR2A","PRKAR2B","PRKX","RELA","RIPK1","TNF","TNFRSF10A","TNFRSF10B",
    "TNFRSF10C","TNFRSF10D","TNFRSF1A","TNFSF10","TP53","TRADD","TRAF2","XIAP"
]

HALLMARK_APOPTOSIS_ALL = [
    "ADD1","AIFM3","ANKH","ANXA1","APP","ATF3","AVPR1A","BAX","BCAP31","BCL10",
    "BCL2L1","BCL2L10","BCL2L11","BCL2L2","BGN","BID","BIK","BIRC3","BMF","BMP2",
    "BNIP3L","BRCA1","BTG2","BTG3","CASP1","CASP2","CASP3","CASP4","CASP6","CASP7",
    "CASP8","CASP9","CAV1","CCNA1","CCND1","CCND2","CD14","CD2","CD38","CD44",
    "CD69","CDC25B","CDK2","CDKN1A","CDKN1B","CFLAR","CLU","CREBBP","CTH","CTNNB1",
    "CYLD","DAP","DAP3","DCN","DDIT3","DFFA","DIABLO","DNAJA1","DNAJC3","DNM1L",
    "DPYD","EBP","EGR3","EMP1","ENO2","ERBB2","ERBB3","EREG","ETF1","F2","F2R",
    "FAS","FASLG","FDXR","FEZ1","GADD45A","GADD45B","GCH1","GNA15","GPX1","GPX3",
    "GPX4","GSN","GSR","GSTM1","GUCY2D","H1-0","HGF","HMGB2","HMOX1","HSPB1","IER3",
    "IFITM3","IFNB1","IFNGR1","IGF2R","IGFBP6","IL18","IL1A","IL1B","IL6","IRF1",
    "ISG20","JUN","KRT18","LEF1","LGALS3","LMNA","LUM","MADD","MCL1","MGMT","MMP2",
    "NEDD9","NEFH","PAK1","PDCD4","PDGFRB","PEA15","PLAT","PLCB2","PLPPR4","PMAIP1",
    "PPP2R5B","PPP3R1","PPT1","PRF1","PSEN1","PSEN2","PTK2","RARA","RELA","RETSAT",
    "RHOB","RHOT2","RNASEL","ROCK1","SAT1","SATB1","SC5D","SLC20A1","SMAD7","SOD1",
    "SOD2","SPTAN1","SQSTM1","TAP1","TGFB2","TGFBR3","TIMP1","TIMP2","TIMP3","TNF",
    "TNFRSF12A","TNFSF10","TOP2A","TSPO","TXNIP","VDAC2","WEE1","XIAP"
]

HALLMARK_PI3K_AKT_MTOR_SIGNALING_ALL = [
    "ACACA","ACTR2","ACTR3","ADCY2","AKT1","AKT1S1","AP2M1","ARF1","ARHGDIA","ARPC3",
    "ATF1","CAB39","CAB39L","CALR","CAMK4","CDK1","CDK2","CDK4","CDKN1A","CDKN1B",
    "CFL1","CLTC","CSNK2B","CXCR4","DAPP1","DDIT3","DUSP3","E2F1","ECSIT","EGFR",
    "EIF4E","FASLG","FGF17","FGF22","FGF6","GNA14","GNGT1","GRB2","GRK2","GSK3B",
    "HRAS","HSP90B1","IL2RG","IL4","IRAK4","ITPR2","LCK","MAP2K3","MAP2K6","MAP3K7",
    "MAPK1","MAPK10","MAPK8","MAPK9","MAPKAP1","MKNK1","MKNK2","MYD88","NCK1","NFKBIB",
    "NGF","NOD1","PAK4","PDK1","PFN1","PIK3R3","PIKFYVE","PIN1","PITX2","PLA2G12A",
    "PLCB1","PLCG1","PPP1CA","PPP2R1B","PRKAA2","PRKAG1","PRKAR2A","PRKCB","PTEN",
    "PTPN11","RAC1","RAF1","RALB","RIPK1","RIT1","RPS6KA1","RPS6KA3","RPTOR","SFN",
    "SLA","SLC2A1","SMAD2","SQSTM1","STAT2","TBK1","THEM4","TIAM1","TNFRSF1A","TRAF2",
    "TRIB3","TSC2","UBE2D3","UBE2N","VAV3","YWHAB"
]

SAUL_SEN_MAYO_ALL = [
    "ACVR1B","ANG","ANGPT1","ANGPTL4","AREG","AXL","BEX3","BMP2","BMP6","C3","CCL1",
    "CCL13","CCL16","CCL2","CCL20","CCL24","CCL26","CCL3","CCL4","CCL5","CCL7","CCL8",
    "CD55","CD9","CSF1","CSF2","CSF2RB","CST4","CTNNB1","CTSB","CXCL1","CXCL10",
    "CXCL12","CXCL16","CXCL2","CXCL3","CXCL8","CXCR2","DKK1","EDN1","EGF","EGFR",
    "EREG","ESM1","ETS2","FAS","FGF1","FGF2","FGF7","GDF15","GEM","GMFG","HGF",
    "HMGB1","ICAM1","ICAM3","IGF1","IGFBP1","IGFBP2","IGFBP3","IGFBP4","IGFBP5",
    "IGFBP6","IGFBP7","IL10","IL13","IL15","IL18","IL1A","IL1B","IL2","IL32","IL6",
    "IL6ST","IL7","INHA","IQGAP2","ITGA2","ITPKA","JUN","KITLG","LCP1","MIF","MMP1",
    "MMP10","MMP12","MMP13","MMP14","MMP2","MMP3","MMP9","NAP1L4","NRG1","PAPPA",
    "PECAM1","PGF","PIGF","PLAT","PLAU","PLAUR","PTBP1","PTGER2","PTGES","RPS6KA5",
    "SCAMP4","SELPLG","SEMA3F","SERPINB4","SERPINE1","SERPINE2","SPP1","SPX","TIMP2",
    "TNF","TNFRSF10C","TNFRSF11B","TNFRSF1A","TNFRSF1B","TUBGCP2","VEGFA","VEGFC",
    "VGF","WNT16","WNT2"
]

#### Gene Sets NAD-related

In [ ]:
WP_NAD_METABOLISM_ALL = [
    "CD38","NADK","NAMPT","NMNAT1","NMNAT2","NMNAT3","NRK","NT5E","PARP1",
    "SIRT1","SIRT2","SIRT3","SIRT4","SIRT5","SIRT6","SIRT7"
]

KEGG_NICOTINATE_AND_NICOTINAMIDE_METABOLISM_ALL = [
    "AOX1","BST1","CD38","ENPP1","ENPP3","NADK","NADSYN1","NAMPT","NMNAT1",
    "NMNAT2","NMNAT3","NMRK1","NNMT","NNT","NT5C","NT5C1A","NT5C1B","NT5C2",
    "NT5C3A","NT5E","NT5M","NUDT12","PNP","QPRT"
]

REACTOME_NICOTINATE_METABOLISM_ALL = [
    "BST1","CD38","NADK","NADK2","NADSYN1","NAMPT","NAPRT","NAXD","NAXE",
    "NMNAT1","NMNAT2","NMNAT3","NMRK1","NMRK2","NNMT","NT5E","NUDT12",
    "PARP10","PARP14","PARP16","PARP4","PARP6","PARP8","PARP9","QPRT","RNLS",
    "SLC22A13","SLC25A51","SLC5A8"
]

MOOTHA_MITOCHONDRIA_ALL = [
    "ABAT","ABCB7","ABCB8","ABCF2","ACAA1","ACAA2","ACAD8","ACADL","ACADM",
    "ACADS","ACADSB","ACADVL","ACAT1","ACO2","ACOT9","ACP6","ACSL1","ADCK2",
    "AFG3L2","AIFM1","AK2","AK4","ALAS1","ALDH2","ALDH4A1","AMACR","AMT",
    "ANXA7","APAF1","APP","ARG2","ATP5F1B","ATP5F1C","ATP5F1D","ATP5F1E",
    "ATP5IF1","ATP5MC1","ATP5MC2","ATP5MC3","ATP5ME","ATP5MF","ATP5MG",
    "ATP5MJ","ATP5PB","ATP5PD","ATP5PF","ATP5PO","ATPAF2","AZIN1","BAK1",
    "BAX","BCAT2","BCKDHA","BCKDHB","BCKDK","BCL2","BCL2A1","BCL2L1","BCS1L",
    "BDH1","BEX3","BID","BIK","BNIP3L","C1QBP","CA5A","CA5B","CAD","CASP2",
    "CASP8","CASQ1","CAT","CBARP","CD40","CDC37P1","CDC42BPB","CFAP410",
    "CKMT1B","CKMT2","CLN3","CLPX","CNNM4","CNOT7","COQ7","COQ8A","COX10",
    "COX11","COX15","COX17","COX4I1","COX5A","COX5B","COX6A1","COX6A2",
    "COX6B1","COX6C","COX7A1","COX7A2","COX7B","COX7C","COX8A","CPOX","CPS1",
    "CPT1A","CPT1B","CRAT","CROT","CRY1","CS","CYB5A","CYB5B","CYB5R3","CYBA",
    "CYBB","CYC1","CYCS","CYP11A1","CYP11B1","CYP11B2","CYP20A1","CYP27A1",
    "CYP27B1","CYP2E1","CYP7A1","DAP3","DBT","DCTN6","DDX28","DECR1","DGUOK",
    "DHODH","DIABLO","DLAT","DLD","DMAC2L","DNASE2","DNM1L","DUT","ECHS1",
    "ECI1","ECI2","EHHADH","EMC8","ENDOG","ERBB2","ETFA","ETFB","ETFDH",
    "FAM3A","FARS2","FARSA","FDX1","FDXR","FECH","FH","FHP2","FIBP","FPGS",
    "FXN","G6PD","GAS8-AS1","GATB","GATM","GCDH","GCFC2","GCK","GFM1","GLS2",
    "GLUD1","GLYAT","GNAS","GNLY","GOT2","GPD2","GPX4","GRB10","H2BK1","HADH",
    "HADHA","HADHB","HAX1","HCCS","HK1","HMGCL","HMGCS2","HPS1","HSPA9",
    "HSPD1","HSPE1","HTATSF1","HTRA2","IDH2","IDH3A","IDH3B","IMMT","IMP3",
    "IVD","LARS2","LCAT","LGALS3","LMF2","LONP1","LYPLA2","MAOA","MAOB",
    "MAP3K1","MAST1","MB","MBD3","MCAT","MDH2","ME2","ME3","METTL17","MFN2",
    "MIPEP","MLYCD","MMUT","MPO","MPST","MRPL11","MRPL12","MRPL13","MRPL15",
    "MRPL16","MRPL17","MRPL18","MRPL19","MRPL2","MRPL20","MRPL22","MRPL24",
    "MRPL28","MRPL3","MRPL33","MRPL34","MRPL35","MRPL39","MRPL4","MRPL42",
    "MRPL44","MRPL46","MRPL48","MRPL49","MRPL57","MRPL9","MRPS10","MRPS11",
    "MRPS11P1","MRPS12","MRPS14","MRPS15","MRPS16","MRPS17","MRPS18A","MRPS18B",
    "MRPS18C","MRPS2","MRPS22","MRPS27","MRPS28","MRPS30","MRPS31","MRPS33",
    "MRPS34","MRPS35","MRPS7","MTCH1","MTCH2","MTHFD1","MTHFD2","MTIF2","MTO1",
    "MTRF1","MTX1","MTX2","MYCBP","NBN","NCAPH2","NDUFA1","NDUFA10","NDUFA2",
    "NDUFA3","NDUFA4","NDUFA5","NDUFA6","NDUFA7","NDUFA8","NDUFAB1","NDUFB1",
    "NDUFB2","NDUFB3","NDUFB4","NDUFB5","NDUFB6","NDUFB7","NDUFB8","NDUFC1",
    "NDUFC2","NDUFS1","NDUFS2","NDUFS3","NDUFS4","NDUFS5","NDUFS6","NDUFS8",
    "NDUFV2","NFS1","NME2","NME4","NNT","NPIPA1","NPIPB3","NR3C1","NT5M","OAT",
    "OAZ3","OTC","OXA1L","PAM16","PC","PCCA","PCCB","PCK2","PDHA1","PDHB",
    "PDHX","PDK1","PDK3","PDK4","PIN4","PKLR","PLA2G1B","PLA2G2A","PMPCB",
    "POLG","POLG2","POLR1B","POLR3B","POLRMT","POR","PPA1","PPOX","PRDX3",
    "PRKCD","PRPF6","PTPRA","PYCR1","PYGB","R3HCC1L","RAF1","RAN","RPAIN",
    "SARDH","SBNO2","SDHA","SDHB","SDHC","SDHD","SHB","SHMT2","SIRT4","SLC1A1",
    "SLC1A2","SLC1A3","SLC25A1","SLC25A10","SLC25A11","SLC25A12","SLC25A13",
    "SLC25A14","SLC25A15","SLC25A17","SLC25A20","SLC25A21","SLC25A22","SLC25A23",
    "SLC25A28","SLC25A3","SLC25A31","SLC25A32","SLC25A36","SLC25A37","SLC25A38",
    "SLC25A4","SLC25A40","SLC25A5","SLC30A9","SLC9A6","SLF2","SMCP","SNCA",
    "SOD2","SPG7","SSBP1","STAR","STARD3","SUCLG1","SUPV3L1","SURF1","TAT",
    "TBCB","TFAM","TFB1M","TFB2M","TIMM10","TIMM10B","TIMM13","TIMM17A",
    "TIMM17B","TIMM22","TIMM23","TIMM44","TIMM8A","TIMM8B","TIMM9","TK2",
    "TNFSF10","TOMM20","TOMM22","TOMM34","TOMM40","TOMM70","TP53","TP53AIP1",
    "TPO","TSFM","TSPO","TST","TUFM","TWNK","TXN2","TXNRD2","TYMP","UCP1",
    "UCP2","UCP3","UQCR11","UQCRB","UQCRC1","UQCRC2","UQCRFS1","UQCRH","VAV1",
    "VDAC2","VDAC3","WARS2","YARS2","YME1L1","YWHAE","ZNF33B"
]

HALLMARK_OXIDATIVE_PHOSPHORYLATION_ALL = [
    "ABCB7","ACAA1","ACAA2","ACADM","ACADSB","ACADVL","ACAT1","ACO2","AFG3L2",
    "AIFM1","ALAS1","ALDH6A1","ATP1B1","ATP5F1A","ATP5F1B","ATP5F1C","ATP5F1D",
    "ATP5F1E","ATP5MC1","ATP5MC2","ATP5MC3","ATP5ME","ATP5MF","ATP5MG","ATP5PB",
    "ATP5PD","ATP5PF","ATP5PO","ATP6AP1","ATP6V0B","ATP6V0C","ATP6V0E1",
    "ATP6V1C1","ATP6V1D","ATP6V1E1","ATP6V1F","ATP6V1G1","ATP6V1H","BAX",
    "BCKDHA","BDH2","CASP7","COX10","COX11","COX15","COX17","COX4I1","COX5A",
    "COX5B","COX6A1","COX6B1","COX6C","COX7A2","COX7A2L","COX7B","COX7C",
    "COX8A","CPT1A","CS","CYB5A","CYB5R3","CYC1","CYCS","DECR1","DLAT","DLD",
    "DLST","ECH1","ECHS1","ECI1","ETFA","ETFB","ETFDH","FDX1","FH","FXN",
    "GLUD1","GOT2","GPI","GPX4","GRPEL1","HADHA","HADHB","HCCS","HSD17B10",
    "HSPA9","HTRA2","IDH1","IDH2","IDH3A","IDH3B","IDH3G","IMMT","ISCA1","ISCU",
    "LDHA","LDHB","LRPPRC","MAOB","MDH1","MDH2","MFN2","MGST3","MPC1","MRPL11",
    "MRPL15","MRPL34","MRPL35","MRPS11","MRPS12","MRPS15","MRPS22","MRPS30",
    "MTRF1","MTRR","MTX2","NDUFA1","NDUFA2","NDUFA3","NDUFA4","NDUFA5","NDUFA6",
    "NDUFA7","NDUFA8","NDUFA9","NDUFAB1","NDUFB1","NDUFB2","NDUFB3","NDUFB4",
    "NDUFB5","NDUFB6","NDUFB7","NDUFB8","NDUFC1","NDUFC2","NDUFS1","NDUFS2",
    "NDUFS3","NDUFS4","NDUFS6","NDUFS7","NDUFS8","NDUFV1","NDUFV2","NNT","NQO2",
    "OAT","OGDH","OPA1","OXA1L","PDHA1","PDHB","PDHX","PDK4","PDP1","PHB2",
    "PHYH","PMPCA","POLR2F","POR","PRDX3","RETSAT","RHOT1","RHOT2","SDHA","SDHB",
    "SDHC","SDHD","SLC25A11","SLC25A12","SLC25A20","SLC25A3","SLC25A4","SLC25A5",
    "SLC25A6","SUCLA2","SUCLG1","SUPV3L1","SURF1","TCIRG1","TIMM10","TIMM13",
    "TIMM17A","TIMM50","TIMM8B","TIMM9","TOMM22","TOMM70","UQCR10","UQCR11",
    "UQCRB","UQCRC1","UQCRC2","UQCRFS1","UQCRH","UQCRQ","VDAC1","VDAC2","VDAC3"
]

GOMF_NADPLUS_BINDING_ALL = [
    "ALDH1A3","CRYL1","EHHADH","GLUD1","HADH","HADHA","HPGD","PARP14","PARP15",
    "PARP9","SIRT1","SIRT2","SIRT3","SIRT4","SIRT5","SIRT6","SIRT7","UXS1",
    "ZC3HAV1"
]

GOBP_NAD_TRANSPORT_ALL = [
    "P2RX7","SLC25A17","SLC25A47","SLC25A51","SLC25A52","SLC25A53"
]

WP_NAD_BIOSYNTHETIC_PATHWAYS_ALL = [
    "ACMSD","BST1","CD38","IDO1","NADSYN1","NAMPT","NAPRT","NMNAT1","PARP1",
    "PARP2","PARP4","QPRT","SIRT1","SIRT2","SIRT3","SIRT4","SIRT5","SIRT6",
    "SIRT7","TDO2","TNKS","TNKS2"
]

#### Pruning and plasticity gene sets

In [ ]:
MONOAMINES_ALL = [
    "DRD1","DRD2","DRD3","DRD4","DRD5",
    "TH","DDC","COMT","MAOA","MAOB","DBH","GCH1","PTS","QDPR","SPR",
    "SLC6A3","SLC18A1","SLC18A2","PPP1R1B","GNAL","ADCY5","PDE1B","PDE10A",
    "HTR1A","HTR1B","HTR1D","HTR1E","HTR1F","HTR2A","HTR2B","HTR2C",
    "HTR3A","HTR3B","HTR3C","HTR3D","HTR3E","HTR4","HTR5A","HTR5BP",
    "HTR6","HTR7",
    "TPH1","TPH2","SLC6A4","AANAT","ASMT",
    "ADRA1A","ADRA1B","ADRA1D","ADRA2A","ADRA2B","ADRA2C",
    "ADRB1","ADRB2","ADRB3",
    "PNMT","SLC6A2",
    "HRH1","HRH2","HRH3","HRH4","HDC","HNMT","SLC22A3",
    "TAAR1","TAAR2","TAAR5","TAAR6","TAAR8","TAAR9",
    "GNB1","GNB2","GNB3","GNB4","GNB5","GNG2","GNG3","GNG4","GNG7",
    "GNG11","GNG12","ADCY1","ADCY2","ADCY7","ADCY8","ADCY9",
    "PDE4A","PDE4B","PDE4C","PDE4D",
    "SYT1","SYT2","SYT4","SYT7","SYT11","SNAP25","VAMP2",
    "STX1A","STX1B","CPLX1","CPLX2"
]

GOBP_REGULATION_OF_SYNAPTIC_PLASTICITY_ALL = [
    "ABHD6","ABL1","ACE","ACP4","ADCY1","ADCY8","ADGRB1","ADORA1","ADORA2A",
    "AGER","AKAP5","ANAPC2","APOE","APP","ARC","ARF1","ATF4","BAIAP2","BCL7A",
    "BEST1","BRAF","BRSK1","C22orf39","CALB1","CALB2","CALHM2","CALM1","CALM2",
    "CALM3","CALN1","CAMK2A","CAMK2B","CAMK2D","CAMK2G","CBLN1","CD2AP","CD38",
    "CDC20","CDK5","CFL1","CHRDL1","CHRNA2","CHRNA7","CLN3","CNTN2","CNTN4",
    "CPEB3","CPLX2","CREB1","CRH","CRHR2","CX3CL1","CX3CR1","CYP46A1","DAG1",
    "DBN1","DLG4","DRD1","DRD2","DRD5","EIF2AK4","EIF4EBP2","ELOVL4","EPHA4",
    "EPHB2","F2R","FAM107A","FMR1","FXR1","FXR2","GFAP","GIPC1","GRIA1","GRIA3",
    "GRID1","GRID2","GRID2IP","GRIK2","GRIN1","GRIN2A","GRIN2B","GRIN2C","GRIN2D",
    "GRIN3A","GRIN3B","GRM2","GRM5","GSG1L","GSK3B","HMGCR","HRAS","HRH1",
    "IGSF11","INS","IQSEC2","ITPKA","ITPR3","JPH3","JPH4","KAT2A","KCNB1",
    "KCNJ10","KCNQ3","KIT","KMT2A","KRAS","LARGE1","LGMN","LILRB2","LRRTM1",
    "LRRTM2","LYPD6","LZTS1","MAP1A","MAP1B","MAPK1","MAPT","MCTP1","MECP2",
    "MEF2C","MIR30B","MIR320A","MIR320B1","MIR320B2","MIR320C1","MIR320C2",
    "MIR320D1","MIR320D2","MIR320E","MIR324","MIR337","MIR342","MIR421",
    "MIR433","MIR541","MIR545","MIR95","MME","MPP2","NCDN","NCSTN","NETO1",
    "NEURL1","NEUROD2","NF1","NFATC4","NOG","NPAS4","NPTN","NR2E1","NRGN",
    "NSG1","NSMF","NTRK2","P2RX3","PAIP2","PDE9A","PENK","PICK1","PLK2",
    "PPFIA3","PPP3CB","PRKAR1B","PRKCG","PRKCZ","PRNP","PRRT1","PRRT2",
    "PSEN1","PTK2B","PTN","RAB11A","RAB3A","RAB3GAP1","RAB5A","RAB8A","RAC1",
    "RAPGEF2","RARA","RASGRF1","RASGRF2","RELN","RGS14","S100B","SCT","SCTR",
    "SERPINE2","SHANK2","SHANK3","SHISA6","SHISA7","SHISA8","SHISA9","SIPA1L1",
    "SLC18A3","SLC1A1","SLC24A1","SLC24A2","SLC38A1","SLC4A10","SLC8A2",
    "SLC8A3","SLITRK4","SNAP25","SNCA","SORCS2","SORCS3","SQSTM1","SRF","SSH1",
    "STAU1","STAU2","STX3","STX4","STXBP1","SYAP1","SYNGAP1","SYNGR1","SYP",
    "SYT12","SYT4","SYT7","TNR","TSHZ3","TYROBP","UBE3A","UNC13C","VAMP2",
    "VGF","VPS13A","YTHDF1","YWHAG","YWHAH","ZDHHC2"]

KEGG_LTP_ALL = [
    "ADCY1","ADCY8","ARAF","ATF4","BRAF","CACNA1C","CALM1","CALM2","CALM3",
    "CALML3","CALML5","CALML6","CAMK2A","CAMK2B","CAMK2D","CAMK2G","CAMK4",
    "CHP1","CHP2","CREBBP","EP300","GNAQ","GRIA1","GRIA2","GRIN1","GRIN2A",
    "GRIN2B","GRIN2C","GRIN2D","GRM1","GRM5","HRAS","ITPR1","ITPR2","ITPR3",
    "KRAS","MAP2K1","MAP2K2","MAPK1","MAPK3","NRAS","PLCB1","PLCB2","PLCB3",
    "PLCB4","PPP1CA","PPP1CB","PPP1CC","PPP1R12A","PPP1R1A","PPP3CA","PPP3CB",
    "PPP3CC","PPP3R1","PPP3R2","PRKACA","PRKACB","PRKACG","PRKCA","PRKCB",
    "PRKCG","PRKX","RAF1","RAP1A","RAP1B","RAPGEF3","RPS6KA1","RPS6KA2",
    "RPS6KA3","RPS6KA6"]

GOBP_GLUTAMATERGIC_ALL = [
    "ABCC8","ABTB3","ADORA1","ADORA2A","ALS2","ATAD1","ATP1A2","CACNB4",
    "CACNG2","CACNG3","CACNG4","CACNG5","CACNG7","CACNG8","CCL2","CCR2","CDH8",
    "CDK5","CLCN3","CLN3","CLSTN3","CNIH2","CNIH3","DISC1","DKK1","DRD1","DRD2",
    "DRD3","DSCAM","DTNBP1","EXT1","FRRS1L","FXR1","GRIA1","GRIA2","GRIA3",
    "GRIA4","GRID1","GRID2","GRIK1","GRIK2","GRIK3","GRIK4","GRIK5","GRIN1",
    "GRIN2A","GRIN2B","GRIN2C","GRIN2D","GRIN3A","GRIN3B","GRM1","GRM2","GRM3",
    "GRM4","GRM5","GRM6","GRM7","GRM8","HCN1","HOMER1","HOMER2","HOMER3",
    "HTR2A","IQSEC2","KCNJ8","KMO","LRRK2","MAPK8IP2","MEF2C","MIR142","NAPA",
    "NAPB","NF1","NLGN1","NLGN2","NLGN3","NPS","NPY2R","NR3C1","NRXN1","NTRK1",
    "OPHN1","P2RX1","PLPPR4","PNOC","PRKN","PSEN1","PTK2B","RAB3GAP1","RELN",
    "RNF167","SERPINE2","SHANK3","SLC17A6","SLC17A7","SLC17A8","SLC1A4",
    "SLC38A2","STXBP1","SYT1","TNF","TNR","TPRG1L","TSHZ3","UCN","UNC13A",
    "UNC13B","UNC13C","VPS54"]

GOBP_SYNAPSE_PRUNING_ALL = [
    "ADGRB3","C1QA","C1QB","C1QC","C1QL1","C3","CX3CL1","CX3CR1","DKK1",
    "EPHA4","ITGAM","ITGB1","KCNK13","NGEF","PLXNC1","SARM1","TREM2","VANGL2"]

REACTOME_COMPLEMENT_ALL = [
    "C1QA","C1QB","C1QC","C1R","C1S","C2","C3","C3AR1","C4A","C4B","C4B_2",
    "C4BPA","C4BPB","C5","C5AR1","C5AR2","C6","C7","C8A","C8B","C8G","C9",
    "CD19","CD46","CD55","CD59","CD81","CFB","CFD","CFH","CFHR1","CFHR2",
    "CFHR3","CFHR4","CFHR5","CFI","CFP","CLU","COLEC10","COLEC11","CPB2",
    "CPN1","CPN2","CR1","CR2","CRP","ELANE","F2","FCN1","FCN2","FCN3","GZMM",
    "IGHG1","IGHG2","IGHG4","IGHV1-2","IGHV1-46","IGHV1-69","IGHV2-5",
    "IGHV2-70","IGHV3-11","IGHV3-13","IGHV3-23","IGHV3-30","IGHV3-33",
    "IGHV3-48","IGHV3-53","IGHV3-7","IGHV4-34","IGHV4-39","IGHV4-59",
    "IGKV1-12","IGKV1-16","IGKV1-17","IGKV1-33","IGKV1-39","IGKV1-5",
    "IGKV1D-12","IGKV1D-16","IGKV1D-33","IGKV1D-39","IGKV2-28","IGKV2-30",
    "IGKV2D-28","IGKV2D-30","IGKV2D-40","IGKV3-11","IGKV3-15","IGKV3-20",
    "IGKV3D-20","IGKV4-1","IGKV5-2","IGLC2","IGLC3","IGLV1-40","IGLV1-44",
    "IGLV1-47","IGLV1-51","IGLV2-11","IGLV2-14","IGLV2-23","IGLV2-8",
    "IGLV3-1","IGLV3-19","IGLV3-21","IGLV3-25","IGLV3-27","IGLV6-57",
    "IGLV7-43","MASP1","MASP2","MBL2","PROS1","SERPING1","VTN"]

REACTOME_SENESCENCE_ALL = [
    "ACD","AGO1","AGO3","AGO4","ANAPC1","ANAPC10","ANAPC11","ANAPC15","ANAPC16",
    "ANAPC2","ANAPC4","ANAPC5","ANAPC7","ASF1A","ATM","BMI1","CABIN1","CBX2",
    "CBX4","CBX6","CBX8","CCNA1","CCNA2","CCNE1","CCNE2","CDC16","CDC23","CDC26",
    "CDC27","CDK2","CDK4","CDK6","CDKN1A","CDKN1B","CDKN2A","CDKN2B","CDKN2C",
    "CDKN2D","CEBPB","CXCL8","E2F1","E2F2","E2F3","EED","EHMT1","EHMT2","EP400",
    "ERF","ETS1","ETS2","EZH2","FOS","FZR1","H1-0","H1-1","H1-2","H1-3","H1-4",
    "H1-5","H2AB1","H2AC14","H2AC18","H2AC19","H2AC20","H2AC4","H2AC6","H2AC7",
    "H2AC8","H2AJ","H2AX","H2AZ2","H2BC1","H2BC10","H2BC11","H2BC12","H2BC12L",
    "H2BC13","H2BC14","H2BC15","H2BC17","H2BC21","H2BC26","H2BC3","H2BC4",
    "H2BC5","H2BC6","H2BC7","H2BC8","H2BC9","H3-3A","H3-3B","H3-4","H3C1",
    "H3C10","H3C11","H3C12","H3C13","H3C14","H3C15","H3C2","H3C3","H3C4",
    "H3C6","H3C7","H3C8","H4C1","H4C11","H4C12","H4C13","H4C14","H4C15",
    "H4C16","H4C2","H4C3","H4C4","H4C5","H4C6","H4C8","H4C9","HIRA","HMGA1",
    "HMGA2","ID1","IFNB1","IGFBP7","IL1A","IL6","JUN","KAT5","KDM6B","LMNB1",
    "MAP2K3","MAP2K4","MAP2K6","MAP2K7","MAP3K5","MAP4K4","MAPK1","MAPK10",
    "MAPK11","MAPK14","MAPK3","MAPK7","MAPK8","MAPK9","MAPKAPK2","MAPKAPK3",
    "MAPKAPK5","MDM2","MDM4","MINK1","MIR24-1","MIR24-2","MOV10","MRE11","NBN",
    "NFKB1","PHC1","PHC2","PHC3","POT1","RAD50","RB1","RBBP4","RBBP7","RELA",
    "RING1","RNF2","RPS27A","RPS6KA1","RPS6KA2","RPS6KA3","SCMH1","SP1","STAT3",
    "SUZ12","TERF1","TERF2","TERF2IP","TFDP1","TFDP2","TINF2","TNIK","TNRC6A",
    "TNRC6B","TNRC6C","TP53","TXN","UBA52","UBB","UBC","UBE2C","UBE2D1",
    "UBE2E1","UBE2S","UBN1","VENTX"]

REACTOME_TELOMERE_ALL = [
    "ACD","ANKRD28","ATRX","BLM","CCNA1","CCNA2","CDK2","CHTF18","CHTF8","CTC1",
    "DAXX","DKC1","DNA2","DSCC1","FEN1","GAR1","H2AB1","H2AC14","H2AC18",
    "H2AC19","H2AC20","H2AC4","H2AC6","H2AC7","H2AC8","H2AJ","H2AX","H2AZ2",
    "H2BC1","H2BC10","H2BC11","H2BC12","H2BC12L","H2BC13","H2BC14","H2BC15",
    "H2BC17","H2BC21","H2BC26","H2BC3","H2BC4","H2BC5","H2BC6","H2BC7","H2BC8",
    "H2BC9","H3-3A","H3-3B","H3-4","H4C1","H4C11","H4C12","H4C13","H4C14",
    "H4C15","H4C16","H4C2","H4C3","H4C4","H4C5","H4C6","H4C8","H4C9","LIG1",
    "NHP2","NOP10","PCNA","PIF1","POLA1","POLA2","POLD1","POLD2","POLD3",
    "POLD4","POLR2A","POLR2B","POLR2C","POLR2D","POLR2E","POLR2F","POLR2G",
    "POLR2H","POLR2I","POLR2J","POLR2K","POLR2L","POT1","PPP6C","PPP6R3",
    "PRIM1","PRIM2","RFC1","RFC2","RFC3","RFC4","RFC5","RPA1","RPA2","RPA3",
    "RTEL1","RUVBL1","RUVBL2","SHQ1","STN1","TEN1","TERF1","TERF2","TERF2IP",
    "TERT","TINF2","WRAP53","WRN"]

ADULT_ASTRO_ALL = [
    "ALDH1L1","EYA1","NWD1","SLCO1C1","HGF","SPARCL1","SLC25A18","FGFR3",
    "SDC4","GFAP","ACSBG1","PRODH","ITGB4","SLC7A11","AGT","PPP1R3C","RANBP3L",
    "RNF43","GJB6","SLC14A1","GJA1","USP9Y","TTTY15","KDM5D","SLC39A12",
    "COL16A1","DLC1","ABO","ABCC9","MGST1","GABRG1","ITGA7","CLTCL1","DCHS2",
    "RGS9","SLC30A10","GPR98","BMPR1B","SLC4A4","GRM3"]

ADULT_OLIGO_ALL = [
    "MSX1","ADAMTS4","TF","TMEM144","ENPP2","KLK6","ERMN","RNASE1","PLP1",
    "OPALIN","CNDP1","FOLH1","UGT8","CNP","PLEKHB1","MAL","GPR37","NR4A2",
    "B3GNT7","GPNMB","GLYCTK","GJB1","PAQR6","GSN","ABCA8","FA2H","CARNS1",
    "MOBP","MBP","CLDN11","RRBP1","OLFML2B","KCNMB4","LGI3","MAG"]

ADULT_MICRO_ALL = [
    "ST8SIA4","KIAA0226L","CSF1R","LAPTM5","CD14","RGS10","PTPRC","C3","C3AR1",
    "ALOX5AP","HLA-DRA","BCL2A1","PLEK","ITGAX","DHRS9","OLR1","CD53","LCP1",
    "CD74","CD83","A2M","GAB3","CMKLR1","TREM1","GALR1"]

HLA_COMPLEX_ALL = [
    "HLA-A","HLA-B","HLA-C","HLA-DMA","HLA-DMB","HLA-DOA","HLA-DOB","HLA-DPA1",
    "HLA-DPA2","HLA-DPA3","HLA-DPB1","HLA-DPB2","HLA-DQA1","HLA-DQA2",
    "HLA-DQB1","HLA-DQB2","HLA-DQB3","HLA-DRA","HLA-DRB1","HLA-DRB2",
    "HLA-DRB3","HLA-DRB4","HLA-DRB5","HLA-DRB6","HLA-DRB7","HLA-DRB8",
    "HLA-DRB9","HLA-E","HLA-F","HLA-G","HLA-H","HLA-J","HLA-K","HLA-L",
    "HLA-N","HLA-P","HLA-S","HLA-T","HLA-U","HLA-V","HLA-W","HLA-X","HLA-Y",
    "HLA-Z"]

# Setting

In [ ]:
dirs_psy = [
    "/content/bip",
    "/content/migraine"
]

lab_psy = [
    "bip",
    "migraine"
]

### All-in-one


In [ ]:
multi_psy = run_multi_geneset_pipeline(
    dirs_psy,
    lab_psy,
    gene_sets={
        "WP_NAD_METABOLISM_ALL": WP_NAD_METABOLISM_ALL,
        "KEGG_NICOTINATE_AND_NICOTINAMIDE_METABOLISM_ALL": KEGG_NICOTINATE_AND_NICOTINAMIDE_METABOLISM_ALL,
        "REACTOME_NICOTINATE_METABOLISM_ALL": REACTOME_NICOTINATE_METABOLISM_ALL,
        "MOOTHA_MITOCHONDRIA_ALL": MOOTHA_MITOCHONDRIA_ALL,
        "HALLMARK_OXIDATIVE_PHOSPHORYLATION_ALL": HALLMARK_OXIDATIVE_PHOSPHORYLATION_ALL,
        "GOMF_NADPLUS_BINDING_ALL": GOMF_NADPLUS_BINDING_ALL,
        "GOBP_NAD_TRANSPORT_ALL": GOBP_NAD_TRANSPORT_ALL,
        "WP_NAD_BIOSYNTHETIC_PATHWAYS_ALL": WP_NAD_BIOSYNTHETIC_PATHWAYS_ALL,
        "GOBP_INSULIN_SECRETION_INVOLVED_IN_CELLULAR_RESPONSE_TO_GLUCOSE_STIMULUS_ALL": GOBP_INSULIN_SECRETION_INVOLVED_IN_CELLULAR_RESPONSE_TO_GLUCOSE_STIMULUS_ALL,
        "REACTOME_REGULATION_OF_INSULIN_SECRETION_ALL": REACTOME_REGULATION_OF_INSULIN_SECRETION_ALL,
        "GOBP_REGULATION_OF_CELLULAR_SENESCENCE_ALL": GOBP_REGULATION_OF_CELLULAR_SENESCENCE_ALL,
        "GOBP_CELLULAR_SENESCENCE_ALL": GOBP_CELLULAR_SENESCENCE_ALL,
        "REACTOME_CELLULAR_SENESCENCE_ALL": REACTOME_CELLULAR_SENESCENCE_ALL,
        "MOOTHA_PGC_ALL": MOOTHA_PGC_ALL,
        "REACTOME_ENERGY_DEPENDENT_REGULATION_OF_MTOR_BY_LKB1_AMPK_ALL": REACTOME_ENERGY_DEPENDENT_REGULATION_OF_MTOR_BY_LKB1_AMPK_ALL,
        "GOBP_CAMKK_AMPK_SIGNALING_CASCADE_ALL": GOBP_CAMKK_AMPK_SIGNALING_CASCADE_ALL,
        "WANG_ADIPOGENIC_GENES_REPRESSED_BY_SIRT1_ALL": WANG_ADIPOGENIC_GENES_REPRESSED_BY_SIRT1_ALL,
        "WP_NAD_METABOLISM_SIRTUINS_AND_AGING_ALL": WP_NAD_METABOLISM_SIRTUINS_AND_AGING_ALL,
        "REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION_ALL": REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION_ALL,
        "KEGG_MEDICUS_REFERENCE_EP_NE_ADRB_CAMP_SIGNALING_PATHWAY_ALL": KEGG_MEDICUS_REFERENCE_EP_NE_ADRB_CAMP_SIGNALING_PATHWAY_ALL,
        "WP_GLP1_FROM_INTESTINE_AND_PANCREAS_AND_ROLE_IN_GLUCOSE_HOMEOSTASIS_ALL": WP_GLP1_FROM_INTESTINE_AND_PANCREAS_AND_ROLE_IN_GLUCOSE_HOMEOSTASIS_ALL,
        "REACTOME_SYNTHESIS_SECRETION_AND_INACTIVATION_OF_GLUCAGON_LIKE_PEPTIDE_1_GLP_1_ALL": REACTOME_SYNTHESIS_SECRETION_AND_INACTIVATION_OF_GLUCAGON_LIKE_PEPTIDE_1_GLP_1_ALL,
        "REACTOME_GLUCAGON_LIKE_PEPTIDE_1_GLP1_REGULATES_INSULIN_SECRETION_ALL": REACTOME_GLUCAGON_LIKE_PEPTIDE_1_GLP1_REGULATES_INSULIN_SECRETION_ALL,
        "GOCC_BCL_2_FAMILY_PROTEIN_COMPLEX_ALL": GOCC_BCL_2_FAMILY_PROTEIN_COMPLEX_ALL,
        "GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL": GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL,
        "REACTOME_INTRINSIC_PATHWAY_FOR_APOPTOSIS_ALL": REACTOME_INTRINSIC_PATHWAY_FOR_APOPTOSIS_ALL,
        "KEGG_APOPTOSIS_ALL": KEGG_APOPTOSIS_ALL,
        "HALLMARK_APOPTOSIS_ALL": HALLMARK_APOPTOSIS_ALL,
        "HALLMARK_PI3K_AKT_MTOR_SIGNALING_ALL": HALLMARK_PI3K_AKT_MTOR_SIGNALING_ALL,
        "SAUL_SEN_MAYO_ALL": SAUL_SEN_MAYO_ALL,
        "MONOAMINES_ALL": MONOAMINES_ALL,
        "GOBP_REGULATION_OF_SYNAPTIC_PLASTICITY_ALL": GOBP_REGULATION_OF_SYNAPTIC_PLASTICITY_ALL,
        "KEGG_LTP_ALL": KEGG_LTP_ALL,
        "GOBP_GLUTAMATERGIC_ALL": GOBP_GLUTAMATERGIC_ALL,
        "GOBP_SYNAPSE_PRUNING_ALL": GOBP_SYNAPSE_PRUNING_ALL,
        "REACTOME_COMPLEMENT_ALL": REACTOME_COMPLEMENT_ALL,
        "REACTOME_SENESCENCE_ALL": REACTOME_SENESCENCE_ALL,
        "REACTOME_TELOMERE_ALL": REACTOME_TELOMERE_ALL,
        "ADULT_ASTRO_ALL": ADULT_ASTRO_ALL,
        "ADULT_OLIGO_ALL": ADULT_OLIGO_ALL,
        "ADULT_MICRO_ALL": ADULT_MICRO_ALL,
        "HLA_COMPLEX_ALL": HLA_COMPLEX_ALL,

        "HALLMARK_INFLAMMATORY_RESPONSE": HALLMARK_INFLAMMATORY_RESPONSE,
        "REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM": REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM,
        "HALLMARK_TNFA_SIGNALING_VIA_NFKB": HALLMARK_TNFA_SIGNALING_VIA_NFKB,
        "REACTOME_CIRCADIAN_CLOCK": REACTOME_CIRCADIAN_CLOCK,
        "KEGG_DOPAMINERGIC_SYNAPSE": KEGG_DOPAMINERGIC_SYNAPSE,
        "REACTOME_NEUROTRANSMITTER_RELEASE_CYCLE": REACTOME_NEUROTRANSMITTER_RELEASE_CYCLE,
        "REACTOME_MITOPHAGY": REACTOME_MITOPHAGY,
        "GOBP_POSITIVE_REGULATION_OF_MICROGLIAL_ACTIVATION": GOBP_POSITIVE_REGULATION_OF_MICROGLIAL_ACTIVATION,
        "REACTOME_ACTIVATION_OF_INNATE_IMMUNE_RESPONSE": REACTOME_ACTIVATION_OF_INNATE_IMMUNE_RESPONSE,
    },
    out="/content/089Av2_results",
    n_perm=10000,
    save_outputs=True,
    plot=True,
    verbose=True,
    family_wise_fdr=True,
    robust=True,
    bootstrap=True,
)

Streaming output truncated to the last 5000 lines.

[4/7] Differential tests across diseases …
  Common genes in all 2 diseases: 267
  Kruskal-Wallis H = 0.0104, p = 0.91891
  → /content/089Av2_results/GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL/differential_pairwise_GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL.csv
Disease_1 Disease_2  MWU_U    MWU_p  Rank_Biserial_r  Paired_Wilcoxon_p  Mean_Z_diff  MWU_FDR  Paired_FDR
      bip  migraine  35463 0.919132           0.0051           0.790665      -0.0395 0.919132    0.790665

[5/7] Profile proximity & distance …
  → /content/089Av2_results/GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL/proximity_GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL.csv
Disease_1 Disease_2  Pearson_r  Pearson_p  Spearman_rho  Spearman_p  Cosine_sim  Euclidean_dist  Manhattan_dist  n_genes
      bip  migraine     0.0538   0.380993       -0.0278    0.651055      0.0539         51.4833        661.8704      267

[6/7] Pairwise disease statistical tests (gene-se

# Summary

In [ ]:
# ===================================================================
#  11.  SIGNIFICANT-RESULTS SUMMARY  (post-hoc on an existing run)
# ===================================================================

def generate_sig_summary(
    results_dir,
    output_path=None,
    # ── significance thresholds (all optional / tunable) ──────────
    p_threshold=0.05,
    fdr_threshold=0.05,
    stouffer_z_threshold=1.96,
    cohens_d_threshold=0.5,
    ccc_threshold=0.5,
    sign_concordance_threshold=0.7,
    kendall_tau_threshold=0.3,
    pearson_r_threshold=0.3,
    rank_biserial_threshold=0.3,
    use_fdr=True,
    include_borderline=False,
    top_n_per_section=None,
    verbose=True,
):
    """
    Walk a pipeline output folder and write a `sig_summary.txt` listing
    every significant finding across all CSVs the pipeline produced.

    Works for BOTH:
      * single-gene-set runs    (`run_pipeline()` output folder)
      * multi-gene-set runs     (`run_multi_geneset_pipeline()` output
                                 folder — including all sub-folders)

    Sections written:
      1. Per-disease enrichment       (Stouffer Z, permutation/Wilcoxon p)
      2. Differential tests           (Kruskal-Wallis, MWU, paired Wilcoxon)
      3. Profile proximity            (Pearson r, Spearman ρ)
      4. Pairwise disease tests       (KS, Welch, paired t/Wilcoxon, perm,
                                       Brunner-Munzel, Levene, Cohen's d)
      5. Concordance metrics          (sign rate, Lin's CCC, Kendall τ)
      6. Gene-level LOO influence     (genes flagged as Influential)
      7. Family-wise FDR              (if computed in the run)

    Parameters
    ----------
    results_dir : str | Path
        Folder produced by the pipeline.
    output_path : str | Path | None
        Destination text file.  Defaults to `<results_dir>/sig_summary.txt`.
    p_threshold, fdr_threshold : float
        Cut-offs for raw p and BH-FDR/global-FDR.
    stouffer_z_threshold, cohens_d_threshold, ccc_threshold,
    sign_concordance_threshold, kendall_tau_threshold,
    pearson_r_threshold, rank_biserial_threshold : float
        Effect-size thresholds (absolute value) used in addition to p-values.
    use_fdr : bool
        If True (default) and an FDR column exists, gate significance on
        FDR rather than raw p-value.  Raw p is still reported.
    include_borderline : bool
        If True, also include rows that pass *any* effect-size threshold
        even when the corresponding p/FDR is missing.
    top_n_per_section : int | None
        If given, truncate each section to its top-N strongest hits.
    verbose : bool
        Print progress to stdout.

    Returns
    -------
    pathlib.Path
        Absolute path of the written `sig_summary.txt`.
    """
    results_dir = Path(results_dir)
    if not results_dir.is_dir():
        raise FileNotFoundError(f"{results_dir} is not a directory")

    if output_path is None:
        output_path = results_dir / "sig_summary.txt"
    output_path = Path(output_path)

    _print = print if verbose else (lambda *a, **k: None)

    # ── helpers ───────────────────────────────────────────────────
    def _fmt(v, fmt=".4g", na="NA"):
        try:
            if v is None or (isinstance(v, float) and np.isnan(v)):
                return na
            return format(float(v), fmt)
        except Exception:
            return na

    def _safe_read(path):
        try:
            return pd.read_csv(path)
        except Exception as e:
            _print(f"  ⚠ could not read {path.name}: {e}")
            return pd.DataFrame()

    def _gs_from_filename(path, prefix):
        stem = path.stem
        if stem.startswith(prefix):
            return stem[len(prefix):]
        return stem

    def _ensure_geneset_col(df, fallback_label):
        """Make sure the dataframe has a 'GeneSet' column."""
        if "GeneSet" not in df.columns:
            df = df.copy()
            df.insert(0, "GeneSet", fallback_label)
        return df

    def _is_multi_mode():
        return any((results_dir / x).exists() for x in (
            "multi_geneset_enrichment_summary.csv",
            "multi_geneset_pairwise_disease_stats.csv",
            "multi_geneset_gene_influence.csv",
            "multi_geneset_run_manifest.json",
        ))

    multi_mode = _is_multi_mode()

    # ── header ────────────────────────────────────────────────────
    lines = []
    bar = "=" * 78
    lines += [
        bar,
        "  SIGNIFICANT RESULTS SUMMARY",
        f"  Source folder : {results_dir.resolve()}",
        f"  Mode detected : {'multi-gene-set' if multi_mode else 'single-gene-set'}",
        f"  Generated     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        bar,
        "",
        "  THRESHOLDS:",
        f"    raw p-value             : < {p_threshold}",
        f"    FDR / global FDR        : < {fdr_threshold}",
        f"    Filter on FDR if avail. : {use_fdr}",
        f"    |Stouffer Z|            : > {stouffer_z_threshold}",
        f"    |Cohen's d|             : > {cohens_d_threshold}",
        f"    |Lin's CCC|             : > {ccc_threshold}",
        f"    Sign-concordance rate   : > {sign_concordance_threshold}",
        f"    |Kendall τ|             : > {kendall_tau_threshold}",
        f"    |Pearson r|             : > {pearson_r_threshold}",
        f"    |Rank-biserial r|       : > {rank_biserial_threshold}",
        f"    include borderline      : {include_borderline}",
        f"    top-N per section       : {top_n_per_section}",
        "",
    ]

    section_counts = {}

    # ==============================================================
    # SECTION 1 — Per-disease enrichment
    # ==============================================================
    lines += [bar, "  [1] PER-DISEASE ENRICHMENT", bar]

    enrich_dfs = []
    if multi_mode:
        f = results_dir / "multi_geneset_enrichment_summary.csv"
        if f.exists():
            df = _safe_read(f)
            if not df.empty:
                enrich_dfs.append(df)
    # Always also pick up per-set files (works for single-mode too)
    for f in sorted(results_dir.rglob("enrichment_*.csv")):
        if f.name == "multi_geneset_enrichment_summary.csv":
            continue
        df = _safe_read(f)
        if not df.empty:
            df = _ensure_geneset_col(df, _gs_from_filename(f, "enrichment_"))
            if "Tissue" not in df.columns:
                df["Tissue"] = "meta_across_tissues"
            enrich_dfs.append(df)

    if not enrich_dfs:
        lines += ["  (no enrichment CSVs found)", ""]
    else:
        e_all = pd.concat(enrich_dfs, ignore_index=True, sort=False)
        # Deduplicate
        dedup_cols = [c for c in ("GeneSet", "Disease", "Tissue") if c in e_all.columns]
        if dedup_cols:
            e_all = e_all.drop_duplicates(subset=dedup_cols, keep="first")

        sig_rows = []
        for _, row in e_all.iterrows():
            stouffer = row.get("stouffer_z", np.nan)
            perm_p   = row.get("permutation_p", np.nan)
            wil_p    = row.get("wilcoxon_p", np.nan)

            reasons = []
            if pd.notna(perm_p) and perm_p < p_threshold:
                reasons.append(f"perm_p={_fmt(perm_p)}")
            if pd.notna(wil_p) and wil_p < p_threshold:
                reasons.append(f"wilcoxon_p={_fmt(wil_p)}")
            if pd.notna(stouffer) and abs(stouffer) > stouffer_z_threshold:
                reasons.append(f"|Stouffer|={abs(stouffer):.2f}")
            if not reasons:
                continue

            ci_lo = row.get("stouffer_ci_low", np.nan)
            ci_hi = row.get("stouffer_ci_high", np.nan)
            ci_str = (f" CI95=[{_fmt(ci_lo,'.2f')},{_fmt(ci_hi,'.2f')}]"
                      if pd.notna(ci_lo) and pd.notna(ci_hi) else "")

            sig_rows.append({
                "GeneSet": row.get("GeneSet", "?"),
                "Disease": row.get("Disease", "?"),
                "Tissue":  row.get("Tissue", "meta_across_tissues"),
                "n":       int(row.get("n_genes_found", 0) or 0),
                "stouffer_z": float(stouffer) if pd.notna(stouffer) else np.nan,
                "perm_p":  perm_p,
                "wil_p":   wil_p,
                "ci_str":  ci_str,
                "_sort":   (perm_p if pd.notna(perm_p)
                            else (wil_p if pd.notna(wil_p) else 1.0)),
                "reasons": "; ".join(reasons),
            })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["enrichment"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant enrichment finding(s):")
            lines.append("")
            for r in sig_rows:
                tissue_blurb = (f" [{r['Tissue']}]"
                                if r['Tissue'] not in ("meta_across_tissues", "meta", "?")
                                else "")
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['Disease']}{tissue_blurb}  "
                    f"n={r['n']}  Stouffer Z={_fmt(r['stouffer_z'],'.3f')}{r['ci_str']}  "
                    f"perm_p={_fmt(r['perm_p'])}  wilcoxon_p={_fmt(r['wil_p'])}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant enrichment at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 2 — Differential tests across diseases
    # ==============================================================
    lines += [bar, "  [2] DIFFERENTIAL TESTS (Kruskal-Wallis + pairwise)", bar]

    diff_files = sorted(results_dir.rglob("differential_pairwise_*.csv"))
    if not diff_files:
        lines += ["  (no differential_pairwise_*.csv files found)", ""]
    else:
        sig_rows = []
        for f in diff_files:
            df = _safe_read(f)
            if df.empty:
                continue
            df = _ensure_geneset_col(df, _gs_from_filename(f, "differential_pairwise_"))
            for _, row in df.iterrows():
                mwu_p   = row.get("MWU_p", np.nan)
                mwu_fdr = row.get("MWU_FDR", np.nan)
                pw_p    = row.get("Paired_Wilcoxon_p", np.nan)
                pw_fdr  = row.get("Paired_FDR", np.nan)
                rbc     = row.get("Rank_Biserial_r", np.nan)

                gate_mwu = mwu_fdr if (use_fdr and pd.notna(mwu_fdr)) else mwu_p
                gate_pw  = pw_fdr  if (use_fdr and pd.notna(pw_fdr))  else pw_p
                cutoff_mwu = fdr_threshold if (use_fdr and pd.notna(mwu_fdr)) else p_threshold
                cutoff_pw  = fdr_threshold if (use_fdr and pd.notna(pw_fdr))  else p_threshold

                reasons = []
                if pd.notna(gate_mwu) and gate_mwu < cutoff_mwu:
                    reasons.append(f"MWU_{'FDR' if use_fdr and pd.notna(mwu_fdr) else 'p'}={_fmt(gate_mwu)}")
                if pd.notna(gate_pw) and gate_pw < cutoff_pw:
                    reasons.append(f"PairedW_{'FDR' if use_fdr and pd.notna(pw_fdr) else 'p'}={_fmt(gate_pw)}")
                if pd.notna(rbc) and abs(rbc) > rank_biserial_threshold:
                    reasons.append(f"|RBC|={abs(rbc):.2f}")
                if not reasons:
                    continue

                sig_rows.append({
                    "GeneSet": row.get("GeneSet", "?"),
                    "pair":    f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}",
                    "mean_z_diff": row.get("Mean_Z_diff", np.nan),
                    "mwu_p":    mwu_p, "mwu_fdr": mwu_fdr,
                    "pw_p":     pw_p,  "pw_fdr":  pw_fdr,
                    "rbc":      rbc,
                    "_sort":    (gate_mwu if pd.notna(gate_mwu) else 1.0),
                    "reasons":  "; ".join(reasons),
                })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["differential"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant differential pair(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  "
                    f"mean_Z_diff={_fmt(r['mean_z_diff'],'.3f')}  "
                    f"MWU p={_fmt(r['mwu_p'])} (FDR={_fmt(r['mwu_fdr'])})  "
                    f"PairedW p={_fmt(r['pw_p'])}  RBC={_fmt(r['rbc'],'.3f')}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant differential results at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 3 — Profile proximity
    # ==============================================================
    lines += [bar, "  [3] PROFILE PROXIMITY (Pearson / Spearman)", bar]

    prox_files = sorted(results_dir.rglob("proximity_*.csv"))
    if not prox_files:
        lines += ["  (no proximity_*.csv files found)", ""]
    else:
        sig_rows = []
        for f in prox_files:
            df = _safe_read(f)
            if df.empty:
                continue
            df = _ensure_geneset_col(df, _gs_from_filename(f, "proximity_"))
            for _, row in df.iterrows():
                pr  = row.get("Pearson_r", np.nan)
                prp = row.get("Pearson_p", np.nan)
                sr  = row.get("Spearman_rho", np.nan)
                srp = row.get("Spearman_p", np.nan)
                cos = row.get("Cosine_sim", np.nan)

                reasons = []
                if pd.notna(prp) and prp < p_threshold and pd.notna(pr) and abs(pr) > pearson_r_threshold:
                    reasons.append(f"Pearson r={pr:.2f} (p={_fmt(prp)})")
                if pd.notna(srp) and srp < p_threshold and pd.notna(sr) and abs(sr) > pearson_r_threshold:
                    reasons.append(f"Spearman ρ={sr:.2f} (p={_fmt(srp)})")
                if not reasons:
                    continue
                sig_rows.append({
                    "GeneSet": row.get("GeneSet", "?"),
                    "pair":    f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}",
                    "n":       int(row.get("n_genes", 0) or 0),
                    "pr": pr, "prp": prp, "sr": sr, "srp": srp, "cos": cos,
                    "_sort":   (prp if pd.notna(prp) else 1.0),
                    "reasons": "; ".join(reasons),
                })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["proximity"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant proximity pair(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  n={r['n']}  "
                    f"r={_fmt(r['pr'],'.3f')} (p={_fmt(r['prp'])})  "
                    f"ρ={_fmt(r['sr'],'.3f')} (p={_fmt(r['srp'])})  "
                    f"cos={_fmt(r['cos'],'.3f')}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant proximity results at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 4 — Pairwise disease statistical tests
    # ==============================================================
    lines += [bar, "  [4] PAIRWISE DISEASE STATISTICAL TESTS", bar]

    pw_files = []
    f_multi = results_dir / "multi_geneset_pairwise_disease_stats.csv"
    if f_multi.exists():
        pw_files.append(f_multi)
    pw_files += [f for f in sorted(results_dir.rglob("pairwise_disease_stats_*.csv"))]

    pw_dfs = []
    for f in pw_files:
        df = _safe_read(f)
        if df.empty:
            continue
        df = _ensure_geneset_col(df, _gs_from_filename(f, "pairwise_disease_stats_"))
        pw_dfs.append(df)

    if not pw_dfs:
        lines += ["  (no pairwise_disease_stats_*.csv files found)", ""]
    else:
        pw_all = pd.concat(pw_dfs, ignore_index=True, sort=False)
        # Deduplicate by GeneSet+pair
        dedup_cols = [c for c in ("GeneSet", "Disease_1", "Disease_2") if c in pw_all.columns]
        if dedup_cols:
            pw_all = pw_all.drop_duplicates(subset=dedup_cols, keep="first")

        # Tests considered (skip concordance — handled in Section 5)
        test_p_cols = [
            ("MWU_p",              "MWU_FDR",           "MWU"),
            ("KS_p",               "KS_FDR",            "KS"),
            ("Welch_p",            "Welch_FDR",         "Welch t"),
            ("Levene_p",           "Levene_FDR",        "Levene"),
            ("BM_p",               "BM_FDR",            "Brunner-Munzel"),
            ("Paired_t_p",         "Paired_t_FDR",      "Paired t"),
            ("Paired_Wilcoxon_p",  "Paired_Wilcoxon_FDR","Paired Wilcoxon"),
            ("Permutation_p",      "Permutation_FDR",   "Sign-flip Perm"),
        ]
        # If multi-mode global FDR exists, prefer it
        for p_col, fdr_col, _ in test_p_cols:
            global_fdr = p_col.replace("_p", "_globalFDR")
            if global_fdr not in pw_all.columns and fdr_col in pw_all.columns:
                pass  # use per-run FDR
            # nothing else to do

        sig_rows = []
        for _, row in pw_all.iterrows():
            pair = f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}"
            n_paired = row.get("n_genes_paired", np.nan)
            cohen_p  = row.get("Cohens_d_paired", np.nan)
            cohen_u  = row.get("Cohens_d_unpaired", np.nan)
            mean_diff = row.get("mean_diff", np.nan)

            row_reasons = []
            best_p = 1.0

            for p_col, fdr_col, name in test_p_cols:
                p_val   = row.get(p_col, np.nan)
                fdr_val = row.get(fdr_col, np.nan)
                # Prefer multi-mode globalFDR if present
                gfdr = row.get(p_col.replace("_p", "_globalFDR"), np.nan)
                if pd.notna(gfdr):
                    fdr_val = gfdr
                gate = fdr_val if (use_fdr and pd.notna(fdr_val)) else p_val
                cutoff = (fdr_threshold
                          if (use_fdr and pd.notna(fdr_val)) else p_threshold)
                if pd.notna(gate) and gate < cutoff:
                    label = "FDR" if (use_fdr and pd.notna(fdr_val)) else "p"
                    row_reasons.append(f"{name} {label}={_fmt(gate)}")
                    if pd.notna(p_val) and p_val < best_p:
                        best_p = p_val

            # Effect-size triggers
            if pd.notna(cohen_p) and abs(cohen_p) > cohens_d_threshold:
                row_reasons.append(f"|Cohen's d (paired)|={abs(cohen_p):.2f}")
            if pd.notna(cohen_u) and abs(cohen_u) > cohens_d_threshold:
                row_reasons.append(f"|Cohen's d (unpaired)|={abs(cohen_u):.2f}")

            if not row_reasons:
                continue

            sig_rows.append({
                "GeneSet": row.get("GeneSet", "?"),
                "pair":    pair,
                "n_paired": int(n_paired) if pd.notna(n_paired) else "?",
                "mean_diff": mean_diff,
                "cohen_p":  cohen_p,
                "cohen_u":  cohen_u,
                "_sort":    best_p,
                "reasons":  "; ".join(row_reasons),
            })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["pairwise_stats"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant pairwise comparison(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  n_paired={r['n_paired']}  "
                    f"mean_diff={_fmt(r['mean_diff'],'.3f')}  "
                    f"Cohen_d(paired)={_fmt(r['cohen_p'],'.2f')}  "
                    f"Cohen_d(unpaired)={_fmt(r['cohen_u'],'.2f')}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant pairwise tests at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 5 — Concordance metrics
    # ==============================================================
    lines += [bar, "  [5] CONCORDANCE METRICS (sign / CCC / Kendall τ)", bar]

    if not pw_dfs:
        lines += ["  (no pairwise files; concordance unavailable)", ""]
    else:
        sig_rows = []
        for _, row in pw_all.iterrows():
            sc_rate = row.get("sign_concordance_rate", np.nan)
            sc_p    = row.get("sign_concordance_p",    np.nan)
            sc_fdr  = row.get("sign_concordance_FDR",  np.nan)
            sc_gfdr = row.get("sign_concordance_globalFDR", np.nan)
            ccc     = row.get("Lin_CCC",     np.nan)
            tau     = row.get("Kendall_tau", np.nan)
            tau_p   = row.get("Kendall_p",   np.nan)
            tau_fdr = row.get("Kendall_FDR", np.nan)
            tau_gfdr= row.get("Kendall_globalFDR", np.nan)

            sc_gate   = sc_gfdr if pd.notna(sc_gfdr) else (sc_fdr if (use_fdr and pd.notna(sc_fdr)) else sc_p)
            sc_cutoff = (fdr_threshold
                         if pd.notna(sc_gfdr) or (use_fdr and pd.notna(sc_fdr))
                         else p_threshold)

            tau_gate   = tau_gfdr if pd.notna(tau_gfdr) else (tau_fdr if (use_fdr and pd.notna(tau_fdr)) else tau_p)
            tau_cutoff = (fdr_threshold
                          if pd.notna(tau_gfdr) or (use_fdr and pd.notna(tau_fdr))
                          else p_threshold)

            reasons = []
            # Sign concordance: significant departure from 0.5 AND |rate-0.5| meaningful
            if (pd.notna(sc_gate) and sc_gate < sc_cutoff
                    and pd.notna(sc_rate)
                    and (sc_rate > sign_concordance_threshold
                         or sc_rate < (1 - sign_concordance_threshold))):
                reasons.append(f"sign_rate={sc_rate:.2f} (p/FDR={_fmt(sc_gate)})")
            # CCC: high agreement
            if pd.notna(ccc) and abs(ccc) > ccc_threshold:
                reasons.append(f"|Lin_CCC|={abs(ccc):.2f}")
            # Kendall τ significant + |τ| above threshold
            if (pd.notna(tau_gate) and tau_gate < tau_cutoff
                    and pd.notna(tau) and abs(tau) > kendall_tau_threshold):
                reasons.append(f"|τ|={abs(tau):.2f} (p/FDR={_fmt(tau_gate)})")

            if not reasons:
                continue

            sig_rows.append({
                "GeneSet": row.get("GeneSet", "?"),
                "pair":    f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}",
                "n_paired": int(row.get("n_genes_paired", 0) or 0),
                "sc_rate": sc_rate, "sc_p": sc_p,
                "ccc":     ccc,
                "tau":     tau, "tau_p": tau_p,
                "_sort":   min(
                    sc_p if pd.notna(sc_p) else 1.0,
                    tau_p if pd.notna(tau_p) else 1.0,
                    1.0 - abs(ccc) if pd.notna(ccc) else 1.0,
                ),
                "reasons": "; ".join(reasons),
            })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["concordance"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant concordance pair(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  n_paired={r['n_paired']}  "
                    f"sign_rate={_fmt(r['sc_rate'],'.3f')} (p={_fmt(r['sc_p'])})  "
                    f"CCC={_fmt(r['ccc'],'.3f')}  τ={_fmt(r['tau'],'.3f')} (p={_fmt(r['tau_p'])})"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant concordance results at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 6 — Gene-level LOO influence
    # ==============================================================
    lines += [bar, "  [6] GENE-LEVEL LOO INFLUENCE (flagged genes)", bar]

    inf_files = []
    f_multi = results_dir / "multi_geneset_gene_influence.csv"
    if f_multi.exists():
        inf_files.append(f_multi)
    inf_files += [f for f in sorted(results_dir.rglob("gene_influence_*.csv"))]

    inf_dfs = []
    for f in inf_files:
        df = _safe_read(f)
        if df.empty:
            continue
        df = _ensure_geneset_col(df, _gs_from_filename(f, "gene_influence_"))
        inf_dfs.append(df)

    if not inf_dfs:
        lines += ["  (no gene_influence_*.csv files found)", ""]
    else:
        inf_all = pd.concat(inf_dfs, ignore_index=True, sort=False)
        dedup_cols = [c for c in ("GeneSet", "Disease", "Gene") if c in inf_all.columns]
        if dedup_cols:
            inf_all = inf_all.drop_duplicates(subset=dedup_cols, keep="first")

        flagged = inf_all[inf_all.get("Influential", False).astype(bool)].copy()
        if flagged.empty:
            lines.append("  (no genes flagged as influential)")
        else:
            flagged["_abs_pct"] = flagged["Pct_Change_Stouffer"].abs()
            flagged = flagged.sort_values(
                ["GeneSet", "Disease", "_abs_pct"], ascending=[True, True, False]
            )
            if top_n_per_section:
                flagged = flagged.head(top_n_per_section)

            section_counts["influence"] = len(flagged)
            lines.append(f"  {len(flagged)} influential gene-disease entries:")
            lines.append("")
            for (gs, dis), sub in flagged.groupby(["GeneSet", "Disease"], sort=False):
                lines.append(f"    [{gs}] {dis}:")
                for _, r in sub.iterrows():
                    sign_flag = " (SIGN-FLIP)" if bool(r.get("Sign_Change", False)) else ""
                    lines.append(
                        f"      • {r.get('Gene','?')}: "
                        f"Z={_fmt(r.get('Z_score'),'.2f')}, "
                        f"ΔStouffer={_fmt(r.get('Delta_Stouffer'),'+.2f')} "
                        f"({_fmt(r.get('Pct_Change_Stouffer'),'.1f')}%)"
                        f"{sign_flag}"
                    )
        lines.append("")

    # ==============================================================
    # SECTION 7 — Family-wise FDR pool
    # ==============================================================
    lines += [bar, "  [7] FAMILY-WISE FDR (pooled across all in-run p-values)", bar]

    fw_files = sorted(results_dir.rglob("family_wise_fdr_*.csv"))
    if not fw_files:
        lines += ["  (no family_wise_fdr_*.csv files found — was the run launched with family_wise_fdr=True?)", ""]
    else:
        sig_rows = []
        for f in fw_files:
            df = _safe_read(f)
            if df.empty:
                continue
            df = _ensure_geneset_col(df, _gs_from_filename(f, "family_wise_fdr_"))
            for _, row in df.iterrows():
                fdr = row.get("family_wise_FDR", np.nan)
                p   = row.get("p_value", np.nan)
                if pd.notna(fdr) and fdr < fdr_threshold:
                    sig_rows.append({
                        "GeneSet":    row.get("GeneSet", "?"),
                        "Source":     row.get("Source", "?"),
                        "Test":       row.get("Test", "?"),
                        "Comparison": row.get("Comparison", "?"),
                        "p":   p,
                        "fdr": fdr,
                        "_sort": fdr,
                    })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["family_wise"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} family-wise FDR-significant entries:")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['Source']}/{r['Test']}  "
                    f"[{r['Comparison']}]  p={_fmt(r['p'])}  FDR={_fmt(r['fdr'])}"
                )
        else:
            lines.append("  (nothing survived family-wise FDR at given threshold)")
        lines.append("")

    # ==============================================================
    # FINAL TALLY
    # ==============================================================
    lines += [bar, "  TOTAL HITS BY SECTION", bar]
    section_titles = {
        "enrichment":      "1. Enrichment            ",
        "differential":    "2. Differential          ",
        "proximity":       "3. Proximity             ",
        "pairwise_stats":  "4. Pairwise stat tests   ",
        "concordance":     "5. Concordance           ",
        "influence":       "6. Influential genes     ",
        "family_wise":     "7. Family-wise FDR       ",
    }
    grand_total = 0
    for k, title in section_titles.items():
        n = section_counts.get(k, 0)
        grand_total += n
        lines.append(f"  {title}: {n}")
    lines.append(f"  ───────────────────────────────")
    lines.append(f"  GRAND TOTAL              : {grand_total}")
    lines.append("")
    lines += [bar]

    # ── write to disk ─────────────────────────────────────────────
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines))
    _print(f"\n✓ Significant-results summary written → {output_path.resolve()}")
    _print(f"  Sections: " + ", ".join(
        f"{k}={section_counts.get(k,0)}" for k in section_titles))
    return output_path.resolve()

In [ ]:
# Filter on raw p-values rather than FDR
generate_sig_summary("/content/089Av2_results", use_fdr=False)


✓ Significant-results summary written → /content/089Av2_results/sig_summary.txt
  Sections: enrichment=144, differential=7, proximity=5, pairwise_stats=20, concordance=2, influence=1954, family_wise=19


PosixPath('/content/089Av2_results/sig_summary.txt')

# Stage 2

In [ ]:
# =====================================================================
#  STAGE-2 DOWNSTREAM PIPELINE (results-only; no S-PrediXcan needed)
#  Consumes outputs of 089Av2_geneset_proximity_(BIP_vs_migraine).py
#  Writes: /content/089Av2_results/downstream/*.csv  +  stage2_summary.txt
# =====================================================================
import os, json, warnings
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
RESULTS   = Path("/content/089Av2_results")
OUT       = RESULTS / "downstream"
OUT.mkdir(parents=True, exist_ok=True)

P_THRESH        = 0.05        # raw p cutoff
FDR_THRESH      = 0.05        # BH-FDR cutoff
Z_THRESH        = 1.96        # |Stouffer Z| / |gene Z| cutoff
MIN_N_GENES     = 10          # min matched genes for a "robust" meta hit
MIN_COVERAGE    = 60.0        # min coverage_pct for a robust meta hit
COHEN_D_THRESH  = 0.5
JACCARD_CUT     = 0.50        # cluster redundancy threshold
CONC_TOP3_CUT   = 0.50        # top-3 |ΔStouffer| fraction => concentrated
ABS_DELTA_CUT   = 0.50        # absolute-ΔStouffer LOO rule
RUN_PERM_PROX   = True        # label-permutation proximity (slower)
PERM_N          = 2000        # permutations per set (raise to 5000 if time allows)
PERM_MIN_GENES  = 5
RNG_SEED        = 42

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    q = p[ok]; order = np.argsort(q); ranked = q[order]; n = len(ranked)
    adj = ranked * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.minimum(adj, 1.0)
    tmp = np.empty(n); tmp[order] = adj; out[ok] = tmp
    return out

def safe_read(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def fmt(v, spec=".4g", na="NA"):
    try:
        if v is None or (isinstance(v, float) and (np.isnan(v))):
            return na
        return format(float(v), spec)
    except Exception:
        return str(v) if v is not None else na

def gini_abs(z):
    x = np.sort(np.abs(np.asarray(z, float)))
    x = x[np.isfinite(x)]
    if len(x) < 3 or x.sum() == 0:
        return np.nan
    n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n

def winsor_mean(x, lo=1, hi=99):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 3:
        return np.nan
    a, b = np.percentile(x, [lo, hi])
    return float(np.mean(np.clip(x, a, b)))

def stouffer(z):
    z = np.asarray(z, float); z = z[np.isfinite(z)]
    return np.nan if len(z) < 3 else float(z.sum() / np.sqrt(len(z)))

META_TISSUE_TAGS = {"meta_across_tissues", "meta"}

# ---------------------------------------------------------------------
# LOAD CORE TABLES
# ---------------------------------------------------------------------
enrich = safe_read(RESULTS / "multi_geneset_enrichment_summary.csv")
pw     = safe_read(RESULTS / "multi_geneset_pairwise_disease_stats.csv")
genes  = safe_read(RESULTS / "all_genes_all_genesets.csv")
inf    = safe_read(RESULTS / "multi_geneset_gene_influence.csv")

load_report = {
    "multi_geneset_enrichment_summary.csv": len(enrich),
    "multi_geneset_pairwise_disease_stats.csv": len(pw),
    "all_genes_all_genesets.csv": len(genes),
    "multi_geneset_gene_influence.csv": len(inf),
}

# detect disease labels + gene Z columns
DISEASES = []
if not enrich.empty and "Disease" in enrich.columns:
    DISEASES = sorted(enrich["Disease"].dropna().unique().tolist())
meta_z_cols = [c for c in genes.columns if c.startswith("Z_") and "__" not in c] if not genes.empty else []
# map disease -> gene column
dis_col = {}
for d in DISEASES:
    c = f"Z_{d}"
    if c in meta_z_cols:
        dis_col[d] = c
# fallback: if exactly two Z_ cols and no disease match
if not dis_col and len(meta_z_cols) >= 2:
    for c in meta_z_cols:
        dis_col[c.replace("Z_", "")] = c
    DISEASES = list(dis_col.keys())

# split meta vs tissue enrichment rows
if not enrich.empty and "Tissue" in enrich.columns:
    meta = enrich[enrich["Tissue"].isin(META_TISSUE_TAGS)].copy()
    tissue = enrich[~enrich["Tissue"].isin(META_TISSUE_TAGS)].copy()
else:
    meta = enrich.copy(); tissue = pd.DataFrame()

SUM = []   # accumulate summary lines
def add(*ls):
    for l in ls:
        SUM.append(l)
BAR = "=" * 84
sub_counts = {}

add(BAR, "  STAGE-2 DOWNSTREAM SUMMARY (results-only)  —  BIP vs migraine",
    f"  Source : {RESULTS.resolve()}",
    f"  Diseases detected : {', '.join(DISEASES) if DISEASES else 'NONE'}",
    f"  Gene sets (meta rows) : {meta['GeneSet'].nunique() if not meta.empty else 0}",
    BAR, "",
    "  INPUT FILES (rows loaded):")
for k, v in load_report.items():
    add(f"    {k:52s}: {v}")
add("",
    "  THRESHOLDS:",
    f"    raw p < {P_THRESH} | FDR < {FDR_THRESH} | |Z| > {Z_THRESH}",
    f"    robust meta hit: n_genes>={MIN_N_GENES}, coverage>={MIN_COVERAGE}%, CI excludes 0",
    f"    Cohen's d > {COHEN_D_THRESH} | Jaccard cluster >= {JACCARD_CUT}",
    f"    concentration top3 fraction > {CONC_TOP3_CUT} | |ΔStouffer| LOO > {ABS_DELTA_CUT}",
    "")

# =====================================================================
# §1  GLOBAL META ENRICHMENT FDR  (all enrichment p) + §24 perm-only
# =====================================================================
long = pd.DataFrame()
one = pd.DataFrame()
if not meta.empty:
    idv = [c for c in ["GeneSet","Disease","Tissue","n_genes_found","n_genes_total",
                       "coverage_pct","stouffer_z","stouffer_ci_low","stouffer_ci_high"]
           if c in meta.columns]
    valcols = [c for c in ["permutation_p","wilcoxon_p"] if c in meta.columns]
    if valcols:
        long = meta.melt(id_vars=idv, value_vars=valcols,
                         var_name="Test", value_name="p_value")
        long = long.dropna(subset=["p_value"]).reset_index(drop=True)
        long["global_FDR"] = bh_fdr(long["p_value"].values)
        long["FDR_within_test"] = long.groupby("Test")["p_value"].transform(lambda x: bh_fdr(x.values))
        long["direction"] = np.where(long["stouffer_z"] > 0, "positive",
                             np.where(long["stouffer_z"] < 0, "negative", "zero"))
        long["robust_enrichment"] = (long["global_FDR"] < FDR_THRESH) & (long["stouffer_z"].abs() > Z_THRESH)
        long.sort_values(["global_FDR","p_value"], inplace=True)
        long.to_csv(OUT/"01_global_meta_enrichment_FDR.csv", index=False)

    # §24 perm-only, one row per set×disease
    if "permutation_p" in meta.columns:
        one = meta.dropna(subset=["permutation_p"]).copy()
        one["FDR_perm_only"] = bh_fdr(one["permutation_p"].values)
        one["pass_perm_only"] = (one["FDR_perm_only"] < FDR_THRESH) & (one["stouffer_z"].abs() > Z_THRESH)
        one.to_csv(OUT/"24_perm_only_FDR.csv", index=False)

# §17 CI + coverage + n gate
if not meta.empty and {"stouffer_ci_low","stouffer_ci_high"}.issubset(meta.columns):
    meta["ci_excludes_zero"] = (
        meta["stouffer_ci_low"].notna() & meta["stouffer_ci_high"].notna() &
        (((meta["stouffer_ci_low"] > 0) & (meta["stouffer_ci_high"] > 0)) |
         ((meta["stouffer_ci_low"] < 0) & (meta["stouffer_ci_high"] < 0))))
else:
    meta["ci_excludes_zero"] = False
if "permutation_p" in meta.columns:
    gated = meta[(meta["permutation_p"] < P_THRESH) & meta["ci_excludes_zero"] &
                 (meta.get("n_genes_found", 0) >= MIN_N_GENES) &
                 (meta.get("coverage_pct", 100).fillna(100) >= MIN_COVERAGE)].copy()
else:
    gated = pd.DataFrame()
if not gated.empty:
    gated.to_csv(OUT/"17_ci_coverage_gated.csv", index=False)

add(BAR, "  [1] GLOBAL META ENRICHMENT (BH across all enrichment p-values)", BAR)
if not long.empty:
    rob = long[long["robust_enrichment"]].drop_duplicates(["GeneSet","Disease","Test"])
    sub_counts["global_enrichment"] = len(rob)
    add(f"  {len(rob)} robust meta findings (global FDR<{FDR_THRESH} & |Z|>{Z_THRESH}):", "")
    for _, r in rob.head(40).iterrows():
        add(f"    • [{r['GeneSet']}] {r['Disease']} ({r['Test']}): "
            f"Z={fmt(r['stouffer_z'],'.3f')}, p={fmt(r['p_value'])}, "
            f"globalFDR={fmt(r['global_FDR'])}, n={int(r.get('n_genes_found',0) or 0)}, dir={r['direction']}")
    if rob.empty:
        add("    (none survived global FDR)")
else:
    add("  (no enrichment p-values available)")
add("")

add(BAR, "  [24] PERM-ONLY FDR (primary family: one perm p per set x disease)", BAR)
if not one.empty:
    passers = one[one["pass_perm_only"]]
    sub_counts["perm_only"] = len(passers)
    add(f"  {len(passers)} perm-only passers (FDR<{FDR_THRESH} & |Z|>{Z_THRESH}):", "")
    for _, r in passers.sort_values("FDR_perm_only").head(40).iterrows():
        add(f"    • [{r['GeneSet']}] {r['Disease']}: Z={fmt(r['stouffer_z'],'.3f')}, "
            f"perm_p={fmt(r['permutation_p'])}, FDR={fmt(r['FDR_perm_only'])}, "
            f"n={int(r.get('n_genes_found',0) or 0)}")
    if passers.empty:
        add("    (none)")
else:
    add("  (no permutation p-values available)")
add("")

add(BAR, "  [17] CI + COVERAGE + N GATE (perm p<0.05, CI excludes 0, n>=%d, cov>=%.0f%%)"
    % (MIN_N_GENES, MIN_COVERAGE), BAR)
sub_counts["ci_gated"] = len(gated)
if not gated.empty:
    add(f"  {len(gated)} rows pass the strict gate:", "")
    for _, r in gated.sort_values("permutation_p").head(40).iterrows():
        add(f"    • [{r['GeneSet']}] {r['Disease']}: Z={fmt(r['stouffer_z'],'.3f')} "
            f"CI95=[{fmt(r.get('stouffer_ci_low'),'.2f')},{fmt(r.get('stouffer_ci_high'),'.2f')}], "
            f"perm_p={fmt(r['permutation_p'])}, n={int(r.get('n_genes_found',0) or 0)}, "
            f"cov={fmt(r.get('coverage_pct'),'.0f')}%")
else:
    add("  (nothing passes the strict CI/coverage gate)")
add("")

# =====================================================================
# §2 TISSUE FDR + §3 DIRECTION CONSISTENCY + §23 CALIBRATION
# =====================================================================
tissue_long = pd.DataFrame(); support = pd.DataFrame(); tdir = pd.DataFrame(); cal = pd.DataFrame()
if not tissue.empty and "permutation_p" in tissue.columns:
    idv = [c for c in ["GeneSet","Disease","Tissue","n_genes_found","coverage_pct","stouffer_z"]
           if c in tissue.columns]
    vc = [c for c in ["permutation_p","wilcoxon_p"] if c in tissue.columns]
    tissue_long = tissue.melt(id_vars=idv, value_vars=vc, var_name="Test", value_name="p_value")
    tissue_long = tissue_long.dropna(subset=["p_value"]).reset_index(drop=True)
    tissue_long["global_FDR"] = bh_fdr(tissue_long["p_value"].values)
    tissue_long["robust_tissue_hit"] = ((tissue_long["global_FDR"] < FDR_THRESH) &
                                        (tissue_long["stouffer_z"].abs() > Z_THRESH) &
                                        (tissue_long.get("n_genes_found",0) >= MIN_N_GENES))
    tissue_long.to_csv(OUT/"02_global_tissue_enrichment_FDR.csv", index=False)
    hit = tissue_long[tissue_long["robust_tissue_hit"]]
    if not hit.empty:
        support = (hit.groupby(["GeneSet","Disease"])
                   .agg(n_supporting_tissues=("Tissue","nunique"),
                        supporting_tissues=("Tissue", lambda x: "; ".join(sorted(set(x)))),
                        min_FDR=("global_FDR","min"),
                        median_Z=("stouffer_z","median"))
                   .reset_index().sort_values(["n_supporting_tissues","min_FDR"],
                                              ascending=[False,True]))
        support.to_csv(OUT/"03_tissue_support_summary.csv", index=False)

if not tissue.empty and "stouffer_z" in tissue.columns:
    rows = []
    for (gs, d), s in tissue.groupby(["GeneSet","Disease"]):
        z = s["stouffer_z"].dropna().values
        if len(z) == 0: continue
        rows.append(dict(GeneSet=gs, Disease=d, n_tissues=len(z),
                         n_pos=int((z>0).sum()), n_neg=int((z<0).sum()),
                         frac_pos=float(np.mean(z>0)), median_tissue_Z=float(np.median(z)),
                         n_absZ_gt=int((np.abs(z)>Z_THRESH).sum()),
                         direction_consistent=bool(np.mean(z>0)>=0.70 or np.mean(z<0)>=0.70)))
    tdir = pd.DataFrame(rows)
    if not tdir.empty:
        tdir.to_csv(OUT/"03_tissue_direction_consistency.csv", index=False)
    # §23 meta vs median-tissue calibration
    med = (tissue.groupby(["GeneSet","Disease"])["stouffer_z"].median()
           .rename("median_tissue_z").reset_index())
    if not meta.empty:
        cal = meta[["GeneSet","Disease","stouffer_z"]].merge(med, on=["GeneSet","Disease"], how="inner")
        cal["same_sign"] = np.sign(cal["stouffer_z"]) == np.sign(cal["median_tissue_z"])
        cal.to_csv(OUT/"23_meta_vs_median_tissue.csv", index=False)

add(BAR, "  [2/3] TISSUE-LEVEL SUPPORT & DIRECTION CONSISTENCY", BAR)
if not support.empty:
    sub_counts["tissue_support"] = int((support["n_supporting_tissues"]>=2).sum())
    add(f"  Sets with >=2 same-direction supporting tissues: {sub_counts['tissue_support']}", "")
    for _, r in support[support["n_supporting_tissues"]>=2].head(40).iterrows():
        add(f"    • [{r['GeneSet']}] {r['Disease']}: {int(r['n_supporting_tissues'])} tissues, "
            f"medZ={fmt(r['median_Z'],'.2f')}, minFDR={fmt(r['min_FDR'])}")
else:
    add("  (no robust multi-tissue support found or no tissue rows)")
add("")
add("  [23] Meta vs median-tissue sign calibration (mismatch = combination artifact):")
if not cal.empty:
    mism = cal[~cal["same_sign"]]
    add(f"    {len(mism)} set x disease with meta/median-tissue SIGN MISMATCH:")
    for _, r in mism.sort_values("stouffer_z", key=lambda s: s.abs(), ascending=False).head(25).iterrows():
        add(f"      • [{r['GeneSet']}] {r['Disease']}: metaZ={fmt(r['stouffer_z'],'.2f')} "
            f"vs medianTissueZ={fmt(r['median_tissue_z'],'.2f')}")
    if mism.empty:
        add("      (none — meta signs agree with median tissue)")
else:
    add("    (no tissue data for calibration)")
add("")

# =====================================================================
# §5/§18  REDUNDANCY (observed membership) + COLLAPSE-THEN-FDR
# =====================================================================
clusters = {}; collapsed = pd.DataFrame(); jac_df = pd.DataFrame()
if not genes.empty and meta_z_cols:
    memb = (genes.dropna(subset=meta_z_cols, how="all")
            .groupby("GeneSet")["Gene"].apply(lambda s: set(map(str, s))))
    labels = list(memb.index)
    if len(labels) >= 2:
        n = len(labels)
        J = np.eye(n)
        for i, j in combinations(range(n), 2):
            a, b = memb[labels[i]], memb[labels[j]]
            u = len(a | b); jv = len(a & b) / u if u else 0.0
            J[i, j] = J[j, i] = jv
        jac_df = pd.DataFrame(J, index=labels, columns=labels)
        jac_df.to_csv(OUT/"05_jaccard_observed_membership.csv")
        # union-find at JACCARD_CUT
        parent = {l: l for l in labels}
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]; x = parent[x]
            return x
        def union(a, b):
            parent[find(a)] = find(b)
        for i, j in combinations(range(n), 2):
            if J[i, j] >= JACCARD_CUT:
                union(labels[i], labels[j])
        for l in labels:
            clusters.setdefault(find(l), []).append(l)
        # representative = largest n_genes_found in meta
        nfound = (meta.groupby("GeneSet")["n_genes_found"].max().to_dict()
                  if "n_genes_found" in meta.columns else {})
        rep_map = {}
        for root, members in clusters.items():
            rep = max(members, key=lambda g: nfound.get(g, 0))
            for m in members:
                rep_map[m] = rep
        clu_df = pd.DataFrame({"GeneSet": list(rep_map.keys()),
                               "cluster_rep": list(rep_map.values())})
        clu_df.to_csv(OUT/"18_cluster_assignment.csv", index=False)
        # collapse-then-FDR (perm only, representatives only)
        if not one.empty:
            c = one.merge(clu_df, on="GeneSet", how="left")
            c["cluster_rep"] = c["cluster_rep"].fillna(c["GeneSet"])
            collapsed = c[c["GeneSet"] == c["cluster_rep"]].copy()
            if not collapsed.empty:
                collapsed["FDR_collapsed"] = bh_fdr(collapsed["permutation_p"].values)
                collapsed["pass_collapsed"] = ((collapsed["FDR_collapsed"] < FDR_THRESH) &
                                               (collapsed["stouffer_z"].abs() > Z_THRESH))
                collapsed.to_csv(OUT/"18_collapsed_perm_FDR.csv", index=False)

add(BAR, "  [5/18] REDUNDANCY MODULES + COLLAPSE-THEN-FDR", BAR)
if clusters:
    multi = {r: m for r, m in clusters.items() if len(m) > 1}
    add(f"  {len(clusters)} clusters at Jaccard>={JACCARD_CUT} "
        f"({len(multi)} multi-set clusters):")
    for root, members in sorted(multi.items(), key=lambda kv: -len(kv[1]))[:20]:
        rep = max(members, key=lambda g: (meta.groupby('GeneSet')['n_genes_found'].max().to_dict().get(g,0) if 'n_genes_found' in meta.columns else 0))
        add(f"    • rep={rep}  <= {{{', '.join(sorted(members))}}}")
    add("")
if not collapsed.empty:
    cp = collapsed[collapsed["pass_collapsed"]]
    sub_counts["collapsed"] = len(cp)
    add(f"  Collapsed (non-redundant) perm-FDR passers: {len(cp)}", "")
    for _, r in cp.sort_values("FDR_collapsed").head(40).iterrows():
        add(f"    • [{r['GeneSet']}] {r['Disease']}: Z={fmt(r['stouffer_z'],'.3f')}, "
            f"FDR_collapsed={fmt(r['FDR_collapsed'])}")
    if cp.empty:
        add("    (none survive collapsed FDR)")
add("")

# =====================================================================
# §6-8  PAIRED GENE CONTRASTS + ROBUST vs CLASSICAL + CLASSIFY
# =====================================================================
contrast = pd.DataFrame(); robust_cmp = pd.DataFrame(); classcnt = pd.DataFrame()
if not genes.empty and len(dis_col) == 2:
    d1, d2 = list(dis_col.keys())[:2]
    c1, c2 = dis_col[d1], dis_col[d2]

    # per-gene contrast table
    g = genes.copy()
    g["diff"] = g[c1] - g[c2]
    g["same_direction"] = np.sign(g[c1]) == np.sign(g[c2])
    g["opposite_direction"] = np.sign(g[c1]) == -np.sign(g[c2])
    g.to_csv(OUT/"06_gene_contrasts.csv", index=False)

    def classify(row):
        b, m = row[c1], row[c2]
        if not (np.isfinite(b) and np.isfinite(m)): return "missing"
        if b > 0 and m > 0: return "positive_both"
        if b < 0 and m < 0: return "negative_both"
        if b > 0 and m < 0: return f"{d1}_pos_{d2}_neg"
        if b < 0 and m > 0: return f"{d1}_neg_{d2}_pos"
        return "near_zero"
    g["contrast_class"] = g.apply(classify, axis=1)
    classcnt = (g.groupby(["GeneSet","contrast_class"]).size().unstack(fill_value=0))
    classcnt.to_csv(OUT/"06_contrast_class_counts.csv")

    # per-set paired location summary (§7) + robust vs classical (§8)
    rows = []; rrows = []
    for gs, s in g.groupby("GeneSet"):
        pair = s[[c1, c2]].dropna()
        if len(pair) < 3: continue
        b = pair[c1].values; m = pair[c2].values; diff = b - m
        sd = np.std(diff, ddof=1)
        paired_d = float(np.mean(diff) / sd) if sd > 0 else np.nan
        nz = diff != 0
        sign_frac = float(np.mean(diff[nz] > 0)) if nz.any() else np.nan
        # Bland-Altman scale dependence (§20)
        try:
            ba_r = stats.pearsonr((b+m)/2, diff)[0]
        except Exception:
            ba_r = np.nan
        rows.append(dict(GeneSet=gs, n_paired=len(diff),
                         mean_diff=float(np.mean(diff)),
                         median_diff=float(np.median(diff)),
                         trimmed_diff=float(stats.trim_mean(diff, 0.10)),
                         paired_cohens_d=paired_d,
                         frac_d1_greater=sign_frac,
                         blandaltman_r=ba_r,
                         gini_absZ_d1=gini_abs(b), gini_absZ_d2=gini_abs(m),
                         maxmed_d1=(np.max(np.abs(b))/np.median(np.abs(b)) if np.median(np.abs(b))>0 else np.nan)))
        rrows.append(dict(GeneSet=gs,
                          mean_diff=float(np.mean(diff)),
                          median_diff=float(np.median(diff)),
                          trimmed_diff=float(stats.trim_mean(diff, 0.10)),
                          winsor_diff=winsor_mean(b) - winsor_mean(m)))
    contrast = pd.DataFrame(rows)
    if not contrast.empty:
        contrast.to_csv(OUT/"07_paired_contrast_summary.csv", index=False)
    robust_cmp = pd.DataFrame(rrows)
    if not robust_cmp.empty:
        robust_cmp["direction_stable"] = (
            (np.sign(robust_cmp["mean_diff"]) == np.sign(robust_cmp["median_diff"])) &
            (np.sign(robust_cmp["mean_diff"]) == np.sign(robust_cmp["trimmed_diff"])) &
            (np.sign(robust_cmp["mean_diff"]) == np.sign(robust_cmp["winsor_diff"])))
        robust_cmp.to_csv(OUT/"08_robust_vs_classical.csv", index=False)

add(BAR, f"  [6/7/8] PAIRED GENE CONTRASTS ({DISEASES[0] if DISEASES else 'D1'} - {DISEASES[1] if len(DISEASES)>1 else 'D2'})", BAR)
if not contrast.empty:
    top = contrast.reindex(contrast["paired_cohens_d"].abs().sort_values(ascending=False).index)
    add("  Top |paired Cohen's d| gene sets (direction-stable flag from robust check):", "")
    stab = dict(zip(robust_cmp["GeneSet"], robust_cmp["direction_stable"])) if not robust_cmp.empty else {}
    for _, r in top.head(25).iterrows():
        st = stab.get(r["GeneSet"], None)
        st_s = "stable" if st else ("UNSTABLE" if st is False else "?")
        add(f"    • [{r['GeneSet']}] n={int(r['n_paired'])} d_paired={fmt(r['paired_cohens_d'],'.3f')} "
            f"meanΔ={fmt(r['mean_diff'],'.3f')} medΔ={fmt(r['median_diff'],'.3f')} "
            f"BA_r={fmt(r['blandaltman_r'],'.2f')} giniZ={fmt(r['gini_absZ_d1'],'.2f')} [{st_s}]")
    unstable = [k for k,v in stab.items() if v is False]
    sub_counts["contrast_sets"] = len(top)
    add("", f"  Direction-UNSTABLE (mean vs median/trimmed/winsor disagree): {len(unstable)}")
    if unstable:
        add("    " + ", ".join(unstable[:30]))
else:
    add("  (need exactly 2 disease Z columns in all_genes_all_genesets.csv)")
add("")

# =====================================================================
# §9  PROFILE GEOMETRY + DROP-2 ROBUSTNESS
# =====================================================================
geometry = pd.DataFrame()
if not genes.empty and len(dis_col) == 2:
    d1, d2 = list(dis_col.keys())[:2]; c1, c2 = dis_col[d1], dis_col[d2]
    def prof(s, drop_top=0):
        x = s[c1].values.astype(float); y = s[c2].values.astype(float)
        k = np.isfinite(x) & np.isfinite(y); x, y = x[k], y[k]
        if drop_top > 0 and len(x) > drop_top:
            infl = np.maximum(np.abs(x), np.abs(y))
            order = np.argsort(infl)[::-1]; keep = np.ones(len(x), bool); keep[order[:drop_top]] = False
            x, y = x[keep], y[keep]
        if len(x) < 3: return None
        cos = float(np.dot(x, y) / (np.linalg.norm(x)*np.linalg.norm(y) + 1e-12))
        return dict(n=len(x), pearson=float(stats.pearsonr(x,y)[0]),
                    spearman=float(stats.spearmanr(x,y)[0]), cosine=cos,
                    frac_opp=float(np.mean(np.sign(x)==-np.sign(y))))
    rows = []
    for gs, s in genes.groupby("GeneSet"):
        f0 = prof(s, 0); f2 = prof(s, 2)
        if f0 is None: continue
        row = {"GeneSet": gs}
        for k, v in f0.items(): row[f"full_{k}"] = v
        if f2:
            for k, v in f2.items(): row[f"drop2_{k}"] = v
        rows.append(row)
    geometry = pd.DataFrame(rows)
    if not geometry.empty and "drop2_pearson" in geometry.columns:
        geometry["corr_sign_stable"] = np.sign(geometry["full_pearson"]) == np.sign(geometry["drop2_pearson"])
        geometry.to_csv(OUT/"09_profile_geometry_robustness.csv", index=False)

add(BAR, "  [9] PROFILE GEOMETRY (BIP vs migraine correlation, drop-2 robustness)", BAR)
if not geometry.empty:
    strong = geometry.reindex(geometry["full_pearson"].abs().sort_values(ascending=False).index)
    add("  Strongest profile correlations (sign stability after dropping top-2 genes):", "")
    for _, r in strong.head(25).iterrows():
        st = r.get("corr_sign_stable", None)
        add(f"    • [{r['GeneSet']}] n={int(r['full_n'])} r={fmt(r['full_pearson'],'.3f')} "
            f"(drop2 r={fmt(r.get('drop2_pearson'),'.3f')}, "
            f"{'stable' if st else 'FLIP'}) cos={fmt(r['full_cosine'],'.3f')} "
            f"frac_opp={fmt(r['full_frac_opp'],'.2f')}")
else:
    add("  (insufficient gene data)")
add("")

# =====================================================================
# §10  LABEL-PERMUTATION PROFILE PROXIMITY (empirical null)
# =====================================================================
perm_prox = pd.DataFrame()
if RUN_PERM_PROX and not genes.empty and len(dis_col) == 2:
    d1, d2 = list(dis_col.keys())[:2]; c1, c2 = dis_col[d1], dis_col[d2]
    rng = np.random.default_rng(RNG_SEED)
    rows = []
    for gs, s in genes.groupby("GeneSet"):
        x = s[c1].values.astype(float); y = s[c2].values.astype(float)
        k = np.isfinite(x) & np.isfinite(y); x, y = x[k], y[k]
        if len(x) < PERM_MIN_GENES: continue
        obs = float(stats.pearsonr(x, y)[0])
        null = np.empty(PERM_N)
        for i in range(PERM_N):
            null[i] = stats.pearsonr(x, rng.permutation(y))[0]
        p = (1 + np.sum(np.abs(null) >= abs(obs))) / (PERM_N + 1)
        rows.append(dict(GeneSet=gs, n=len(x), observed_r=obs, empirical_p=float(p),
                         null_q025=float(np.quantile(null,0.025)),
                         null_q975=float(np.quantile(null,0.975))))
    perm_prox = pd.DataFrame(rows)
    if not perm_prox.empty:
        perm_prox["FDR"] = bh_fdr(perm_prox["empirical_p"].values)
        perm_prox.sort_values("FDR", inplace=True)
        perm_prox.to_csv(OUT/"10_permuted_profile_proximity.csv", index=False)

add(BAR, "  [10] LABEL-PERMUTATION PROFILE PROXIMITY (secondary null)", BAR)
if not perm_prox.empty:
    sig = perm_prox[perm_prox["FDR"] < FDR_THRESH]
    sub_counts["perm_proximity"] = len(sig)
    add(f"  {len(sig)} sets with empirical proximity FDR<{FDR_THRESH} (perm={PERM_N}):", "")
    for _, r in sig.head(30).iterrows():
        add(f"    • [{r['GeneSet']}] n={int(r['n'])} r={fmt(r['observed_r'],'.3f')} "
            f"emp_p={fmt(r['empirical_p'])} FDR={fmt(r['FDR'])} "
            f"null95=[{fmt(r['null_q025'],'.2f')},{fmt(r['null_q975'],'.2f')}]")
    if sig.empty:
        add("    (none significant vs label-permutation null)")
else:
    add("  (skipped or insufficient data)")
add("")

# =====================================================================
# §11 / §22  MITOCHONDRIAL VARIANCE-vs-LOCATION + PAIRWISE VOLCANO
# =====================================================================
mito_names = ["MOOTHA_MITOCHONDRIA_ALL","MOOTHA_PGC_ALL","HALLMARK_OXIDATIVE_PHOSPHORYLATION_ALL"]
mito_out = pd.DataFrame(); volcano = pd.DataFrame()
if not pw.empty:
    if "Paired_Wilcoxon_p" in pw.columns:
        pw = pw.copy()
        pw["FDR_paired"] = bh_fdr(pw["Paired_Wilcoxon_p"].values)
        pw["neglog10_paired_p"] = -np.log10(pw["Paired_Wilcoxon_p"].clip(lower=1e-300))
        volcano = pw[["GeneSet","Disease_1","Disease_2","mean_diff","Cohens_d_paired",
                      "Paired_Wilcoxon_p","FDR_paired","neglog10_paired_p"]].copy() \
                    if {"mean_diff","Cohens_d_paired"}.issubset(pw.columns) else pd.DataFrame()
        if not volcano.empty:
            volcano.to_csv(OUT/"22_pairwise_volcano.csv", index=False)
    mp = pw[pw["GeneSet"].isin(mito_names)].copy()
    if not mp.empty:
        mp["location_sig"] = ((mp.get("Welch_p", np.nan) < P_THRESH) |
                              (mp.get("Paired_Wilcoxon_p", np.nan) < P_THRESH))
        mp["variance_sig"] = mp.get("Levene_p", np.nan) < P_THRESH
        mito_out = mp
        mito_out.to_csv(OUT/"11_mito_location_vs_variance.csv", index=False)

# variance ratio from gene table
if not genes.empty and len(dis_col) == 2:
    d1, d2 = list(dis_col.keys())[:2]; c1, c2 = dis_col[d1], dis_col[d2]
    vrows = []
    for gs, s in genes[genes["GeneSet"].isin(mito_names)].groupby("GeneSet"):
        b = s[c1].dropna().values; m = s[c2].dropna().values
        if len(b) > 1 and len(m) > 1:
            vrows.append(dict(GeneSet=gs, var_d1=float(np.var(b,ddof=1)),
                              var_d2=float(np.var(m,ddof=1)),
                              var_ratio=float(np.var(b,ddof=1)/np.var(m,ddof=1))))
    if vrows:
        pd.DataFrame(vrows).to_csv(OUT/"11_mito_variance_ratio.csv", index=False)

add(BAR, "  [11] MITOCHONDRIAL: LOCATION vs VARIANCE", BAR)
if not mito_out.empty:
    for _, r in mito_out.iterrows():
        add(f"    • [{r['GeneSet']}] {r.get('Disease_1','?')} vs {r.get('Disease_2','?')}: "
            f"meanΔ={fmt(r.get('mean_diff'),'.3f')} d_unpaired={fmt(r.get('Cohens_d_unpaired'),'.2f')} "
            f"Welch_p={fmt(r.get('Welch_p'))} PairedW_p={fmt(r.get('Paired_Wilcoxon_p'))} "
            f"Levene_p={fmt(r.get('Levene_p'))} | loc_sig={bool(r['location_sig'])} "
            f"var_sig={bool(r['variance_sig'])}")
else:
    add("  (no mitochondrial rows in pairwise file)")
add("")
add(BAR, "  [22] PAIRWISE CONTRAST VOLCANO (paired Wilcoxon FDR)", BAR)
if not volcano.empty:
    vsig = volcano[volcano["FDR_paired"] < FDR_THRESH]
    sub_counts["volcano"] = len(vsig)
    add(f"  {len(vsig)} sets with paired-Wilcoxon FDR<{FDR_THRESH}:", "")
    for _, r in vsig.reindex(vsig["Cohens_d_paired"].abs().sort_values(ascending=False).index).head(30).iterrows():
        add(f"    • [{r['GeneSet']}] {r['Disease_1']} vs {r['Disease_2']}: "
            f"meanΔ={fmt(r['mean_diff'],'.3f')} d_paired={fmt(r['Cohens_d_paired'],'.2f')} "
            f"p={fmt(r['Paired_Wilcoxon_p'])} FDR={fmt(r['FDR_paired'])}")
    if vsig.empty:
        add("    (none survive paired-Wilcoxon FDR)")
else:
    add("  (no paired-Wilcoxon column in pairwise file)")
add("")

# =====================================================================
# §13/§14/§21  INFLUENCE CONCENTRATION + ABSOLUTE-ΔZ LOO
# =====================================================================
conc = pd.DataFrame(); robust_inf = pd.DataFrame()
if not inf.empty and "Delta_Stouffer" in inf.columns:
    inf = inf.copy()
    inf["abs_delta"] = inf["Delta_Stouffer"].abs()
    rows = []
    for (gs, d), s in inf.groupby(["GeneSet","Disease"]):
        s = s.sort_values("abs_delta", ascending=False)
        tot = s["abs_delta"].sum()
        if tot == 0: continue
        rows.append(dict(GeneSet=gs, Disease=d, n_genes=len(s),
                         n_flagged=int(s.get("Influential", False).sum()),
                         top1=s.head(1)["abs_delta"].sum()/tot,
                         top3=s.head(3)["abs_delta"].sum()/tot,
                         top5=s.head(5)["abs_delta"].sum()/tot,
                         max_abs_delta=float(s["abs_delta"].max()),
                         sign_flips=int(s.get("Sign_Change", False).sum())))
    conc = pd.DataFrame(rows)
    if not conc.empty:
        conc["concentrated_top3"] = conc["top3"] > CONC_TOP3_CUT
        conc.to_csv(OUT/"13_influence_concentration.csv", index=False)
    # §14 absolute-ΔZ robust flag
    inf["abs_change_flag"] = inf["Delta_Stouffer"].abs() > ABS_DELTA_CUT
    inf["robust_influential"] = inf.get("Sign_Change", False).astype(bool) | inf["abs_change_flag"]
    robust_inf = (inf.groupby(["GeneSet","Disease"])
                  .agg(n_flagged_original=("Influential","sum"),
                       n_flagged_absolute=("abs_change_flag","sum"),
                       n_flagged_robust=("robust_influential","sum"),
                       n_sign_flips=("Sign_Change","sum"),
                       max_abs_delta=("Delta_Stouffer", lambda x: float(np.max(np.abs(x)))))
                  .reset_index())
    robust_inf.to_csv(OUT/"14_robust_influence_summary.csv", index=False)

add(BAR, "  [13/14] INFLUENCE CONCENTRATION (outlier-driven detection)", BAR)
if not conc.empty:
    concentrated = conc[conc["concentrated_top3"]]
    sub_counts["concentrated"] = len(concentrated)
    add(f"  {len(concentrated)} set x disease are top-3 concentrated (>{CONC_TOP3_CUT} of |ΔStouffer|):", "")
    for _, r in concentrated.sort_values("top3", ascending=False).head(30).iterrows():
        add(f"    • [{r['GeneSet']}] {r['Disease']}: top3={fmt(r['top3'],'.2f')} "
            f"top1={fmt(r['top1'],'.2f')} n={int(r['n_genes'])} sign_flips={int(r['sign_flips'])}")
    if concentrated.empty:
        add("    (no strongly concentrated sets)")
else:
    add("  (no influence table)")
add("")

# =====================================================================
# §19  RECURRENT DRIVER GENES (same gene across many sets)
# =====================================================================
drivers = {}; opp_drivers = pd.DataFrame()
if not genes.empty and len(dis_col) == 2:
    d1, d2 = list(dis_col.keys())[:2]; c1, c2 = dis_col[d1], dis_col[d2]
    for d, col in [(d1, c1), (d2, c2)]:
        s = genes.dropna(subset=[col])
        s = s[s[col].abs() >= Z_THRESH]
        dd = (s.groupby("Gene").agg(n_sets=("GeneSet","nunique"),
                                    median_Z=(col,"median"),
                                    sets=("GeneSet", lambda x: "; ".join(sorted(set(x))[:8])))
              .query("n_sets >= 3").sort_values("n_sets", ascending=False))
        drivers[d] = dd
        if not dd.empty:
            dd.to_csv(OUT/f"19_recurrent_drivers_{d}.csv")
    # recurrent opposite-sign genes
    gg = genes.dropna(subset=[c1, c2]).copy()
    gg = gg[(np.sign(gg[c1]) != np.sign(gg[c2])) &
            (np.minimum(gg[c1].abs(), gg[c2].abs()) >= Z_THRESH)]
    if not gg.empty:
        opp_drivers = (gg.groupby("Gene").agg(n_sets=("GeneSet","nunique"),
                                              Z_d1=(c1,"median"), Z_d2=(c2,"median"),
                                              sets=("GeneSet", lambda x: "; ".join(sorted(set(x))[:8])))
                       .query("n_sets >= 2").sort_values("n_sets", ascending=False))
        opp_drivers.to_csv(OUT/"19_recurrent_opposite_sign_genes.csv")

add(BAR, "  [19] RECURRENT DRIVER GENES (same gene inflating many sets)", BAR)
for d, dd in drivers.items():
    add(f"  {d}: {len(dd)} genes with |Z|>={Z_THRESH} in >=3 sets")
    for gene, r in dd.head(15).iterrows():
        add(f"    • {gene}: {int(r['n_sets'])} sets, medZ={fmt(r['median_Z'],'.2f')}")
if not opp_drivers.empty:
    add("", f"  Recurrent OPPOSITE-SIGN genes (colocalization shortlist): {len(opp_drivers)}")
    for gene, r in opp_drivers.head(20).iterrows():
        add(f"    • {gene}: {int(r['n_sets'])} sets | "
            f"{DISEASES[0]}Z={fmt(r['Z_d1'],'.2f')} {DISEASES[1]}Z={fmt(r['Z_d2'],'.2f')}")
add("")

# =====================================================================
# §15/§16  EVIDENCE CLASSIFICATION + PUBLICATION TABLE
# =====================================================================
evidence = pd.DataFrame()
if not long.empty:
    meta_hits = (long[(long["global_FDR"] < FDR_THRESH) & (long["stouffer_z"].abs() > Z_THRESH)]
                 .groupby(["GeneSet","Disease"])
                 .agg(best_FDR=("global_FDR","min"),
                      meta_direction=("stouffer_z", lambda x: "positive" if np.median(x)>0 else "negative"),
                      max_abs_meta_Z=("stouffer_z", lambda x: float(np.max(np.abs(x)))))
                 .reset_index())
    tsup = (tdir[tdir["direction_consistent"]]
            .groupby(["GeneSet","Disease"])
            .agg(n_tissue_consistent=("n_tissues","size"),
                 median_tissue_Z=("median_tissue_Z","median")).reset_index()) if not tdir.empty else pd.DataFrame(columns=["GeneSet","Disease","n_tissue_consistent","median_tissue_Z"])
    ci = conc[["GeneSet","Disease","top3","sign_flips"]].rename(columns={"top3":"top3_influence"}) if not conc.empty else pd.DataFrame(columns=["GeneSet","Disease","top3_influence","sign_flips"])
    ev = meta_hits.merge(tsup, on=["GeneSet","Disease"], how="left") \
                  .merge(ci, on=["GeneSet","Disease"], how="left")
    ev["n_tissue_consistent"] = ev["n_tissue_consistent"].fillna(0)
    # attach gates
    if not one.empty:
        ev = ev.merge(one[["GeneSet","Disease","FDR_perm_only"]], on=["GeneSet","Disease"], how="left")
    if not collapsed.empty:
        ev = ev.merge(collapsed[["GeneSet","Disease","FDR_collapsed"]], on=["GeneSet","Disease"], how="left")
    if "ci_excludes_zero" in meta.columns:
        ev = ev.merge(meta[["GeneSet","Disease","ci_excludes_zero"]].drop_duplicates(),
                      on=["GeneSet","Disease"], how="left")

    def classify_ev(r):
        top3 = r.get("top3_influence", np.nan)
        if pd.notna(top3) and top3 >= CONC_TOP3_CUT:
            return "outlier_sensitive"
        if (r["best_FDR"] < FDR_THRESH) and (r["n_tissue_consistent"] >= 2) and (pd.isna(top3) or top3 < CONC_TOP3_CUT):
            return "robust_multi_tissue"
        if (r["best_FDR"] < FDR_THRESH) and (r["n_tissue_consistent"] < 2):
            return "meta_level_only"
        return "hypothesis_generating"
    ev["classification"] = ev.apply(classify_ev, axis=1)
    evidence = ev.sort_values(["classification","best_FDR"])
    evidence.to_csv(OUT/"15_evidence_classification.csv", index=False)

    pub = evidence.rename(columns={
        "GeneSet":"Gene_set","Disease":"Condition","best_FDR":"Global_FDR",
        "max_abs_meta_Z":"Max_abs_meta_Z","n_tissue_consistent":"Consistent_tissues",
        "median_tissue_Z":"Median_tissue_Z","top3_influence":"Top3_influence",
        "sign_flips":"LOO_sign_flips","classification":"Evidence_class"})
    keep = [c for c in ["Gene_set","Condition","meta_direction","Global_FDR","FDR_perm_only",
                        "FDR_collapsed","ci_excludes_zero","Max_abs_meta_Z","Consistent_tissues",
                        "Median_tissue_Z","Top3_influence","LOO_sign_flips","Evidence_class"] if c in pub.columns]
    pub[keep].to_csv(OUT/"16_publication_ready.csv", index=False)

add(BAR, "  [15/16] EVIDENCE CLASSIFICATION", BAR)
if not evidence.empty:
    order = ["robust_multi_tissue","meta_level_only","outlier_sensitive","hypothesis_generating"]
    for cls in order:
        sub = evidence[evidence["classification"] == cls]
        sub_counts[f"class_{cls}"] = len(sub)
        add(f"  {cls.upper()}: {len(sub)}")
        for _, r in sub.head(30).iterrows():
            extra = []
            if "FDR_perm_only" in r and pd.notna(r["FDR_perm_only"]): extra.append(f"permFDR={fmt(r['FDR_perm_only'])}")
            if "FDR_collapsed" in r and pd.notna(r.get("FDR_collapsed")): extra.append(f"collFDR={fmt(r['FDR_collapsed'])}")
            if "ci_excludes_zero" in r: extra.append(f"CIexcl0={bool(r.get('ci_excludes_zero'))}")
            add(f"      • [{r['GeneSet']}] {r['Disease']} ({r['meta_direction']}): "
                f"globalFDR={fmt(r['best_FDR'])}, maxZ={fmt(r['max_abs_meta_Z'],'.2f')}, "
                f"tissues={int(r['n_tissue_consistent'])}, top3={fmt(r.get('top3_influence'),'.2f')} "
                + (" ".join(extra)))
        add("")
else:
    add("  (no meta hits to classify)")
add("")

# =====================================================================
# TOP-LINE: sets that clear the strict reporting bar
# =====================================================================
add(BAR, "  ★ REPORTABLE SETS (perm-only OR collapsed FDR pass, CI excludes 0,",
    "     not top-3 concentrated, AND multi-tissue OR stable paired contrast)", BAR)
reportable = []
if not evidence.empty:
    stab = dict(zip(robust_cmp["GeneSet"], robust_cmp["direction_stable"])) if not robust_cmp.empty else {}
    for _, r in evidence.iterrows():
        perm_pass = (pd.notna(r.get("FDR_perm_only")) and r["FDR_perm_only"] < FDR_THRESH)
        coll_pass = (pd.notna(r.get("FDR_collapsed")) and r["FDR_collapsed"] < FDR_THRESH)
        ci_ok = bool(r.get("ci_excludes_zero", False))
        not_conc = not (pd.notna(r.get("top3_influence")) and r["top3_influence"] >= CONC_TOP3_CUT)
        multi_tissue = r.get("n_tissue_consistent", 0) >= 2
        stable_contrast = bool(stab.get(r["GeneSet"], False))
        if (perm_pass or coll_pass) and ci_ok and not_conc and (multi_tissue or stable_contrast):
            reportable.append(r)
            add(f"    ✔ [{r['GeneSet']}] {r['Disease']} ({r['meta_direction']}): "
                f"maxZ={fmt(r['max_abs_meta_Z'],'.2f')}, permFDR={fmt(r.get('FDR_perm_only'))}, "
                f"collFDR={fmt(r.get('FDR_collapsed'))}, tissues={int(r.get('n_tissue_consistent',0))}, "
                f"stable_contrast={stable_contrast}")
sub_counts["reportable"] = len(reportable)
if not reportable:
    add("    (none clear the full reporting bar — treat all hits as hypothesis-generating)")
add("")

# =====================================================================
# FINAL TALLY
# =====================================================================
add(BAR, "  TALLY", BAR)
for k in sorted(sub_counts):
    add(f"    {k:28s}: {sub_counts[k]}")
add(BAR)
add("  Interpretation guardrails:",
    "   - Do NOT count tissue rows and meta rows together.",
    "   - Do NOT treat every 'Influential' flag as a gene list (near-zero Stouffer inflates %change).",
    "   - Ignore CCC / Spearman p on n<=4 sets (NAD transport, WP GLP-1).",
    "   - Colocalization / PrediXcan R2 / correlation-aware set tests require the original S-PrediXcan files.",
    BAR)

# ---------------------------------------------------------------------
# WRITE SUMMARY
# ---------------------------------------------------------------------
summary_path = OUT / "stage2_summary.txt"
summary_path.write_text("\n".join(SUM))
print("\n".join(SUM[:60]))
print("\n...\n")
print(f"✓ Detailed summary written → {summary_path.resolve()}")
print(f"✓ All intermediate CSVs   → {OUT.resolve()}/")
print(f"  Files written: {sorted(p.name for p in OUT.glob('*.csv'))}")

  STAGE-2 DOWNSTREAM SUMMARY (results-only)  —  BIP vs migraine
  Source : /content/089Av2_results
  Diseases detected : bip, migraine
  Gene sets (meta rows) : 51

  INPUT FILES (rows loaded):
    multi_geneset_enrichment_summary.csv                : 714
    multi_geneset_pairwise_disease_stats.csv            : 51
    all_genes_all_genesets.csv                          : 4107
    multi_geneset_gene_influence.csv                    : 5980

  THRESHOLDS:
    raw p < 0.05 | FDR < 0.05 | |Z| > 1.96
    robust meta hit: n_genes>=10, coverage>=60.0%, CI excludes 0
    Cohen's d > 0.5 | Jaccard cluster >= 0.5
    concentration top3 fraction > 0.5 | |ΔStouffer| LOO > 0.5

  [1] GLOBAL META ENRICHMENT (BH across all enrichment p-values)
  0 robust meta findings (global FDR<0.05 & |Z|>1.96):

    (none survived global FDR)

  [24] PERM-ONLY FDR (primary family: one perm p per set x disease)
  0 perm-only passers (FDR<0.05 & |Z|>1.96):

    (none)

  [17] CI + COVERAGE + N GATE (perm p<0.05, CI 

# Stage 3

In [ ]:
# =====================================================================
#  STAGE-3 GENE-LEVEL PIPELINE (results-only; no S-PrediXcan needed)
#  Consumes outputs of 089Av2_geneset_proximity_(BIP_vs_migraine).py
#  (+ optional Stage-2 downstream) and works at UNIQUE-GENE resolution.
#  Writes: /content/089Av2_results/downstream_stage3/*.csv + stage3_summary.txt
# =====================================================================
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
RESULTS = Path("/content/089Av2_results")
OUT     = RESULTS / "downstream_stage3"
OUT.mkdir(parents=True, exist_ok=True)

Z_THRESH        = 1.96      # |Z| threshold to "clear"
STRONG_Z        = 3.0       # narrative "strong" threshold
REPEATER_NSETS  = 3         # gene in >= this many sets = redundancy repeater
OPP_MIN_ROWS    = 2         # Layer A: opposite in >= this many set-rows
GATED_TOP_N     = 25        # top-|Z| genes pulled from each gated set

# The four robustness-gated sets from Stage-2 (set, disease).
# Edit if your Stage-2 gate produced different survivors.
GATED = {
    ("KEGG_LTP_ALL", "bip"),
    ("REACTOME_MITOPHAGY", "bip"),
    ("HALLMARK_OXIDATIVE_PHOSPHORYLATION_ALL", "migraine"),
    ("GOBP_INSULIN_SECRETION_INVOLVED_IN_CELLULAR_RESPONSE_TO_GLUCOSE_STIMULUS_ALL", "migraine"),
}

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def safe_read(p):
    try:
        return pd.read_csv(p)
    except Exception:
        return pd.DataFrame()

def fmt(v, spec=".3f", na="NA"):
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return na
        return format(float(v), spec)
    except Exception:
        return str(v) if v is not None else na

META_TISSUE_TAGS = {"meta_across_tissues", "meta"}

# ---------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------
genes  = safe_read(RESULTS / "all_genes_all_genesets.csv")
enrich = safe_read(RESULTS / "multi_geneset_enrichment_summary.csv")

load_report = {
    "all_genes_all_genesets.csv": len(genes),
    "multi_geneset_enrichment_summary.csv": len(enrich),
}

# detect the two disease Z columns (expects Z_bip, Z_migraine)
meta_z_cols = [c for c in genes.columns if c.startswith("Z_") and "__" not in c] if not genes.empty else []
DISEASES = []
if not enrich.empty and "Disease" in enrich.columns:
    DISEASES = sorted(enrich["Disease"].dropna().unique().tolist())
dis_col = {}
for d in DISEASES:
    if f"Z_{d}" in meta_z_cols:
        dis_col[d] = f"Z_{d}"
if len(dis_col) != 2 and len(meta_z_cols) >= 2:
    dis_col = {c.replace("Z_", ""): c for c in meta_z_cols[:2]}
    DISEASES = list(dis_col.keys())

SUM = []; BAR = "=" * 88
def add(*ls):
    for l in ls: SUM.append(l)
counts = {}

add(BAR, "  STAGE-3 GENE-LEVEL SUMMARY (results-only)  —  BIP vs migraine",
    f"  Source : {RESULTS.resolve()}",
    f"  Diseases detected : {', '.join(DISEASES) if DISEASES else 'NONE'}",
    BAR, "", "  INPUT FILES (rows loaded):")
for k, v in load_report.items():
    add(f"    {k:44s}: {v}")
add("")

if genes.empty or len(dis_col) != 2:
    add("  ✗ Cannot proceed: need all_genes_all_genesets.csv with two Z_<disease> columns.")
    (OUT / "stage3_summary.txt").write_text("\n".join(SUM))
    print("\n".join(SUM)); raise SystemExit

D1, D2 = list(dis_col.keys())[:2]      # e.g. bip, migraine
C1, C2 = dis_col[D1], dis_col[D2]

add(f"  Disease-1 column : {C1}",
    f"  Disease-2 column : {C2}",
    "",
    "  THRESHOLDS:",
    f"    |Z| clear     : {Z_THRESH}    strong : {STRONG_Z}",
    f"    repeater      : gene in >= {REPEATER_NSETS} sets",
    f"    Layer A       : opposite-sign in >= {OPP_MIN_ROWS} set-rows",
    f"    gated top-N   : {GATED_TOP_N}",
    "")

# =====================================================================
# CLASSIFIER
# =====================================================================
def classify(zb, zm, z=Z_THRESH):
    if not (np.isfinite(zb) and np.isfinite(zm)): return "missing"
    if abs(zb) < z and abs(zm) < z: return "both_weak"
    if zb >  z and zm >  z: return "concordant_pos"
    if zb < -z and zm < -z: return "concordant_neg"
    if zb >  z and zm < -z: return f"opp_{D1}pos_{D2}neg"
    if zb < -z and zm >  z: return f"opp_{D1}neg_{D2}pos"
    if abs(zb) >= z: return f"{D1}_private"
    if abs(zm) >= z: return f"{D2}_private"
    return "mixed_subthreshold"

# =====================================================================
# 1. COLLAPSE TO UNIQUE GENES (median Z across sets)
# =====================================================================
g = genes.dropna(subset=[C1, C2], how="all").copy()

agg = (g.groupby("Gene")
       .agg(n_sets=("GeneSet", "nunique"),
            Z_d1_median=(C1, "median"),
            Z_d2_median=(C2, "median"),
            sets=("GeneSet", lambda s: "; ".join(sorted(set(s))[:10])))
       .reset_index())
agg["class"] = [classify(b, m) for b, m in zip(agg.Z_d1_median, agg.Z_d2_median)]
agg["median_delta"] = agg.Z_d1_median - agg.Z_d2_median
agg["is_repeater"] = agg.n_sets >= REPEATER_NSETS

# count opposite-sign set-ROWS per gene (both |Z|>=1.96, signs differ)
pair = g.dropna(subset=[C1, C2])
opp_rows = pair[(np.sign(pair[C1]) != np.sign(pair[C2])) &
                (np.minimum(pair[C1].abs(), pair[C2].abs()) >= Z_THRESH)]
n_opp = opp_rows.groupby("Gene").size().rename("n_opposite_rows")
agg = agg.merge(n_opp, on="Gene", how="left")
agg["n_opposite_rows"] = agg["n_opposite_rows"].fillna(0).astype(int)

agg = agg.rename(columns={"Z_d1_median": f"Z_{D1}_median",
                          "Z_d2_median": f"Z_{D2}_median"})
agg.sort_values("median_delta", key=lambda s: s.abs(),
                ascending=False, inplace=True)
agg.to_csv(OUT / "01_unique_genes.csv", index=False)
counts["unique_genes"] = len(agg)

add(BAR, "  [1] UNIQUE-GENE COLLAPSE", BAR)
add(f"  {len(agg)} unique genes (from {len(g)} gene x set rows).")
cls_counts = agg["class"].value_counts()
for k, v in cls_counts.items():
    add(f"    {k:28s}: {v}")
add(f"  Repeaters (>= {REPEATER_NSETS} sets): {int(agg.is_repeater.sum())}", "")

# =====================================================================
# 2. LAYER A — recurrent opposite-sign genes (priority-scored)
# =====================================================================
zb = agg[f"Z_{D1}_median"]; zm = agg[f"Z_{D2}_median"]
agg["priority"] = (agg["median_delta"].abs()
                   * np.minimum(zb.abs(), zm.abs())
                   * (1 + agg["n_opposite_rows"]))
layer_A = (agg[agg["class"].str.startswith("opp_") &
               (agg["n_opposite_rows"] >= OPP_MIN_ROWS)]
           .sort_values("priority", ascending=False))
layer_A.to_csv(OUT / "02_opposite_core.csv", index=False)
counts["layer_A_opposite"] = len(layer_A)

add(BAR, "  [2] LAYER A — RECURRENT OPPOSITE-SIGN GENES (the only gene-level contrast)", BAR)
if not layer_A.empty:
    add(f"  {len(layer_A)} genes: opposite sign, both |Z|>={Z_THRESH}, in >= {OPP_MIN_ROWS} set-rows.", "")
    for _, r in layer_A.head(40).iterrows():
        add(f"    • {r['Gene']:12s} {D1}Z={fmt(r[f'Z_{D1}_median'],'+.2f')} "
            f"{D2}Z={fmt(r[f'Z_{D2}_median'],'+.2f')} Δ={fmt(r['median_delta'],'+.2f')} "
            f"n_sets={r['n_sets']} opp_rows={r['n_opposite_rows']} "
            f"[{r['class']}] prio={fmt(r['priority'],'.1f')}")
else:
    add("  (no recurrent opposite-sign genes at these thresholds)")
add("")

# =====================================================================
# 3. LAYER B — concordant (shared liability, one paragraph)
# =====================================================================
layer_B = (agg[agg["class"].isin(["concordant_pos", "concordant_neg"])]
           .sort_values(f"Z_{D1}_median", key=lambda s: s.abs(), ascending=False))
layer_B.to_csv(OUT / "03_concordant_core.csv", index=False)
counts["layer_B_concordant"] = len(layer_B)

add(BAR, "  [3] LAYER B — CONCORDANT GENES (shared direction; NOT a contrast)", BAR)
if not layer_B.empty:
    add(f"  {len(layer_B)} genes same sign in both, both |Z|>={Z_THRESH}.", "")
    for _, r in layer_B.head(30).iterrows():
        add(f"    • {r['Gene']:12s} {D1}Z={fmt(r[f'Z_{D1}_median'],'+.2f')} "
            f"{D2}Z={fmt(r[f'Z_{D2}_median'],'+.2f')} n_sets={r['n_sets']} [{r['class']}]")
else:
    add("  (no concordant genes clearing both thresholds)")
add("")

# =====================================================================
# 4/5. PRIVATE DRIVERS (one disease clears 1.96)
# =====================================================================
for tag, cls_name, fname in [(D1, f"{D1}_private", "04"), (D2, f"{D2}_private", "05")]:
    priv = (agg[agg["class"] == cls_name]
            .sort_values(f"Z_{tag}_median", key=lambda s: s.abs(), ascending=False))
    priv.to_csv(OUT / f"{fname}_{tag}_private.csv", index=False)
    counts[f"{tag}_private"] = len(priv)

add(BAR, "  [4/5] DISEASE-PRIVATE GENES (only one disease clears |Z|>=1.96)", BAR)
for tag in (D1, D2):
    priv = agg[agg["class"] == f"{tag}_private"]
    add(f"  {tag}-private: {len(priv)}")
    for _, r in priv.sort_values(f"Z_{tag}_median", key=lambda s: s.abs(),
                                 ascending=False).head(15).iterrows():
        add(f"    • {r['Gene']:12s} {D1}Z={fmt(r[f'Z_{D1}_median'],'+.2f')} "
            f"{D2}Z={fmt(r[f'Z_{D2}_median'],'+.2f')} n_sets={r['n_sets']}")
    add("")

# =====================================================================
# 6. LAYER C — gated-set leaders that are NOT repeaters
# =====================================================================
gated_rows = []
avail_sets = set(g["GeneSet"].unique())
for (gs, dis) in GATED:
    if gs not in avail_sets:
        continue
    col = dis_col.get(dis)
    if col is None:
        continue
    sub = (g[g["GeneSet"] == gs][["Gene", C1, C2]].dropna(subset=[col])
           .drop_duplicates("Gene").copy())
    if sub.empty:
        continue
    sub["gated_set"] = gs; sub["gated_disease"] = dis
    sub["absZ"] = sub[col].abs()
    sub = sub.sort_values("absZ", ascending=False).head(GATED_TOP_N)
    gated_rows.append(sub)
gated_leaders = pd.concat(gated_rows, ignore_index=True) if gated_rows else pd.DataFrame()
if not gated_leaders.empty:
    gated_leaders = gated_leaders.merge(
        agg[["Gene", "n_sets", "is_repeater", "class"]], on="Gene", how="left")
    gated_leaders.to_csv(OUT / "06_gated_set_leading_genes.csv", index=False)
counts["gated_leaders"] = len(gated_leaders)

add(BAR, "  [6] LAYER C — GATED-SET LEADERS (private, non-repeater)", BAR)
if not gated_leaders.empty:
    for (gs, dis), sub in gated_leaders.groupby(["gated_set", "gated_disease"], sort=False):
        col = dis_col[dis]
        non_rep = sub[~sub["is_repeater"].fillna(False)]
        add(f"  [{gs}]  driving disease={dis}  "
            f"({len(non_rep)}/{len(sub)} non-repeater in top {GATED_TOP_N}):")
        for _, r in non_rep.head(12).iterrows():
            zval = r[C1] if dis == D1 else r[C2]
            add(f"    • {r['Gene']:12s} Z={fmt(zval,'+.2f')} n_sets={int(r['n_sets']) if pd.notna(r['n_sets']) else '?'} [{r.get('class','?')}]")
        rep = sub[sub["is_repeater"].fillna(False)]["Gene"].tolist()
        if rep:
            add(f"    (repeaters ignored here: {', '.join(rep[:10])})")
        add("")
else:
    add("  (no gated sets found in the data — check GATED set names)")
add("")

# =====================================================================
# 7. MODULE TAGGING + opposite-gene x module incidence
# =====================================================================
MODULES = {
    "synaptic_plasticity": ["KEGG_LTP_ALL", "GOBP_REGULATION_OF_SYNAPTIC_PLASTICITY_ALL",
                            "GOBP_GLUTAMATERGIC_ALL", "REACTOME_NEUROTRANSMITTER_RELEASE_CYCLE",
                            "KEGG_DOPAMINERGIC_SYNAPSE", "MONOAMINES_ALL"],
    "mitochondria_oxphos": ["MOOTHA_MITOCHONDRIA_ALL", "MOOTHA_PGC_ALL",
                            "HALLMARK_OXIDATIVE_PHOSPHORYLATION_ALL", "REACTOME_MITOPHAGY"],
    "apoptosis": ["GOCC_BCL_2_FAMILY_PROTEIN_COMPLEX_ALL",
                  "GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_ALL",
                  "REACTOME_INTRINSIC_PATHWAY_FOR_APOPTOSIS_ALL",
                  "HALLMARK_APOPTOSIS_ALL", "KEGG_APOPTOSIS_ALL"],
    "nad_sirtuin": ["WP_NAD_METABOLISM_ALL", "KEGG_NICOTINATE_AND_NICOTINAMIDE_METABOLISM_ALL",
                    "REACTOME_NICOTINATE_METABOLISM_ALL", "GOMF_NADPLUS_BINDING_ALL",
                    "GOBP_NAD_TRANSPORT_ALL", "WP_NAD_BIOSYNTHETIC_PATHWAYS_ALL",
                    "WP_NAD_METABOLISM_SIRTUINS_AND_AGING_ALL",
                    "REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION_ALL"],
    "senescence_telomere": ["GOBP_REGULATION_OF_CELLULAR_SENESCENCE_ALL",
                            "GOBP_CELLULAR_SENESCENCE_ALL", "REACTOME_CELLULAR_SENESCENCE_ALL",
                            "REACTOME_SENESCENCE_ALL", "REACTOME_TELOMERE_ALL",
                            "SAUL_SEN_MAYO_ALL"],
    "immune_glia": ["HALLMARK_INFLAMMATORY_RESPONSE", "REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM",
                    "HALLMARK_TNFA_SIGNALING_VIA_NFKB", "GOBP_POSITIVE_REGULATION_OF_MICROGLIAL_ACTIVATION",
                    "REACTOME_ACTIVATION_OF_INNATE_IMMUNE_RESPONSE", "REACTOME_COMPLEMENT_ALL",
                    "GOBP_SYNAPSE_PRUNING_ALL", "HLA_COMPLEX_ALL", "ADULT_MICRO_ALL"],
    "energy_sensing": ["REACTOME_ENERGY_DEPENDENT_REGULATION_OF_MTOR_BY_LKB1_AMPK_ALL",
                       "GOBP_CAMKK_AMPK_SIGNALING_CASCADE_ALL", "HALLMARK_PI3K_AKT_MTOR_SIGNALING_ALL"],
    "insulin_glp1": ["GOBP_INSULIN_SECRETION_INVOLVED_IN_CELLULAR_RESPONSE_TO_GLUCOSE_STIMULUS_ALL",
                     "REACTOME_REGULATION_OF_INSULIN_SECRETION_ALL",
                     "WP_GLP1_FROM_INTESTINE_AND_PANCREAS_AND_ROLE_IN_GLUCOSE_HOMEOSTASIS_ALL",
                     "REACTOME_SYNTHESIS_SECRETION_AND_INACTIVATION_OF_GLUCAGON_LIKE_PEPTIDE_1_GLP_1_ALL",
                     "REACTOME_GLUCAGON_LIKE_PEPTIDE_1_GLP1_REGULATES_INSULIN_SECRETION_ALL",
                     "KEGG_MEDICUS_REFERENCE_EP_NE_ADRB_CAMP_SIGNALING_PATHWAY_ALL",
                     "WANG_ADIPOGENIC_GENES_REPRESSED_BY_SIRT1_ALL"],
    "circadian": ["REACTOME_CIRCADIAN_CLOCK"],
    "glia_markers": ["ADULT_ASTRO_ALL", "ADULT_OLIGO_ALL", "ADULT_MICRO_ALL"],
}
set_to_module = {}
for mod, sets in MODULES.items():
    for s in sets:
        set_to_module.setdefault(s, []).append(mod)

# module membership of opposite-core genes (Layer A)
if not layer_A.empty:
    rows = []
    for _, r in layer_A.iterrows():
        gene = r["Gene"]
        gene_sets = set(g[g["Gene"] == gene]["GeneSet"].unique())
        mods = sorted({m for s in gene_sets for m in set_to_module.get(s, [])})
        row = {"Gene": gene, f"Z_{D1}": r[f"Z_{D1}_median"],
               f"Z_{D2}": r[f"Z_{D2}_median"], "class": r["class"]}
        for mod in MODULES:
            row[mod] = int(mod in mods)
        rows.append(row)
    opp_by_module = pd.DataFrame(rows)
    opp_by_module.to_csv(OUT / "07_opposite_gene_by_module.csv", index=False)
    mod_tally = {m: int(opp_by_module[m].sum()) for m in MODULES}
else:
    opp_by_module = pd.DataFrame(); mod_tally = {}

add(BAR, "  [7] OPPOSITE-CORE GENES BY BIOLOGICAL MODULE", BAR)
if mod_tally:
    for m, n in sorted(mod_tally.items(), key=lambda kv: -kv[1]):
        if n == 0: continue
        genes_in = opp_by_module.loc[opp_by_module[m] == 1, "Gene"].tolist()
        add(f"    {m:22s}: {n}  ({', '.join(genes_in[:12])})")
else:
    add("  (no opposite-core genes to tag)")
add("")

# =====================================================================
# 8. PARADOX PANELS (symbol presence + shared/opposite geometry)
# =====================================================================
u = agg.set_index("Gene")
def gene_class(name):
    return u.loc[name, "class"] if name in u.index else "absent"
def gene_z(name):
    if name in u.index:
        return u.loc[name, f"Z_{D1}_median"], u.loc[name, f"Z_{D2}_median"]
    return np.nan, np.nan

PANELS = {
    "lithium_biology":       ["GSK3B","IMPA1","IMPA2","INPP1","CACNA1C","ANK3",
                              "CLOCK","ARNTL","PER2","CRY1","GRIN2A","AKT1","MTOR","BCL2"],
    "valproate_hdac_gaba":   ["HDAC1","HDAC2","HDAC3","GAD1","GAD2","GABRA1","GABRB2",
                              "ABAT","SLC6A1","GRIN2B"],
    "cgrp_axis":             ["CALCA","CALCB","CALCRL","RAMP1","RAMP2","RAMP3","ADM","ADCYAP1"],
    "circadian_sleep":       ["CLOCK","ARNTL","PER1","PER2","PER3","CRY1","CRY2",
                              "NR1D1","NR1D2","RORA","RORB","CSNK1D","CSNK1E","DBP"],
    "migraine_channel_vascular": ["CACNA1A","ATP1A2","SCN1A","KCNK18","TRPM8","TRPA1",
                                  "TRPV1","NOTCH3","EDNRA","NOS1","NOS3"],
    "snare_release":         ["STX1A","STX1B","STXBP1","SNAP25","VAMP2","SYT1","SYT2",
                              "CPLX1","CPLX2","UNC13A","RAB3A"],
    "apoptotic_rheostat":    ["BAD","BAK1","BAX","BCL2","BCL2L1","BCL2L11","BNIP3L",
                              "MCL1","PMAIP1","GPX4","CYCS"],
    "oxphos_opposite_candidates": ["NDUFAB1","NDUFA5","NDUFA6","NDUFS3","NDUFV2","SDHA",
                                   "SDHB","UQCRC1","COX5B","ATP5F1A","FDX1","ACADS"],
    "dopamine_monoamine":    ["DRD1","DRD2","DRD3","COMT","MAOA","MAOB","TH","DDC",
                              "SLC6A3","SLC6A4","PPP1R1B"],
    "tnf_receptor_immune":   ["TNF","TNFRSF1A","TNFRSF1B","IL6","IL1B","NFKB1","RELA","C3"],
}

panel_rows = []; panel_gene_rows = []
for pname, plist in PANELS.items():
    present = [x for x in plist if x in u.index]
    n_in = len(present); n_tot = len(plist)
    if n_in == 0:
        panel_rows.append(dict(panel=pname, n_in_twas=0, n_total=n_tot,
                               frac_opposite=np.nan, frac_concordant=np.nan,
                               paradox_score=np.nan, verdict="not_represented"))
        continue
    classes = [gene_class(x) for x in present]
    frac_opp  = np.mean([c.startswith("opp_") for c in classes])
    frac_conc = np.mean([c.startswith("concordant_") for c in classes])
    frac_weak = np.mean([c in ("both_weak", "mixed_subthreshold") for c in classes])
    score = frac_opp - frac_conc
    if frac_weak >= 0.6:
        verdict = "mostly_weak_uninformative"
    elif score > 0.15:
        verdict = "CONTRAST_panel"
    elif score < -0.15:
        verdict = "SHARED_panel"
    else:
        verdict = "mixed"
    panel_rows.append(dict(panel=pname, n_in_twas=n_in, n_total=n_tot,
                           frac_opposite=round(frac_opp,3),
                           frac_concordant=round(frac_conc,3),
                           frac_weak=round(frac_weak,3),
                           paradox_score=round(score,3), verdict=verdict))
    for x in present:
        zb_, zm_ = gene_z(x)
        panel_gene_rows.append(dict(panel=pname, Gene=x,
                                    **{f"Z_{D1}": zb_, f"Z_{D2}": zm_},
                                    cls=gene_class(x)))
panel_df = pd.DataFrame(panel_rows).sort_values("paradox_score", ascending=False)
panel_df.to_csv(OUT / "08_paradox_panels.csv", index=False)
if panel_gene_rows:
    pd.DataFrame(panel_gene_rows).to_csv(OUT / "08_paradox_panel_genes.csv", index=False)
counts["panels_contrast"]  = int((panel_df["verdict"] == "CONTRAST_panel").sum())
counts["panels_shared"]    = int((panel_df["verdict"] == "SHARED_panel").sum())
counts["panels_absent"]    = int((panel_df["verdict"] == "not_represented").sum())

add(BAR, "  [8] PARADOX PANELS (which paradox still has symbols; shared vs opposite)", BAR)
add("  paradox_score = frac_opposite - frac_concordant  (+ = contrast, - = shared)", "")
for _, r in panel_df.iterrows():
    if r["verdict"] == "not_represented":
        add(f"    • {r['panel']:28s} : NOT REPRESENTED in TWAS matrix (0/{int(r['n_total'])})")
        continue
    add(f"    • {r['panel']:28s} : {r['verdict']:26s} "
        f"score={fmt(r['paradox_score'],'+.2f')} "
        f"(opp={fmt(r['frac_opposite'],'.2f')}, conc={fmt(r['frac_concordant'],'.2f')}, "
        f"weak={fmt(r['frac_weak'],'.2f')}) [{int(r['n_in_twas'])}/{int(r['n_total'])} genes]")
add("")

# per-panel gene detail for the informative panels
add("  PANEL GENE DETAIL (informative panels only):")
if panel_gene_rows:
    pg = pd.DataFrame(panel_gene_rows)
    informative = panel_df[panel_df["verdict"].isin(
        ["CONTRAST_panel", "SHARED_panel", "mixed"])]["panel"].tolist()
    for pname in informative:
        sub = pg[pg["panel"] == pname]
        add(f"    [{pname}]")
        for _, r in sub.iterrows():
            add(f"      {r['Gene']:12s} {D1}Z={fmt(r[f'Z_{D1}'],'+.2f')} "
                f"{D2}Z={fmt(r[f'Z_{D2}'],'+.2f')} [{r['cls']}]")
        add("")
add("")

# =====================================================================
# 9. CIRCADIAN-ONLY CUT + LITHIUM ∩ LTP CUT
# =====================================================================
add(BAR, "  [9] TARGETED CUTS", BAR)

# circadian set only
clk = g[g["GeneSet"] == "REACTOME_CIRCADIAN_CLOCK"][["Gene", C1, C2]].drop_duplicates("Gene")
if not clk.empty:
    clk["class"] = [classify(b, m) for b, m in zip(clk[C1], clk[C2])]
    clk = clk.sort_values(C1, key=lambda s: s.abs(), ascending=False)
    clk.to_csv(OUT / "09_circadian_only.csv", index=False)
    n_clear = int(((clk[C1].abs() >= Z_THRESH) | (clk[C2].abs() >= Z_THRESH)).sum())
    add(f"  Circadian set (REACTOME_CIRCADIAN_CLOCK): {len(clk)} genes, "
        f"{n_clear} clear |Z|>={Z_THRESH} in >=1 disease.")
    for _, r in clk.head(10).iterrows():
        add(f"    • {r['Gene']:10s} {D1}Z={fmt(r[C1],'+.2f')} "
            f"{D2}Z={fmt(r[C2],'+.2f')} [{r['class']}]")
    if n_clear <= 1:
        add("    ⚠ Circadian signal carried by <=1 gene → sleep-phenotype story NOT supported here.")
else:
    add("  (REACTOME_CIRCADIAN_CLOCK not in data)")
add("")

# lithium panel ∩ LTP membership
ltp_genes = set(g.loc[g["GeneSet"] == "KEGG_LTP_ALL", "Gene"]) if "KEGG_LTP_ALL" in avail_sets else set()
lith = set(PANELS["lithium_biology"])
inter = sorted(lith & ltp_genes)
add("  Lithium panel ∩ LTP membership (is 'lithium biology' just the LTP hit?):")
if inter:
    add(f"    shared genes: {', '.join(inter)}")
    lith_outside = sorted([x for x in lith if x in u.index and x not in ltp_genes])
    add(f"    lithium symbols OUTSIDE LTP (independent contrast candidates): "
        f"{', '.join(lith_outside) if lith_outside else '(none in TWAS)'}")
    add("    → If opposite-sign lithium genes are OUTSIDE LTP while LTP genes are "
        f"{D1}-private positives, that is a TWO-LAYER model (plasticity-up + lithium/clock contrast).")
else:
    add("    (no overlap — lithium panel and LTP set are disjoint in these data)")
add("")

# =====================================================================
# 10. PUBLICATION GENE TABLE (Layer A + non-repeater gated leaders)
# =====================================================================
pub_rows = []
for _, r in layer_A.iterrows():
    pub_rows.append(dict(Gene=r["Gene"], Layer="A_opposite",
                         **{f"Z_{D1}": r[f"Z_{D1}_median"], f"Z_{D2}": r[f"Z_{D2}_median"]},
                         delta=r["median_delta"], n_sets=r["n_sets"],
                         opp_rows=r["n_opposite_rows"], klass=r["class"]))
if not gated_leaders.empty:
    for (gs, dis), sub in gated_leaders.groupby(["gated_set", "gated_disease"], sort=False):
        col = dis_col[dis]
        non_rep = sub[~sub["is_repeater"].fillna(False)].head(8)
        for _, r in non_rep.iterrows():
            pub_rows.append(dict(Gene=r["Gene"], Layer=f"C_{gs}[{dis}]",
                                 **{f"Z_{D1}": r[C1], f"Z_{D2}": r[C2]},
                                 delta=r[C1]-r[C2] if pd.notna(r[C1]) and pd.notna(r[C2]) else np.nan,
                                 n_sets=r.get("n_sets", np.nan),
                                 opp_rows=np.nan, klass=r.get("class", "?")))
pub = pd.DataFrame(pub_rows).drop_duplicates(subset=["Gene", "Layer"])
pub.to_csv(OUT / "10_publication_gene_table.csv", index=False)
counts["publication_rows"] = len(pub)

add(BAR, "  [10] PUBLICATION GENE TABLE (Layer A + non-repeater gated leaders)", BAR)
add(f"  {len(pub)} rows written to 10_publication_gene_table.csv", "")
for _, r in pub.head(45).iterrows():
    add(f"    • {r['Gene']:12s} [{r['Layer']:34s}] "
        f"{D1}Z={fmt(r[f'Z_{D1}'],'+.2f')} {D2}Z={fmt(r[f'Z_{D2}'],'+.2f')} "
        f"[{r['klass']}]")
add("")

# =====================================================================
# NARRATIVE PARAGRAPH (auto-generated, results-only wording)
# =====================================================================
add(BAR, "  ★ AUTO-DRAFT RESULTS PARAGRAPH (verify before use)", BAR)
topA = layer_A.head(8)["Gene"].tolist() if not layer_A.empty else []
topB_neg = layer_B[layer_B["class"] == "concordant_neg"].head(4)["Gene"].tolist() if not layer_B.empty else []
para = (
    f"  No gene set survived global correction (Stage-2). At unique-gene resolution "
    f"({counts.get('unique_genes',0)} genes), {D1} and {D2} predicted-expression Z-scores "
    f"repeatedly disagree at the same symbols"
    + (f": {', '.join(topA)}" if topA else "")
    + f". These {counts.get('layer_A_opposite',0)} recurrent opposite-sign genes are the only "
    f"gene-level contrast the data support. "
    + (f"Shared-direction genes ({', '.join(topB_neg)} among {counts.get('layer_B_concordant',0)} "
       f"concordant genes) reflect common liability, not a {D1}-{D2} difference. "
       if topB_neg else "")
    + f"Within the robustness-gated sets, leading genes that are not multi-set repeaters carry "
    f"the residual {D1}/{D2} shifts. Colocalization and eQTL R2 (Stage-4) require the original "
    f"S-PrediXcan files."
)
add(para, "")

# =====================================================================
# TALLY + GUARDRAILS
# =====================================================================
add(BAR, "  TALLY", BAR)
for k in sorted(counts):
    add(f"    {k:26s}: {counts[k]}")
add(BAR)
add("  GUARDRAILS:",
    "   - Layer A (opposite, >=2 set-rows) is the ONLY gene-level BIP-vs-migraine contrast claim.",
    "   - Layer B is shared liability; it does NOT rescue any failed gene-set test.",
    "   - Absent panel symbols (e.g. CGRP) mean the RUN is silent, not that biology is absent.",
    "   - Repeater genes (>=3 sets) are one signal counted many times, not many discoveries.",
    "   - No causal / colocalization / MR claim is possible from these CSVs.",
    BAR)

# ---------------------------------------------------------------------
# WRITE
# ---------------------------------------------------------------------
summary_path = OUT / "stage3_summary.txt"
summary_path.write_text("\n".join(SUM))
print("\n".join(SUM[:80]))
print("\n...\n")
print(f"✓ Detailed summary → {summary_path.resolve()}")
print(f"✓ CSVs             → {OUT.resolve()}/")
print(f"  Files: {sorted(p.name for p in OUT.glob('*.csv'))}")

  STAGE-3 GENE-LEVEL SUMMARY (results-only)  —  BIP vs migraine
  Source : /content/089Av2_results
  Diseases detected : bip, migraine

  INPUT FILES (rows loaded):
    all_genes_all_genesets.csv                  : 4107
    multi_geneset_enrichment_summary.csv        : 714

  Disease-1 column : Z_bip
  Disease-2 column : Z_migraine

  THRESHOLDS:
    |Z| clear     : 1.96    strong : 3.0
    repeater      : gene in >= 3 sets
    Layer A       : opposite-sign in >= 2 set-rows
    gated top-N   : 25

  [1] UNIQUE-GENE COLLAPSE
  1966 unique genes (from 3059 gene x set rows).
    both_weak                   : 909
    bip_private                 : 436
    migraine_private            : 296
    missing                     : 87
    concordant_neg              : 70
    concordant_pos              : 62
    opp_bipneg_migrainepos      : 58
    opp_bippos_migraineneg      : 48
  Repeaters (>= 3 sets): 260

  [2] LAYER A — RECURRENT OPPOSITE-SIGN GENES (the only gene-level contrast)
  30 genes: opp

# Stage 4 Coloc

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_GWAS/bip2024_eur_noUKB_no23andMe.gz" "/content/"

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_GWAS/CNS/migraine/GCST90671940.tsv.gz" "/content"

In [ ]:
# =====================================================================
#  STAGE-4  CROSS-DISORDER (BIP vs migraine) COLOCALIZATION @ 8 LOCI
#  Stage-4a: GWAS<->GWAS coloc.abf  (runs now, GWAS files only)
#  Stage-4b: GWAS<->eQTL coloc.abf  (auto-runs iff eQTL slices supplied)
#
#  Amendments applied (per bug report):
#   A. ONE build for the whole run (file-level vote); lift hg19->hg38 or
#      hard BUILD_MISMATCH.  Never subtract pos across builds.
#   B. Two windows: gene ±250kb (4b scale) and lead ±250kb (only if signal).
#   C. Gate 4a on signal (no_regional_signal skips coloc entirely).
#   D. shared_snp printed ONLY if PP.H4 >= 0.5.
#   E. Harmonize on chr:pos AFTER lift; rsid only as fallback.
#   F. Restrict 4a to the 8 jackknife-core genes (+BNIP3L +FDX1).
#   G. 4b needs rsid,chr,pos,ea,oa,beta,se ; merge+flip ; sd2=0.15.
#   H. Label BIP/migraine GWAS provenance in every row.
#   I. Lead-conditioned re-run at BAD (lead ±250kb).
#
#  Inputs (preloaded):
#     BIP      : /content/bip2024_eur_noUKB_no23andMe.gz
#     migraine : /content/GCST90671940.tsv.gz
#  Writes: /content/089Av2_results/downstream_stage4/*.csv + stage4_summary.txt
# =====================================================================
import os, gzip, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# optional liftover ---------------------------------------------------
try:
    from pyliftover import LiftOver          # pip install pyliftover
    _HAVE_LIFT = True
except Exception:
    _HAVE_LIFT = False

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
BIP_PATH = Path("/content/bip2024_eur_noUKB_no23andMe.gz")
MIG_PATH = Path("/content/GCST90671940.tsv.gz")
D1, D2   = "bip", "migraine"

BIP_GWAS_LABEL = "PGC_bip2024_eur_noUKB_no23andMe"
MIG_GWAS_LABEL = "UKB_GCST90671940_proxy"     # NOT IHGC/Hautakangas

OUTROOT  = Path("/content/089Av2_results")
OUT      = OUTROOT / "downstream_stage4"
OUT.mkdir(parents=True, exist_ok=True)
EQTL_DIR = OUT / "eqtl_slices"
EQTL_DIR.mkdir(parents=True, exist_ok=True)

LOAD_WIN       = 1_000_000    # wide load window (covers both builds + lead shrink)
WIN_GENE       = 250_000      # gene-body window (cis scale; used for 4b)
WIN_LEAD       = 250_000      # lead-SNP window (only if a trait has signal)
LEAD_APART_MAX = 500_000      # if leads differ by more than this -> not same locus
GWS            = 5e-8
SUGG           = 1e-5
MIN_MATCH      = 50
H_CALL         = 0.80
SHARED_SNP_H4  = 0.50         # only report shared SNP if H4 >= this
SD_PRIOR_CC    = 0.2          # coloc sd.prior for case-control GWAS
SD_PRIOR_EQTL  = 0.15         # coloc sd.prior for quantitative eQTL
P1 = 1e-4; P2 = 1e-4; P12 = 1e-5
CHUNK          = 1_000_000

# Eight jackknife-core genes.  Centres = GENCODE TSS-ish midpoints.
# VERIFY against GENCODE v26 / GTEx before publishing.
GENES = {
    "TNFRSF1A": {"chr":"12","hg38":6_442_676,  "hg19":6_443_676,
                 "note":"jackknife core; immune/TNF contrast"},
    "BAD":      {"chr":"11","hg38":64_269_830, "hg19":64_037_302,
                 "note":"jackknife core; BIP GWS near rs484201; rheostat"},
    "BCL2L1":   {"chr":"20","hg38":31_664_452, "hg19":30_252_256,
                 "note":"jackknife core; opposite to BAD"},
    "STX1A":    {"chr":"7", "hg38":73_708_398, "hg19":73_272_791,
                 "note":"jackknife core; SNARE"},
    "NDUFAB1":  {"chr":"16","hg38":23_598_000, "hg19":23_609_000,
                 "note":"jackknife core; OXPHOS"},
    "GPX4":     {"chr":"19","hg38":1_105_000,  "hg19":1_105_000,
                 "note":"jackknife core; ferroptosis/stress"},
    "BNIP3L":   {"chr":"8", "hg38":26_384_000, "hg19":26_241_000,
                 "note":"jackknife core; rheostat support"},
    "FDX1":     {"chr":"11","hg38":110_834_000,"hg19":110_705_000,
                 "note":"jackknife core; mito Fe-S"},
}
BAD_LEAD_RSID = "rs484201"    # for the lead-conditioned BAD re-run (I)

# ---------------------------------------------------------------------
# COLUMN CANDIDATES (lowercased)
# ---------------------------------------------------------------------
CAND = {
    "chr":  ["chromosome","chr","chrom","#chrom","hm_chrom","chr_id","chrom_b37","#chr"],
    "pos":  ["base_pair_location","pos","bp","position","hm_pos","chr_position",
             "bp_b37","bp_hg38","position_b38","base_pair"],
    "ea":   ["effect_allele","a1","allele1","alt","hm_effect_allele","ea","tested_allele"],
    "oa":   ["other_allele","a2","allele2","ref","hm_other_allele","nea","noneffect_allele"],
    "beta": ["beta","effect","hm_beta","log_odds","logor","effect_size","b"],
    "or":   ["or","odds_ratio","oddsratio","hm_odds_ratio"],
    "se":   ["standard_error","se","stderr","sebeta","standarderror","hm_se",
             "log_odds_se","se_beta"],
    "p":    ["p_value","pval","p","p.value","pvalue","hm_p","p_bolt_lmm","frequentist_add_pvalue"],
    "eaf":  ["effect_allele_frequency","eaf","freq","frq","maf","a1freq",
             "hm_effect_allele_frequency","af","freq1"],
    "rsid": ["rs_id","rsid","variant_id","snp","id","markername","hm_rsid","rs","snpid"],
}
PALINDROME = {("A","T"),("T","A"),("C","G"),("G","C")}

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def _sniff_sep(path):
    with gzip.open(path, "rt") as fh:
        line = fh.readline()
        while line and not line.strip():
            line = fh.readline()
    if "\t" in line: return "\t"
    if line.count(",") >= 2: return ","
    return r"\s+"

def _map_columns(cols):
    low = {c.lower(): c for c in cols}
    out = {}
    for key, cands in CAND.items():
        for c in cands:
            if c in low:
                out[key] = low[c]; break
    return out

def _norm_chr(s):
    s = str(s).strip().upper()
    if s.startswith("CHR"): s = s[3:]
    if s in ("23","XY"): s = "X"
    if s == "24": s = "Y"
    return s

def load_gwas(path, label):
    """Chunk-read; keep rows inside any 8-gene window (union of both builds)."""
    info = {"path": str(path), "loaded": False, "n_kept": 0, "colmap": {}, "sep": None}
    if not path.exists():
        info["error"] = "file not found"; return pd.DataFrame(), info
    sep = _sniff_sep(path); info["sep"] = sep
    try:
        head = pd.read_csv(path, sep=sep, engine="python", nrows=5)
    except Exception as e:
        info["error"] = f"header read failed: {e}"; return pd.DataFrame(), info
    cmap = _map_columns(head.columns); info["colmap"] = cmap
    need = ["chr","pos","ea","oa","se","p"]
    if not all(k in cmap for k in need) or not ("beta" in cmap or "or" in cmap):
        info["error"] = f"missing required columns; detected={cmap}"
        return pd.DataFrame(), info

    win = {}
    for g in GENES.values():
        ch = g["chr"]
        for b in ("hg38","hg19"):
            win.setdefault(ch, []).append((g[b]-LOAD_WIN, g[b]+LOAD_WIN))

    usecols = list(dict.fromkeys([v for v in cmap.values() if v is not None]))
    engine = "python" if sep == r"\s+" else "c"
    kept = []
    try:
        reader = pd.read_csv(path, sep=sep, engine=engine, usecols=usecols,
                             chunksize=CHUNK, dtype=str)
    except Exception:
        reader = pd.read_csv(path, sep=sep, engine="python", usecols=usecols,
                             chunksize=CHUNK, dtype=str)
    for chunk in reader:
        chunk = chunk.rename(columns={v: k for k, v in cmap.items()})
        chunk["chr"] = chunk["chr"].map(_norm_chr)
        chunk = chunk[chunk["chr"].isin(win.keys())]
        if chunk.empty: continue
        chunk["pos"] = pd.to_numeric(chunk["pos"], errors="coerce")
        chunk = chunk.dropna(subset=["pos"]); chunk["pos"] = chunk["pos"].astype("int64")
        m = np.zeros(len(chunk), bool)
        posv = chunk["pos"].values; chrv = chunk["chr"].values
        for ch, ivs in win.items():
            cm = chrv == ch
            if not cm.any(): continue
            sub = posv[cm]; keepsub = np.zeros(sub.shape, bool)
            for lo, hi in ivs:
                keepsub |= (sub >= lo) & (sub <= hi)
            m[np.where(cm)[0]] = keepsub
        chunk = chunk[m]
        if not chunk.empty: kept.append(chunk)

    if not kept:
        info["loaded"] = True; info["error"] = "no SNPs in any window"
        return pd.DataFrame(), info
    df = pd.concat(kept, ignore_index=True)
    df["se"] = pd.to_numeric(df["se"], errors="coerce")
    df["p"]  = pd.to_numeric(df["p"], errors="coerce")
    if "beta" in df.columns:
        df["beta"] = pd.to_numeric(df["beta"], errors="coerce")
    else:
        df["beta"] = np.log(pd.to_numeric(df["or"], errors="coerce"))
    for c in ("ea","oa"):
        df[c] = df[c].astype(str).str.strip().str.upper()
    if "eaf" in df.columns:
        df["eaf"] = pd.to_numeric(df["eaf"], errors="coerce")
    if "rsid" in df.columns:
        df["rsid"] = df["rsid"].astype(str).str.strip()
    df = df.dropna(subset=["beta","se","p"])
    df = df[(df["se"] > 0) & (df["p"] > 0)]
    df["z"] = df["beta"] / df["se"]
    info["loaded"] = True; info["n_kept"] = len(df)
    return df, info

# ---- A. file-level build vote --------------------------------------
def file_build(df):
    votes = {"hg19": 0, "hg38": 0}
    if df.empty: return "hg38", votes
    for g in GENES.values():
        ch = df[df["chr"] == g["chr"]]
        if ch.empty: continue
        pv = ch["pos"].values
        for b in ("hg19","hg38"):
            votes[b] += int(((pv >= g[b]-WIN_GENE) & (pv <= g[b]+WIN_GENE)).sum())
    return (max(votes, key=votes.get) if max(votes.values()) > 0 else "hg38"), votes

def lift_hg19_to_hg38(df):
    """Return (lifted_df, ok). Requires pyliftover; else ok=False."""
    if df.empty or not _HAVE_LIFT:
        return df, False
    try:
        lo = LiftOver("hg19", "hg38")
    except Exception:
        return df, False
    new_pos = np.full(len(df), -1, dtype="int64")
    chrs = df["chr"].values; poss = df["pos"].values
    for i in range(len(df)):
        res = lo.convert_coordinate("chr"+str(chrs[i]), int(poss[i]))
        if res:
            new_pos[i] = int(res[0][1])
    out = df.copy(); out["pos"] = new_pos
    out = out[out["pos"] >= 0].reset_index(drop=True)
    return out, True

def slice_window(df, chrom, centre, win):
    if df.empty: return df.iloc[0:0]
    sub = df[df["chr"] == chrom]
    return sub[(sub["pos"] >= centre-win) & (sub["pos"] <= centre+win)]

def lead(loc):
    if loc.empty: return None
    r = loc.loc[loc["p"].idxmin()]
    return dict(rsid=str(r.get("rsid","NA")), pos=int(r["pos"]), p=float(r["p"]),
                beta=float(r["beta"]), ea=r["ea"], oa=r["oa"])

# ---- coloc.abf (Wakefield ABF + coloc-5 combination) ----------------
def _logsumexp(a):
    a = np.asarray(a, float); a = a[np.isfinite(a)]
    if a.size == 0: return -np.inf
    m = a.max(); return m + math.log(np.sum(np.exp(a - m)))

def _logdiff(x, y):
    if not np.isfinite(y): return x
    if x <= y: return -np.inf
    return x + math.log1p(-math.exp(y - x))

def _abf(beta, se, sd_prior):
    V = se**2; z = beta / se
    r = sd_prior**2 / (sd_prior**2 + V)
    return 0.5 * (np.log(1.0 - r) + r * z**2)

def coloc_abf(beta1, se1, beta2, se2,
              sd1=SD_PRIOR_CC, sd2=SD_PRIOR_CC, p1=P1, p2=P2, p12=P12):
    beta1=np.asarray(beta1,float); se1=np.asarray(se1,float)
    beta2=np.asarray(beta2,float); se2=np.asarray(se2,float)
    l1=_abf(beta1,se1,sd1); l2=_abf(beta2,se2,sd2)
    ls1=_logsumexp(l1); ls2=_logsumexp(l2); ls12=_logsumexp(l1+l2)
    lH0=0.0
    lH1=math.log(p1)+ls1
    lH2=math.log(p2)+ls2
    lH3=math.log(p1)+math.log(p2)+_logdiff(ls1+ls2, ls12)
    lH4=math.log(p12)+ls12
    arr=np.array([lH0,lH1,lH2,lH3,lH4]); denom=_logsumexp(arr)
    pp=np.exp(arr-denom); idx=int(np.argmax(l1+l2))
    return dict(zip(["H0","H1","H2","H3","H4"], pp)), idx

# ---- E. harmonize: chr:pos first, rsid fallback, conditional palindrome
def harmonize_merge(b, m):
    """Both frames assumed SAME build. Merge chr:pos; rsid only if that fails."""
    how = "chr:pos"
    bb = b.copy(); mm = m.copy()
    bb["key"] = bb["chr"]+":"+bb["pos"].astype(str)
    mm["key"] = mm["chr"]+":"+mm["pos"].astype(str)
    mrg = bb.drop_duplicates("key").merge(mm.drop_duplicates("key"),
                                          on="key", suffixes=("_b","_m"))
    if len(mrg) < MIN_MATCH and "rsid" in b.columns and "rsid" in m.columns:
        rb = b["rsid"].astype(str).str.startswith("rs").mean()
        rm = m["rsid"].astype(str).str.startswith("rs").mean()
        if rb > 0.5 and rm > 0.5:
            how = "rsid(fallback)"
            mrg = (b.drop_duplicates("rsid")
                     .merge(m.drop_duplicates("rsid"), on="rsid", suffixes=("_b","_m")))
    if mrg.empty:
        return mrg, how

    # conditional palindrome drop: only when EAF missing or ambiguous (0.4-0.6)
    pal_mask = np.array([(ea,oa) in PALINDROME
                         for ea,oa in zip(mrg["ea_b"], mrg["oa_b"])])
    if pal_mask.any():
        if "eaf_b" in mrg.columns:
            eaf = pd.to_numeric(mrg["eaf_b"], errors="coerce").values
            ambiguous = ~np.isfinite(eaf) | ((eaf > 0.4) & (eaf < 0.6))
        else:
            ambiguous = np.ones(len(mrg), bool)
        drop = pal_mask & ambiguous
        mrg = mrg[~drop]
    if mrg.empty:
        return mrg, how

    same = (mrg["ea_b"]==mrg["ea_m"]) & (mrg["oa_b"]==mrg["oa_m"])
    flip = (mrg["ea_b"]==mrg["oa_m"]) & (mrg["oa_b"]==mrg["ea_m"])
    mrg = mrg[same | flip].copy()
    if mrg.empty:
        return mrg, how
    mrg["beta_m_h"] = np.where(mrg["ea_b"]==mrg["ea_m"], mrg["beta_m"], -mrg["beta_m"])
    return mrg, how

def classify(pp, lb, lm):
    any_sugg = (lb and lb["p"] < SUGG) or (lm and lm["p"] < SUGG)
    if not any_sugg:
        return "no_regional_signal"
    if pp["H4"] >= H_CALL: return "H4_shared_causal"
    if pp["H3"] >= H_CALL: return "H3_distinct_causal"
    if pp["H1"] >= H_CALL or pp["H2"] >= H_CALL: return "single_trait_only"
    if pp["H0"] >= H_CALL: return "no_signal_either"
    return "ambiguous"

def fmt(v, s=".3g", na="NA"):
    try:
        if v is None or (isinstance(v,float) and (np.isnan(v) or not np.isfinite(v))):
            return na
        return format(float(v), s)
    except Exception:
        return na

# ---------------------------------------------------------------------
# LOAD + BUILD RESOLUTION (A)
# ---------------------------------------------------------------------
SUM = []; BAR = "=" * 92
def add(*ls):
    for l in ls: SUM.append(l)

add(BAR, "  STAGE-4  CROSS-DISORDER COLOCALIZATION (BIP vs migraine)  @ 8 JACKKNIFE LOCI",
    "  Method : coloc.abf (Wakefield; single-causal), pure-Python", BAR, "")

bip_df, bip_info = load_gwas(BIP_PATH, D1)
mig_df, mig_info = load_gwas(MIG_PATH, D2)

add("  INPUT FILES:")
for lab, inf, glab in [(D1, bip_info, BIP_GWAS_LABEL), (D2, mig_info, MIG_GWAS_LABEL)]:
    add(f"    [{lab}={glab}] {inf['path']}")
    add(f"        sep={repr(inf.get('sep'))}  loaded={inf.get('loaded')}  "
        f"SNPs_in_windows={inf.get('n_kept',0)}")
    add(f"        detected columns: " + ", ".join(f"{k}->{v}" for k,v in inf.get('colmap',{}).items()))
    if inf.get("error"): add(f"        ⚠ {inf['error']}")
add("")

bip_build, bip_votes = file_build(bip_df)
mig_build, mig_votes = file_build(mig_df)
add("  BUILD DETECTION (file-level vote; ±%d bp SNP counts):" % WIN_GENE,
    f"    bip      -> {bip_build}   votes={bip_votes}",
    f"    migraine -> {mig_build}   votes={mig_votes}")

BUILD = "hg38"; build_ok = True; lift_note = ""
if bip_df.empty or mig_df.empty:
    build_ok = False; lift_note = "one or both GWAS not loaded"
elif bip_build == mig_build:
    BUILD = bip_build
    lift_note = f"both files agree on {BUILD}; no lift needed"
else:
    # lift the hg19 frame to hg38
    lift_note = f"MISMATCH bip={bip_build} vs migraine={mig_build}"
    if bip_build == "hg19":
        bip_df, ok = lift_hg19_to_hg38(bip_df)
        if ok: BUILD = "hg38"; lift_note += " -> lifted BIP hg19->hg38"
        else:  build_ok = False; lift_note += " -> pyliftover unavailable"
    elif mig_build == "hg19":
        mig_df, ok = lift_hg19_to_hg38(mig_df)
        if ok: BUILD = "hg38"; lift_note += " -> lifted migraine hg19->hg38"
        else:  build_ok = False; lift_note += " -> pyliftover unavailable"
add(f"    resolution: {lift_note}",
    f"    pyliftover installed: {_HAVE_LIFT}",
    f"    analysis build: {BUILD if build_ok else 'UNRESOLVED (genes -> BUILD_MISMATCH)'}", "")

add("  CONFIG:",
    f"    gene window ±{WIN_GENE:,} bp | lead window ±{WIN_LEAD:,} bp | "
    f"leads-same-locus if within {LEAD_APART_MAX:,} bp",
    f"    GWS<{GWS} | suggestive<{SUGG} | min matched SNPs={MIN_MATCH}",
    f"    priors p1={P1} p2={P2} p12={P12} | sd.prior cc={SD_PRIOR_CC} eqtl={SD_PRIOR_EQTL}",
    f"    call PP thresh={H_CALL} | shared_snp printed only if H4>={SHARED_SNP_H4}", "")

def centre(g): return g[BUILD] if build_ok else g["hg38"]

# ---------------------------------------------------------------------
# STAGE-4a  per-locus GWAS<->GWAS coloc
# ---------------------------------------------------------------------
rows = []
add(BAR, "  [4a] PER-LOCUS CROSS-DISORDER COLOCALIZATION (8 jackknife genes)", BAR, "")

def run_locus(gname, g, ctr_pos, tag=""):
    rec = {"Gene": gname+tag, "chr": g["chr"], "note": g["note"],
           "build": BUILD if build_ok else "UNRESOLVED",
           "bip_gwas": BIP_GWAS_LABEL, "mig_gwas": MIG_GWAS_LABEL}
    if not build_ok:
        rec["call"] = "BUILD_MISMATCH"; return rec, None, None

    bwin = slice_window(bip_df, g["chr"], ctr_pos, WIN_GENE)
    mwin = slice_window(mig_df, g["chr"], ctr_pos, WIN_GENE)
    rec["n_bip_gene"] = len(bwin); rec["n_mig_gene"] = len(mwin)
    lb, lm = lead(bwin), lead(mwin)
    if lb:
        rec.update(bip_lead_rsid=lb["rsid"], bip_lead_pos=lb["pos"],
                   bip_lead_p=lb["p"], bip_gws=bool(lb["p"]<GWS), bip_sugg=bool(lb["p"]<SUGG))
    if lm:
        rec.update(mig_lead_rsid=lm["rsid"], mig_lead_pos=lm["pos"],
                   mig_lead_p=lm["p"], mig_gws=bool(lm["p"]<GWS), mig_sugg=bool(lm["p"]<SUGG))

    # C. neither trait has a lead at all
    if lb is None and lm is None:
        rec["call"] = "no_regional_signal"; return rec, lb, lm
    # B. leads not the same locus -> do not ABF
    if lb and lm and abs(lb["pos"]-lm["pos"]) > LEAD_APART_MAX:
        rec["call"] = "LEADS_NOT_SAME_LOCUS"
        rec["lead_distance"] = int(abs(lb["pos"]-lm["pos"]))
        return rec, lb, lm

    # B. if a trait has suggestive signal, shrink both windows to that lead ±WIN_LEAD
    anchor = lb if (lb and lb["p"] < SUGG) else (lm if (lm and lm["p"] < SUGG) else None)
    if anchor is not None:
        bwin = slice_window(bip_df, g["chr"], anchor["pos"], WIN_LEAD)
        mwin = slice_window(mig_df, g["chr"], anchor["pos"], WIN_LEAD)
        rec["anchor_pos"] = anchor["pos"]

    mrg, how = harmonize_merge(bwin, mwin)
    rec["merge_key"] = how; rec["n_matched"] = len(mrg)
    if lb and lm:
        rec["lead_distance"] = int(abs(lb["pos"]-lm["pos"]))

    if len(mrg) < MIN_MATCH:
        rec["call"] = "insufficient_overlap"
        for h in ("H0","H1","H2","H3","H4"): rec[f"PP.{h}"] = np.nan
        return rec, lb, lm

    # C. hard gate on signal before coloc
    if not ((lb and lb["p"] < SUGG) or (lm and lm["p"] < SUGG)):
        rec["call"] = "no_regional_signal"
        for h in ("H0","H1","H2","H3","H4"): rec[f"PP.{h}"] = np.nan
        return rec, lb, lm

    pp, idx = coloc_abf(mrg["beta_b"].values, mrg["se_b"].values,
                        mrg["beta_m_h"].values, mrg["se_m"].values)
    for h in ("H0","H1","H2","H3","H4"): rec[f"PP.{h}"] = pp[h]
    rec["call"] = classify(pp, lb, lm)

    # D. shared SNP only if H4 >= 0.5
    if pp["H4"] >= SHARED_SNP_H4:
        sc = mrg.iloc[idx]
        rec["shared_snp"] = str(sc.get("rsid", sc.get("key","?")))
        rec["shared_bip_beta"] = float(sc["beta_b"])
        rec["shared_mig_beta"] = float(sc["beta_m_h"])
        rec["shared_bip_p"] = float(sc["p_b"])
        rec["shared_mig_p"] = float(sc["p_m"])
        rec["dir_concordant"] = bool(np.sign(sc["beta_b"]) == np.sign(sc["beta_m_h"]))
    return rec, lb, lm

for gname, g in GENES.items():
    if bip_df.empty or mig_df.empty:
        rec = {"Gene":gname,"chr":g["chr"],"note":g["note"],"call":"gwas_not_loaded",
               "bip_gwas":BIP_GWAS_LABEL,"mig_gwas":MIG_GWAS_LABEL}
        rows.append(rec); add(f"  {gname}: GWAS not loaded — skipped"); add(""); continue

    rec, lb, lm = run_locus(gname, g, centre(g))
    rows.append(rec)

    # narrative
    add(f"  ● {gname}  (chr{g['chr']}; {g['note']})  [build={rec.get('build')}]")
    add(f"      GWAS: bip={BIP_GWAS_LABEL} | migraine={MIG_GWAS_LABEL}")
    if rec["call"] == "BUILD_MISMATCH":
        add("      CALL : BUILD_MISMATCH (builds unresolved; no coloc, no pos subtraction)"); add(""); continue
    add(f"      n_gene: bip={rec.get('n_bip_gene')} mig={rec.get('n_mig_gene')} "
        f"matched={rec.get('n_matched','-')} ({rec.get('merge_key','-')})")
    if lb: add(f"      BIP lead      : {lb['rsid']} @ {lb['pos']:,}  p={fmt(lb['p'],'.2e')} "
               f"beta={fmt(lb['beta'],'+.3f')}  {'GWS' if lb['p']<GWS else ('sugg' if lb['p']<SUGG else 'ns')}")
    if lm: add(f"      migraine lead : {lm['rsid']} @ {lm['pos']:,}  p={fmt(lm['p'],'.2e')} "
               f"beta={fmt(lm['beta'],'+.3f')}  {'GWS' if lm['p']<GWS else ('sugg' if lm['p']<SUGG else 'ns')}")
    if "lead_distance" in rec:
        same = (lb and lm and lb["rsid"]==lm["rsid"] and lb["rsid"]!="NA")
        add(f"      lead distance : {rec['lead_distance']:,} bp" + ("  (same variant)" if same else ""))
    if not np.isnan(rec.get("PP.H4", np.nan)):
        add(f"      coloc PP : H0={fmt(rec['PP.H0'],'.2f')} H1={fmt(rec['PP.H1'],'.2f')} "
            f"H2={fmt(rec['PP.H2'],'.2f')} H3={fmt(rec['PP.H3'],'.3f')} H4={fmt(rec['PP.H4'],'.3f')}")
        if "shared_snp" in rec:
            add(f"      H4-best SNP: {rec['shared_snp']}  "
                f"BIPbeta={fmt(rec.get('shared_bip_beta'),'+.3f')} "
                f"MIGbeta={fmt(rec.get('shared_mig_beta'),'+.3f')} "
                f"-> {'CONCORDANT' if rec.get('dir_concordant') else 'OPPOSITE'} direction")
    add(f"      CALL : {rec['call'].upper()}")
    add("")

# I. lead-conditioned BAD re-run (rs484201 ± WIN_LEAD)
if build_ok and not bip_df.empty and not mig_df.empty and "BAD" in GENES:
    gB = GENES["BAD"]
    bwin_full = slice_window(bip_df, gB["chr"], centre(gB), LOAD_WIN)
    hit = bwin_full[bwin_full.get("rsid","").astype(str) == BAD_LEAD_RSID] if "rsid" in bwin_full.columns else pd.DataFrame()
    bad_lead_pos = int(hit["pos"].iloc[0]) if not hit.empty else None
    if bad_lead_pos is None:
        lb0 = lead(slice_window(bip_df, gB["chr"], centre(gB), WIN_GENE))
        bad_lead_pos = lb0["pos"] if lb0 else None
    add(BAR, f"  [4a-I] BAD lead-conditioned re-run ({BAD_LEAD_RSID or 'BIP lead'} ± {WIN_LEAD:,} bp)", BAR)
    if bad_lead_pos is not None:
        recI, lbI, lmI = run_locus("BAD", gB, bad_lead_pos, tag="__leadcond")
        rows.append(recI)
        if not np.isnan(recI.get("PP.H4", np.nan)):
            add(f"    PP : H0={fmt(recI['PP.H0'],'.2f')} H1={fmt(recI['PP.H1'],'.2f')} "
                f"H2={fmt(recI['PP.H2'],'.2f')} H3={fmt(recI['PP.H3'],'.3f')} H4={fmt(recI['PP.H4'],'.3f')}")
        add(f"    CALL : {recI['call'].upper()}")
        add("    (If H1≈high & H4≈low here, the SINGLE_TRAIT_ONLY BAD call is solid.)")
    else:
        add("    (BAD lead not locatable; skipped)")
    add("")

res = pd.DataFrame(rows)
res.to_csv(OUT / "stage4_coloc_bip_vs_migraine.csv", index=False)

# ---------------------------------------------------------------------
# 4a interpretation
# ---------------------------------------------------------------------
add(BAR, "  [4a-INTERPRETATION]  meaning of each call for the Layer-A contrast", BAR)
add("   H4_shared_causal    : SAME causal variant in both disorders. A Layer-A opposite-sign",
    "                         TWAS then implies opposite effect/eQTL direction of that variant.",
    "   H3_distinct_causal  : DIFFERENT causal variants in LD -> gene-name overlap is LD, not shared.",
    "   single_trait_only   : locus real in one disorder only (H1/H2).",
    "   no_regional_signal  : neither trait suggestive here; coloc uninformative (NOT a null).",
    "   LEADS_NOT_SAME_LOCUS: leads >%d kb apart post-build; ABF would be meaningless -> skipped." % (LEAD_APART_MAX//1000),
    "   BUILD_MISMATCH      : builds could not be unified; no cross-build pos math performed.",
    "   insufficient_overlap/gwas_not_loaded : technical, not biological.", "")
if not res.empty and "call" in res.columns:
    add("  CALL TALLY:")
    for k, v in res["call"].value_counts().to_dict().items(): add(f"    {k:24s}: {v}")
    h4 = res[res["call"]=="H4_shared_causal"]["Gene"].tolist()
    h3 = res[res["call"]=="H3_distinct_causal"]["Gene"].tolist()
    if h4: add("", f"  ★ Shared-causal (H4): {', '.join(h4)}")
    if h3: add(f"  ✗ Distinct-causal (H3; LD artifacts): {', '.join(h3)}")
add("")

# ---------------------------------------------------------------------
# STAGE-4b  GWAS<->eQTL coloc (G) — needs rsid,chr,pos,ea,oa,beta,se
# ---------------------------------------------------------------------
add(BAR, "  [4b] GWAS<->eQTL COLOCALIZATION (allele-harmonized; requires eQTL slices)", BAR)
eqtl_files = (sorted(EQTL_DIR.glob("*.tsv")) + sorted(EQTL_DIR.glob("*.tsv.gz"))
              + sorted(EQTL_DIR.glob("*.csv")))
if not build_ok:
    add("  (build unresolved — Stage-4b SKIPPED)", "")
elif not eqtl_files:
    add("  (no eQTL slices in "+str(EQTL_DIR)+" — Stage-4b SKIPPED)", "",
        "  Drop one file per gene×tissue named e.g. BAD__Brain_Frontal_Cortex_BA9.tsv",
        "  REQUIRED columns: rsid, chr, pos, ea, oa, beta, se   (cis-eQTL in gene ±250kb, hg38).",
        "  Run 4b first on genes with a suggestive/GWS GWAS peak (right now: BAD; maybe NDUFAB1).", "",
        "  Data sources:",
        "    GTEx v8 all-pairs (all-associations, not sig-only): "
        "gs://gtex-resources/GTEx_Analysis_v8_eQTL_all_associations (requester-pays);",
        "      https://gtexportal.org/home/downloads/adult-gtex#qtl",
        "    eQTLGen blood cis : https://eqtlgen.org/cis-eqtls.html",
        "    MetaBrain cortex  : https://www.metabrain.nl/",
        "    PsychENCODE       : http://resource.psychencode.org/  (DER-08* eQTL, PEC_TWAS_weights)",
        "    PredictDB models  : https://predictdb.org/",
        "    1000G hg38 EUR LD : http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage/",
        "  Then re-run this cell — coloc_abf() is reused automatically.", "")
else:
    add(f"  {len(eqtl_files)} eQTL slice(s) found — running GWAS<->eQTL coloc.", "")
    eqrows = []
    for f in eqtl_files:
        stem = f.name.replace(".tsv.gz","").replace(".tsv","").replace(".csv","")
        parts = stem.split("__")
        gname = parts[0]; tissue = parts[1] if len(parts) > 1 else "unknown"
        if gname not in GENES:
            add(f"    ⚠ {f.name}: gene '{gname}' not in priority list — skipped"); continue
        try:
            ed = pd.read_csv(f, sep=None, engine="python", dtype=str)
        except Exception as e:
            add(f"    ⚠ {f.name}: read failed ({e})"); continue
        ecm = _map_columns(ed.columns)
        need_e = ["rsid","chr","pos","ea","oa","se"]
        if not all(k in ecm for k in need_e) or ("beta" not in ecm and "or" not in ecm):
            add(f"    ⚠ {f.name}: need rsid,chr,pos,ea,oa,beta,se (got {ecm})"); continue
        ed = ed.rename(columns={v:k for k,v in ecm.items()})
        ed["chr"] = ed["chr"].map(_norm_chr)
        ed["pos"] = pd.to_numeric(ed["pos"], errors="coerce")
        ed["se"]  = pd.to_numeric(ed["se"], errors="coerce")
        if "beta" in ed.columns:
            ed["beta"] = pd.to_numeric(ed["beta"], errors="coerce")
        else:
            ed["beta"] = np.log(pd.to_numeric(ed["or"], errors="coerce"))
        for c in ("ea","oa"): ed[c] = ed[c].astype(str).str.strip().str.upper()
        ed = ed.dropna(subset=["chr","pos","ea","oa","beta","se"])
        ed = ed[ed["se"] > 0]; ed["pos"] = ed["pos"].astype("int64")

        for trait, gdf in [(D1,bip_df),(D2,mig_df)]:
            if gdf.empty: continue
            loc = slice_window(gdf, GENES[gname]["chr"], centre(GENES[gname]), WIN_GENE)
            if loc.empty: continue
            mrg, how = harmonize_merge(loc, ed)   # reuse chr:pos + allele flip
            if len(mrg) < MIN_MATCH:
                eqrows.append(dict(Gene=gname, tissue=tissue, trait=trait,
                                   n=len(mrg), merge_key=how, call="insufficient_overlap"))
                continue
            pp,_ = coloc_abf(mrg["beta_b"].values, mrg["se_b"].values,
                             mrg["beta_m_h"].values, mrg["se_m"].values,
                             sd2=SD_PRIOR_EQTL)
            row = dict(Gene=gname, tissue=tissue, trait=trait, n=len(mrg), merge_key=how,
                       **{f"PP.{h}":pp[h] for h in ["H0","H1","H2","H3","H4"]})
            row["call"] = ("H4_coloc" if pp["H4"]>=H_CALL else
                           "H3_distinct" if pp["H3"]>=H_CALL else "ambiguous")
            eqrows.append(row)
            add(f"    {gname:10s} {trait:8s} {tissue:30s} n={len(mrg):5d} ({how}) "
                f"H3={fmt(pp['H3'],'.2f')} H4={fmt(pp['H4'],'.2f')} -> {row['call']}")
    if eqrows:
        eqdf = pd.DataFrame(eqrows)
        eqdf.to_csv(OUT / "stage4b_gwas_eqtl_coloc.csv", index=False)
        add("", "  GENE-LEVEL KEEP rule: H4>=0.80 in >=2 tissues (or 1 cortex + eQTLGen):")
        for gname in GENES:
            for trait in (D1,D2):
                sub = eqdf[(eqdf["Gene"]==gname)&(eqdf["trait"]==trait)&(eqdf.get("PP.H4",0)>=H_CALL)]
                if len(sub) >= 2:
                    add(f"    ★ {gname} × {trait}: eQTL-coloc in {len(sub)} tissues "
                        f"({', '.join(sub['tissue'].tolist())})")
    add("")

# ---------------------------------------------------------------------
# GUARDRAILS
# ---------------------------------------------------------------------
add(BAR, "  LIMITATIONS & GUARDRAILS", BAR,
    "   - Stage-4a is BIP<->migraine GWAS coloc, NOT the eQTL coloc that names a causal gene.",
    "   - coloc.abf assumes ONE causal variant/locus; multi-causal loci need coloc.susie + 1000G LD.",
    f"   - Build unified to {BUILD if build_ok else 'UNRESOLVED'}; positions never subtracted across builds.",
    "   - Gene centres are GENCODE TSS-ish; VERIFY vs GENCODE v26 before publishing.",
    f"   - migraine = {MIG_GWAS_LABEL} (~10.9k cases): a SENSITIVITY proxy, not IHGC/Hautakangas.",
    "     A null here (esp. TNFRSF1A) is NOT a biological null; replicate on IHGC + 23andMe.",
    "   - BIP is the noUKB_no23andMe subset, mitigating (not eliminating) sample overlap on H4.",
    "   - shared_snp is reported only when H4>=0.5; below that no 'shared variant' is claimed.",
    "   - No causal / clinical / drug-response claim follows from coloc alone.",
    BAR)

# ---------------------------------------------------------------------
# WRITE
# ---------------------------------------------------------------------
sp = OUT / "stage4_summary.txt"
sp.write_text("\n".join(SUM))
print("\n".join(SUM))
print(f"\n✓ Summary → {sp.resolve()}")
print(f"✓ CSVs    → {OUT.resolve()}/")
print(f"  Files: {sorted(p.name for p in OUT.glob('*.csv'))}")

  STAGE-4  CROSS-DISORDER COLOCALIZATION (BIP vs migraine)  @ 8 JACKKNIFE LOCI
  Method : coloc.abf (Wakefield; single-causal), pure-Python

  INPUT FILES:
    [bip=PGC_bip2024_eur_noUKB_no23andMe] /content/bip2024_eur_noUKB_no23andMe.gz
        sep='\\s+'  loaded=True  SNPs_in_windows=37892
        detected columns: chr->CHR, pos->BP, ea->A1, oa->A2, or->OR, se->SE, p->P, rsid->SNP
    [migraine=UKB_GCST90671940_proxy] /content/GCST90671940.tsv.gz
        sep='\t'  loaded=True  SNPs_in_windows=56620
        detected columns: chr->chromosome, pos->base_pair_location, ea->effect_allele, oa->other_allele, beta->beta, se->standard_error, p->p_value, eaf->effect_allele_frequency, rsid->rs_id

  BUILD DETECTION (file-level vote; ±250000 bp SNP counts):
    bip      -> hg38   votes={'hg19': 9535, 'hg38': 9694}
    migraine -> hg38   votes={'hg19': 13827, 'hg38': 13899}
    resolution: both files agree on hg38; no lift needed
    pyliftover installed: False
    analysis build: hg38

  CONFIG:

# Stage 4 Robustness

In [ ]:
# =====================================================================
#  STAGE-4 ROBUSTNESS PIPELINE (results-only; no S-PrediXcan / GWAS needed)
#  Consumes outputs of 089Av2_geneset_proximity_(BIP_vs_migraine).py
#  Stress-tests the Stage-3 framing (shared liability + opposite
#  apoptotic/TNF/OXPHOS set-points + BIP-shifted LTP != lithium biology)
#  by turning the only knobs these CSVs allow: |Z| cut, recurrence cut,
#  which gene set may vote, and whether repeaters may vote.
#  Writes: /content/089Av2_results/downstream_robustness/*.csv
#          + stage4_robustness_summary.txt
# =====================================================================
import warnings
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
RESULTS = Path("/content/089Av2_results")
OUT     = RESULTS / "downstream_robustness"
OUT.mkdir(parents=True, exist_ok=True)

Z_BASE          = 1.96          # headline |Z| cut
K_BASE          = 2             # headline min opposite-sign set-rows
Z_GRID          = [1.64, 1.96, 2.58, 3.0]
K_GRID          = [1, 2, 3]
REPEATER_NSETS  = 3
KEEP_FRAC       = 0.66          # fraction of headline that must survive => KEEP
EXPLODE_N       = 80            # Layer A larger than this at 1.64 => threshold fog
N_BOOT          = 2000
RNG_SEED        = 42
rng = np.random.default_rng(RNG_SEED)

# Panels / cassettes referenced by the framing
RHEOSTAT   = ["BAD", "BCL2L1", "BNIP3L", "GPX4", "BCL2L11", "MCL1", "BAX", "PMAIP1"]
LTP_CASSETTE = ["EP300", "PLCB3", "MAP2K2", "CALML6", "GNAQ", "CREBBP",
                "PRKACA", "PRKCG", "CAMK2A", "PPP3CA", "RAF1"]
TWO_LAYER  = ["GSK3B", "ARNTL", "PER2", "CLOCK", "CRY1", "GRIN2A", "MTOR",
              "IMPA1", "IMPA2", "CACNA1C", "ANK3"]
KEY_ANCHORS = ["TNFRSF1A", "BAD", "STX1A", "NDUFAB1", "BCL2L1", "GPX4", "BNIP3L"]
OXPHOS_SET  = "HALLMARK_OXIDATIVE_PHOSPHORYLATION_ALL"
LTP_SET     = "KEGG_LTP_ALL"

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def safe_read(p):
    try:
        return pd.read_csv(p)
    except Exception:
        return pd.DataFrame()

def fmt(v, spec=".3f", na="NA"):
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return na
        return format(float(v), spec)
    except Exception:
        return str(v) if v is not None else na

# ---------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------
genes  = safe_read(RESULTS / "all_genes_all_genesets.csv")
enrich = safe_read(RESULTS / "multi_geneset_enrichment_summary.csv")

load_report = {
    "all_genes_all_genesets.csv": len(genes),
    "multi_geneset_enrichment_summary.csv": len(enrich),
}

meta_z_cols = [c for c in genes.columns if c.startswith("Z_") and "__" not in c] if not genes.empty else []
DISEASES = sorted(enrich["Disease"].dropna().unique().tolist()) if (not enrich.empty and "Disease" in enrich.columns) else []
dis_col = {d: f"Z_{d}" for d in DISEASES if f"Z_{d}" in meta_z_cols}
if len(dis_col) != 2 and len(meta_z_cols) >= 2:
    dis_col = {c.replace("Z_", ""): c for c in meta_z_cols[:2]}
    DISEASES = list(dis_col.keys())

SUM = []; BAR = "=" * 92
def add(*ls):
    for l in ls: SUM.append(l)
verdicts = {}

add(BAR, "  STAGE-4 ROBUSTNESS SUMMARY (results-only)  —  BIP vs migraine",
    f"  Source : {RESULTS.resolve()}",
    f"  Diseases detected : {', '.join(DISEASES) if DISEASES else 'NONE'}",
    BAR, "", "  INPUT FILES (rows loaded):")
for k, v in load_report.items():
    add(f"    {k:44s}: {v}")
add("")

if genes.empty or len(dis_col) != 2:
    add("  ✗ Cannot proceed: need all_genes_all_genesets.csv with two Z_<disease> columns.")
    (OUT / "stage4_robustness_summary.txt").write_text("\n".join(SUM))
    print("\n".join(SUM)); raise SystemExit

D1, D2 = list(dis_col.keys())[:2]
C1, C2 = dis_col[D1], dis_col[D2]
# identify which is migraine / bip for private-direction naming
MIG = next((d for d in (D1, D2) if "mig" in d.lower()), D2)
BIP = D1 if MIG == D2 else D2
add(f"  D1={D1} ({C1}) | D2={D2} ({C2})  |  BIP-label={BIP}  MIG-label={MIG}",
    "", "  KNOBS:",
    f"    headline rule : |Z|>={Z_BASE}, opposite in >= {K_BASE} set-rows",
    f"    z grid        : {Z_GRID}",
    f"    k grid        : {K_GRID}",
    f"    repeater      : gene in >= {REPEATER_NSETS} sets",
    f"    KEEP fraction : {KEEP_FRAC} of headline must survive",
    f"    bootstrap     : {N_BOOT}",
    "")

# ---------------------------------------------------------------------
# CLASSIFIER + per-gene median aggregate
# ---------------------------------------------------------------------
def classify(zb, zm, z=Z_BASE):
    if not (np.isfinite(zb) and np.isfinite(zm)): return "missing"
    if abs(zb) < z and abs(zm) < z: return "both_weak"
    if zb >  z and zm >  z: return "concordant_pos"
    if zb < -z and zm < -z: return "concordant_neg"
    if zb >  z and zm < -z: return f"opp_{D1}pos_{D2}neg"
    if zb < -z and zm >  z: return f"opp_{D1}neg_{D2}pos"
    if abs(zb) >= z: return f"{D1}_private"
    if abs(zm) >= z: return f"{D2}_private"
    return "mixed_subthreshold"

g = genes.dropna(subset=[C1, C2], how="all").copy()
agg = (g.groupby("Gene")
       .agg(n_sets=("GeneSet", "nunique"),
            **{f"Z_{D1}_median": (C1, "median"),
               f"Z_{D2}_median": (C2, "median")})
       .reset_index())
agg["is_repeater"] = agg.n_sets >= REPEATER_NSETS
u = agg.set_index("Gene")

def med_z(gene):
    if gene in u.index:
        return u.loc[gene, f"Z_{D1}_median"], u.loc[gene, f"Z_{D2}_median"]
    return np.nan, np.nan

# paired set-rows with opposite-sign flags
pair = g.dropna(subset=[C1, C2]).copy()
pair["opp"]    = np.sign(pair[C1]) != np.sign(pair[C2])
pair["minabs"] = np.minimum(pair[C1].abs(), pair[C2].abs())

def layer_A_from(frame, zcut=Z_BASE, k=K_BASE):
    hit = frame[frame["opp"] & (frame["minabs"] >= zcut)]
    cnt = hit.groupby("Gene").size()
    return set(cnt[cnt >= k].index)

HEADLINE = layer_A_from(pair, Z_BASE, K_BASE)
add(f"  HEADLINE Layer-A (|Z|>={Z_BASE}, k>={K_BASE}): {len(HEADLINE)} genes")
add("    " + ", ".join(sorted(HEADLINE)[:40]), "")

# =====================================================================
# R1 — THRESHOLD GRID vs the headline definition
# =====================================================================
r1_rows = []
for zc in Z_GRID:
    for k in K_GRID:
        s = layer_A_from(pair, zc, k)
        inter = len(s & HEADLINE)
        r1_rows.append(dict(zcut=zc, k=k, n_layerA=len(s),
                            intersect_headline=inter,
                            frac_headline_kept=(inter/len(HEADLINE) if HEADLINE else np.nan)))
r1 = pd.DataFrame(r1_rows)
r1.to_csv(OUT / "R1_threshold_grid.csv", index=False)

kept_258 = float(r1[(r1.zcut == 2.58) & (r1.k == 2)]["frac_headline_kept"].iloc[0]) if not r1.empty else np.nan
kept_196_3 = float(r1[(r1.zcut == 1.96) & (r1.k == 3)]["frac_headline_kept"].iloc[0]) if not r1.empty else np.nan
n_164 = int(r1[(r1.zcut == 1.64) & (r1.k == 2)]["n_layerA"].iloc[0]) if not r1.empty else 0
r1_keep = ((max(kept_258, kept_196_3) >= KEEP_FRAC) and (n_164 <= EXPLODE_N))
verdicts["R1_threshold_stability"] = "KEEP" if r1_keep else "WEAKEN"

add(BAR, "  [R1] THRESHOLD GRID (does the 30-gene list survive its own knobs?)", BAR)
add(f"    headline kept @ (2.58,2) : {fmt(kept_258,'.2f')}")
add(f"    headline kept @ (1.96,3) : {fmt(kept_196_3,'.2f')}")
add(f"    Layer-A size @ (1.64,2)  : {n_164}  (explode cut {EXPLODE_N})")
add(f"    VERDICT                  : {verdicts['R1_threshold_stability']}", "")
for _, r in r1.iterrows():
    add(f"      z={r.zcut:<4} k={int(r.k)}  n={int(r.n_layerA):<4} "
        f"∩headline={int(r.intersect_headline):<3} ({fmt(r.frac_headline_kept,'.2f')})")
add("")

# =====================================================================
# R2 — DROP-ONE-SET JACKKNIFE (which set secretly carries Layer A?)
# =====================================================================
r2_rows = []
fragile_counter = {gene: 0 for gene in HEADLINE}
for gs in sorted(pair.GeneSet.unique()):
    s = layer_A_from(pair[pair.GeneSet != gs], Z_BASE, K_BASE)
    lost = sorted(HEADLINE - s)
    for gene in lost:
        fragile_counter[gene] += 1
    r2_rows.append(dict(dropped_set=gs, n_lost=len(lost),
                        lost_genes="; ".join(lost[:20])))
r2 = pd.DataFrame(r2_rows).sort_values("n_lost", ascending=False)
r2.to_csv(OUT / "R2_drop_one_set_jackknife.csv", index=False)
fragile = sorted([gk for gk, c in fragile_counter.items() if c > 0])
robust_core = sorted([gk for gk, c in fragile_counter.items() if c == 0])
worst = r2.iloc[0] if not r2.empty else None
r2_keep = (worst is None) or (int(worst["n_lost"]) <= 5)
verdicts["R2_not_list_bleed"] = "KEEP" if r2_keep else "WEAKEN"

pd.DataFrame({"Gene": list(fragile_counter.keys()),
              "n_sets_whose_removal_drops_it": list(fragile_counter.values())}
             ).sort_values("n_sets_whose_removal_drops_it", ascending=False
             ).to_csv(OUT / "R2_gene_fragility.csv", index=False)

add(BAR, "  [R2] DROP-ONE-SET JACKKNIFE (list-bleed detector)", BAR)
if worst is not None:
    add(f"    worst single set : {worst['dropped_set']} removes {int(worst['n_lost'])} headline genes")
add(f"    fragile genes    : {len(fragile)}  ({', '.join(fragile[:20])})")
add(f"    robust core      : {len(robust_core)}  ({', '.join(robust_core[:20])})")
add(f"    VERDICT          : {verdicts['R2_not_list_bleed']}", "")
for _, r in r2.head(12).iterrows():
    add(f"      drop {r['dropped_set']:52s} lost {int(r['n_lost'])}")
add("")

# =====================================================================
# R3 — MAJORITY SIGN (opposite is the typical row, not one lucky row)
# =====================================================================
r3_rows = []
for gene in sorted(HEADLINE):
    rows = pair[pair.Gene == gene]
    n_opp  = int(rows["opp"].sum())
    n_conc = int((~rows["opp"]).sum())
    r3_rows.append(dict(Gene=gene, n_rows=len(rows), n_opp=n_opp, n_conc=n_conc,
                        majority_opp=bool(n_opp > n_conc)))
r3 = pd.DataFrame(r3_rows).sort_values("majority_opp", ascending=False)
r3.to_csv(OUT / "R3_majority_sign.csv", index=False)
n_majopp = int(r3["majority_opp"].sum()) if not r3.empty else 0
r3_keep = (len(HEADLINE) > 0) and (n_majopp >= 0.83 * len(HEADLINE))
verdicts["R3_majority_opposite"] = "KEEP" if r3_keep else "WEAKEN"

add(BAR, "  [R3] MAJORITY SIGN (is opposite the typical set-row?)", BAR)
add(f"    {n_majopp}/{len(HEADLINE)} headline genes are majority-opposite across their rows")
add(f"    VERDICT : {verdicts['R3_majority_opposite']}", "")
for _, r in r3[~r3["majority_opp"]].head(15).iterrows():
    add(f"      ⚠ {r['Gene']:12s} opp_rows={r['n_opp']} conc_rows={r['n_conc']} (not majority-opp)")
add("")

# =====================================================================
# R4 — DROP ANNOTATION HUBS (n_sets >= 3)
# =====================================================================
hub_genes = set(agg[agg.is_repeater]["Gene"])
layerA_nonhub = sorted(HEADLINE - hub_genes)
anchor_status = {a: ("in-Layer-A" if a in HEADLINE else "absent") +
                     (" / hub" if a in hub_genes else " / non-hub") for a in KEY_ANCHORS}
r4_keep = len(layerA_nonhub) > 0
verdicts["R4_not_only_hubs"] = "KEEP" if r4_keep else "WEAKEN"
pd.DataFrame({"Gene": sorted(HEADLINE),
              "is_hub": [gk in hub_genes for gk in sorted(HEADLINE)]}
             ).to_csv(OUT / "R4_hub_status.csv", index=False)

add(BAR, "  [R4] DROP ANNOTATION HUBS (Layer A must not be only >=3-set genes)", BAR)
add(f"    Layer-A genes that are NON-hubs: {len(layerA_nonhub)}/{len(HEADLINE)}")
add(f"    non-hub Layer-A: {', '.join(layerA_nonhub[:25])}")
add("    key anchors:")
for a, st in anchor_status.items():
    add(f"      {a:12s} : {st}")
add(f"    VERDICT : {verdicts['R4_not_only_hubs']}", "")

# =====================================================================
# R5 — RHEOSTAT LEAVE-ONE-OUT (panel, not a single star)
# =====================================================================
def paradox(gene_list, zc=Z_BASE):
    s = agg[agg.Gene.isin(gene_list)]
    if len(s) == 0:
        return np.nan, 0
    cl = [classify(b, m, zc) for b, m in zip(s[f"Z_{D1}_median"], s[f"Z_{D2}_median"])]
    fo = np.mean([c.startswith("opp_") for c in cl])
    fc = np.mean([c.startswith("concordant_") for c in cl])
    return float(fo - fc), len(s)

present_rheo = [x for x in RHEOSTAT if x in u.index]
full_score, n_full = paradox(present_rheo)
r5_rows = [dict(removed="(none)", score=full_score, n=n_full)]
loo_scores = []
for gene in present_rheo:
    sc, nn = paradox([x for x in present_rheo if x != gene])
    r5_rows.append(dict(removed=gene, score=sc, n=nn))
    loo_scores.append(sc)
r5 = pd.DataFrame(r5_rows)
r5.to_csv(OUT / "R5_rheostat_LOO.csv", index=False)
r5_keep = (not np.isnan(full_score)) and (full_score > 0) and all(s > 0 for s in loo_scores if not np.isnan(s))
verdicts["R5_rheostat_is_panel"] = "KEEP" if r5_keep else "WEAKEN"

add(BAR, "  [R5] APOPTOTIC RHEOSTAT LOO (BAD vs BCL2L1 ± support, not BAD alone)", BAR)
add(f"    genes present in TWAS: {', '.join(present_rheo) if present_rheo else '(none)'}")
add(f"    full paradox score   : {fmt(full_score,'+.2f')} (n={n_full})")
for _, r in r5[r5.removed != "(none)"].iterrows():
    flag = "" if (pd.notna(r.score) and r.score > 0) else "  ⚠ score<=0 without this gene"
    add(f"      −{r['removed']:10s} score={fmt(r['score'],'+.2f')}{flag}")
add(f"    VERDICT : {verdicts['R5_rheostat_is_panel']}", "")

# =====================================================================
# R6 — OXPHOS GEOMETRY BOOTSTRAP (mixed geometry with CIs)
# =====================================================================
r6 = pd.DataFrame(); r6_keep = False
if OXPHOS_SET in set(g.GeneSet.unique()):
    ox = (g[g.GeneSet == OXPHOS_SET][["Gene", C1, C2]]
          .dropna(subset=[C1, C2]).drop_duplicates("Gene"))
    if not ox.empty:
        ox["class"] = [classify(b, m, Z_BASE) for b, m in zip(ox[C1], ox[C2])]
        mig_priv_lbl = f"{MIG}_private"
        def frac(cl_series, pred):
            return float(np.mean([pred(c) for c in cl_series]))
        cats = {
            "opposite":  lambda c: c.startswith("opp_"),
            "mig_priv":  lambda c: c == mig_priv_lbl,
            "conc_neg":  lambda c: c == "concordant_neg",
            "bip_priv":  lambda c: c == f"{BIP}_private",
            "weak":      lambda c: c in ("both_weak", "mixed_subthreshold"),
        }
        classes = ox["class"].tolist()
        n = len(classes)
        boot = {k: np.empty(N_BOOT) for k in cats}
        for b in range(N_BOOT):
            idx = rng.integers(0, n, n)
            samp = [classes[i] for i in idx]
            for k, pred in cats.items():
                boot[k][b] = frac(samp, pred)
        rows = []
        for k, pred in cats.items():
            obs = frac(classes, pred)
            lo, hi = np.percentile(boot[k], [2.5, 97.5])
            rows.append(dict(geometry=k, observed=round(obs, 3),
                             ci_low=round(float(lo), 3), ci_high=round(float(hi), 3),
                             excludes_zero=bool(lo > 0)))
        r6 = pd.DataFrame(rows)
        r6.to_csv(OUT / "R6_oxphos_geometry_bootstrap.csv", index=False)
        opp_ok = bool(r6[r6.geometry == "opposite"]["excludes_zero"].iloc[0])
        mig_ok = bool(r6[r6.geometry == "mig_priv"]["excludes_zero"].iloc[0])
        r6_keep = opp_ok and mig_ok
verdicts["R6_oxphos_mixed_geometry"] = "KEEP" if r6_keep else "WEAKEN"

add(BAR, "  [R6] OXPHOS GEOMETRY BOOTSTRAP (is 'mixed geometry' real?)", BAR)
if not r6.empty:
    add(f"    n unique OXPHOS genes bootstrapped ({N_BOOT}x):")
    for _, r in r6.iterrows():
        star = "  ✔ CI>0" if r["excludes_zero"] else ""
        add(f"      {r['geometry']:10s} obs={fmt(r['observed'],'.2f')} "
            f"CI95=[{fmt(r['ci_low'],'.2f')},{fmt(r['ci_high'],'.2f')}]{star}")
else:
    add("    (OXPHOS set not present)")
add(f"    VERDICT : {verdicts['R6_oxphos_mixed_geometry']}", "")

# =====================================================================
# R7 — PRIVATE-GENE COUNTS vs |Z| CUT (is 436 vs 296 just power?)
# =====================================================================
r7_rows = []
for zc in Z_GRID:
    cl = [classify(b, m, zc) for b, m in zip(agg[f"Z_{D1}_median"], agg[f"Z_{D2}_median"])]
    n_bip = sum(c == f"{BIP}_private" for c in cl)
    n_mig = sum(c == f"{MIG}_private" for c in cl)
    r7_rows.append(dict(zcut=zc, bip_private=n_bip, mig_private=n_mig,
                        ratio=(n_bip / n_mig if n_mig else np.nan)))
r7 = pd.DataFrame(r7_rows)
r7.to_csv(OUT / "R7_private_vs_zcut.csv", index=False)
ratios = r7["ratio"].dropna().values
r7_stable = len(ratios) >= 2 and (np.nanmax(ratios) - np.nanmin(ratios)) < 0.5
verdicts["R7_private_ratio_stable"] = "KEEP" if r7_stable else "WEAKEN(report as power)"

add(BAR, "  [R7] PRIVATE-GENE COUNTS vs |Z| CUT", BAR)
for _, r in r7.iterrows():
    add(f"      z={r.zcut:<4} {BIP}_private={int(r.bip_private):<4} "
        f"{MIG}_private={int(r.mig_private):<4} ratio={fmt(r.ratio,'.2f')}")
add(f"    VERDICT : {verdicts['R7_private_ratio_stable']}  "
    f"(do NOT quote raw counts unless KEEP)", "")

# =====================================================================
# R8 — LTP CASSETTE stays BIP-private (not Layer A)
# =====================================================================
r8_rows = []
for gene in LTP_CASSETTE:
    zb, zm = med_z(gene)
    row = {"Gene": gene, "in_TWAS": gene in u.index,
           "in_LTP_set": gene in set(g.loc[g.GeneSet == LTP_SET, "Gene"])}
    for zc in Z_GRID:
        row[f"class@{zc}"] = classify(zb, zm, zc) if gene in u.index else "absent"
    r8_rows.append(row)
r8 = pd.DataFrame(r8_rows)
r8.to_csv(OUT / "R8_ltp_cassette.csv", index=False)
def is_bip_priv(c): return c == f"{BIP}_private"
bipprivers = [r["Gene"] for r in r8_rows
              if r.get(f"class@1.64") and is_bip_priv(r[f"class@1.64"])
              and is_bip_priv(r.get(f"class@1.96", ""))]
r8_keep = len(bipprivers) >= 1
verdicts["R8_ltp_cassette_bip_private"] = "KEEP" if r8_keep else "WEAKEN"

add(BAR, "  [R8] LTP CASSETTE (BIP-private plasticity, not a contrast)", BAR)
for _, r in r8.iterrows():
    if not r["in_TWAS"]:
        add(f"      {r['Gene']:10s} : absent from TWAS matrix"); continue
    add(f"      {r['Gene']:10s} : " +
        "  ".join(f"z{zc}={r[f'class@{zc}']}" for zc in Z_GRID))
add(f"    stable {BIP}-private (1.64 & 1.96): {', '.join(bipprivers) if bipprivers else '(none)'}")
add(f"    VERDICT : {verdicts['R8_ltp_cassette_bip_private']}", "")

# =====================================================================
# R9 — TWO-LAYER MODEL (lithium/clock genes != LTP)
# =====================================================================
r9_rows = []
for gene in TWO_LAYER:
    zb, zm = med_z(gene)
    row = {"Gene": gene, "in_TWAS": gene in u.index}
    for zc in (1.64, 1.96, 2.58):
        row[f"class@{zc}"] = classify(zb, zm, zc) if gene in u.index else "absent"
    r9_rows.append(row)
r9 = pd.DataFrame(r9_rows)
r9.to_csv(OUT / "R9_two_layer.csv", index=False)
def stable_bip(row): return is_bip_priv(row.get("class@1.64", "")) and is_bip_priv(row.get("class@1.96", ""))
anchor_bip = [r["Gene"] for r in r9_rows if r["Gene"] in ("GSK3B", "ARNTL") and stable_bip(r)]
clock_mig = any(r["Gene"] == "CLOCK" and r.get("class@1.96", "") == f"{MIG}_private" for r in r9_rows)
r9_keep = len(anchor_bip) >= 1
verdicts["R9_two_layer_model"] = ("KEEP" if r9_keep else "WEAKEN(threshold fog)")

add(BAR, "  [R9] TWO-LAYER MODEL (GSK3B/ARNTL BIP-private, CLOCK migraine-private)", BAR)
for _, r in r9.iterrows():
    if not r["in_TWAS"]:
        add(f"      {r['Gene']:10s} : absent"); continue
    add(f"      {r['Gene']:10s} : " +
        "  ".join(f"z{zc}={r[f'class@{zc}']}" for zc in (1.64, 1.96, 2.58)))
add(f"    lithium/clock anchors stable-BIP-private: {', '.join(anchor_bip) if anchor_bip else '(none)'}")
add(f"    CLOCK migraine-private @1.96: {clock_mig}")
add(f"    VERDICT : {verdicts['R9_two_layer_model']}", "")

# =====================================================================
# R10 — SCORECARD (KEEP/WEAKEN per claim)
# =====================================================================
scorecard = {
    "R1_threshold_stability":       ("30 recurrent opposite genes survive their own |Z|/k knobs", verdicts["R1_threshold_stability"]),
    "R2_not_list_bleed":            ("Opposite core is not one-set annotation bleed",             verdicts["R2_not_list_bleed"]),
    "R3_majority_opposite":         ("Opposite is the typical set-row, not one lucky row",         verdicts["R3_majority_opposite"]),
    "R4_not_only_hubs":             ("Layer A is not only >=3-set annotation hubs",                verdicts["R4_not_only_hubs"]),
    "R5_rheostat_is_panel":         ("Apoptotic rheostat is a panel (survives LOO), not BAD alone",verdicts["R5_rheostat_is_panel"]),
    "R6_oxphos_mixed_geometry":     ("OXPHOS opposite + mig-private fractions have CI>0",          verdicts["R6_oxphos_mixed_geometry"]),
    "R7_private_ratio_stable":      ("BIP/MIG private ratio stable across |Z| (not just power)",   verdicts["R7_private_ratio_stable"]),
    "R8_ltp_cassette_bip_private":  ("LTP cassette stays BIP-private (plasticity, not contrast)",  verdicts["R8_ltp_cassette_bip_private"]),
    "R9_two_layer_model":           ("Lithium/clock two-layer model survives threshold change",    verdicts["R9_two_layer_model"]),
}
sc_df = pd.DataFrame([{"test": k, "claim": v[0], "verdict": v[1]} for k, v in scorecard.items()])
sc_df.to_csv(OUT / "R10_scorecard.csv", index=False)

n_keep = sum(1 for _, v in scorecard.values() if v.startswith("KEEP"))
add(BAR, "  [R10] ROBUSTNESS SCORECARD (raise confidence only on KEEP)", BAR)
for k, (claim, verd) in scorecard.items():
    mark = "✔" if verd.startswith("KEEP") else "✗"
    add(f"    {mark} {k:30s} [{verd}]")
    add(f"        {claim}")
add("")
add(f"    KEEP: {n_keep}/{len(scorecard)} claims", "")

# =====================================================================
# AUTO-DRAFT ROBUSTNESS PARAGRAPH
# =====================================================================
add(BAR, "  ★ AUTO-DRAFT ROBUSTNESS PARAGRAPH (verify before use)", BAR)
keep_list = [k for k, (_, v) in scorecard.items() if v.startswith("KEEP")]
weak_list = [k for k, (_, v) in scorecard.items() if not v.startswith("KEEP")]
para = (
    f"  The headline opposite-sign gene set ({len(HEADLINE)} genes at |Z|>={Z_BASE}, k>={K_BASE}) "
    f"was stress-tested on the four knobs available to results-only analysis. "
    f"Claims passing (KEEP): {', '.join(keep_list) if keep_list else 'none'}. "
    f"Claims that weakened under re-thresholding or jackknife: "
    f"{', '.join(weak_list) if weak_list else 'none'}. "
    f"Robust-core opposite genes (surviving every drop-one-set jackknife): "
    f"{', '.join(robust_core[:15]) if robust_core else 'none'}. "
    f"Fragile / likely list-bleed genes to footnote: {', '.join(fragile[:15]) if fragile else 'none'}. "
    f"Confidence should move upward only for KEEP claims; all WEAKEN claims stay hypothesis-generating. "
    f"No colocalization, eQTL R2, or MR claim is possible from these CSVs."
)
add(para, "")

add(BAR, "  GUARDRAILS", BAR)
add("   - Only claims marked KEEP in R10 may have their rubric-robustness raised toward ~70.",
    "   - Publish Layer A as genes passing R1∩R3; footnote R2-fragile names.",
    "   - State the rheostat as BAD vs BCL2L1 (± GPX4/BNIP3L) only if R5 = KEEP.",
    "   - State OXPHOS as geometries-with-CIs (R6), not 'migraine OXPHOS-down'.",
    "   - Do NOT quote raw private counts (436 vs 296) unless R7 = KEEP.",
    "   - LTP cassette genes named only if R8 keeps them BIP-private.",
    BAR)

# ---------------------------------------------------------------------
# WRITE
# ---------------------------------------------------------------------
summary_path = OUT / "stage4_robustness_summary.txt"
summary_path.write_text("\n".join(SUM))
print("\n".join(SUM[:90]))
print("\n...\n")
print(f"✓ Detailed summary → {summary_path.resolve()}")
print(f"✓ CSVs             → {OUT.resolve()}/")
print(f"  Files: {sorted(p.name for p in OUT.glob('*.csv'))}")

  STAGE-4 ROBUSTNESS SUMMARY (results-only)  —  BIP vs migraine
  Source : /content/089Av2_results
  Diseases detected : bip, migraine

  INPUT FILES (rows loaded):
    all_genes_all_genesets.csv                  : 4107
    multi_geneset_enrichment_summary.csv        : 714

  D1=bip (Z_bip) | D2=migraine (Z_migraine)  |  BIP-label=bip  MIG-label=migraine

  KNOBS:
    headline rule : |Z|>=1.96, opposite in >= 2 set-rows
    z grid        : [1.64, 1.96, 2.58, 3.0]
    k grid        : [1, 2, 3]
    repeater      : gene in >= 3 sets
    KEEP fraction : 0.66 of headline must survive
    bootstrap     : 2000

  HEADLINE Layer-A (|Z|>=1.96, k>=2): 30 genes
    ACADS, BAD, BCL2L1, BNIP3L, CRAT, DNAJA3, F2, FDX1, GCDH, GNAQ, GPX4, IQGAP1, MRE11, MRPL18, MRPS17, NDUFAB1, NLRP3, NOS2, OLR1, PAWR, PHC2, RETSAT, SCMH1, SLC25A3, STX1A, TLR2, TNFRSF1A, UBN1, VEGFA, VENTX

  [R1] THRESHOLD GRID (does the 30-gene list survive its own knobs?)
    headline kept @ (2.58,2) : 0.60
    headline kept @ (1.9

In [ ]:
!cp "/content/089Av2_results" "/content/drive/MyDrive/Dr Uccello/00_Studies/089_BIP" -r

### The End